This code is adapted from Marina's NuMI $\nu_e$ and $\bar{\nu}_e$ xsec analysis for the use of a similar NuMI $\nu_\mu$ and $\bar{\nu}_\mu$ xsec analysis. 

This file contains the analysis using MicroBooNE run 3b samples.

There are two stages:
* Separate $\nu_\mu$ and $\bar{\nu}_\mu$ by trainin a BDT, based on the BDT Marina developed for her analysis.
* Then, individual $\nu_\mu$ and $\bar{\nu}_\mu$ xsec will be measure using MicroBooNE off-axis NuMI beam data (run 1 and 3)

In [1]:
#====================#
#  Import Packages   #
#====================#

import math 
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import gridspec
from scipy.stats import beta

import uproot3 as uproot

# BDT
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_curve, auc, recall_score, precision_score, average_precision_score, precision_recall_curve
from sklearn.metrics import confusion_matrix
from sklearn.metrics import ConfusionMatrixDisplay
from sklearn.metrics import classification_report
from sklearn.utils.class_weight import compute_sample_weight
import joblib
import xgboost as xgb
from matplotlib.pylab import rcParams

import os

In [2]:
# dataframe creation lock 
lock1 = False

# dataframe loading lock
lock2 = not lock1


dfs_dir = "/scratch/wwang/dfs"

### Function Definitions

In [3]:
#===========================================#
#  Define the boundaries of the TPCc volume #
#===========================================#

tpc_xmin = -1.55
tpc_xmax = 254.8
tpc_ymin = -115.53
tpc_ymax = 117.47
tpc_zmin = 0.1
tpc_zmax = 1036.9

In [4]:
#========================================================#
#  A function for creating dataframes and importing data #
#  from root files                                       #
#========================================================#
def new_df(file, fType, pot_data, pot_ext, pot_mc, pot_dirt):
    
    
    #—————————————————————————————————————————————————
    # First, define variables needed from the WC trees
    #—————————————————————————————————————————————————
    
    
    #reco vars
    eval_vars_reco = ['flash_time', 'match_isFC']
    pfeval_vars_reco = ['reco_nuvtxX','reco_nuvtxY', 'reco_nuvtxZ', 'run', 'subrun',
                        'event', 'reco_muonMomentum', 'reco_showerKE', 'reco_mother', 'reco_pdg', 'reco_id', 'reco_Ntrack',
                        'reco_startMomentum', 'reco_startXYZT', 'reco_endXYZT']
    bdt_vars_reco = ['numu_cc_flag', 'gap_energy', 'gap_flag_single_shower','gap_flag', 'muonvtx_diff', 'numu_1_score', 
                     'numu_cc_3_track_length', 'numu_cc_3_max_length_all', 'cosmict_2_dQ_dx_front', 'cosmict_2_dQ_dx_end', 
                     'cosmict_2_angle_beam', 'cosmict_2_phi', 'numu_cc_3_max_length', 'numu_cc_3_max_muon_length', 
                     'lem_shower_main_length', 'lem_n_3seg', 'lem_e_charge', 'lem_e_dQdx', 'lem_shower_num_main_segs']
    kine_vars_reco = ['kine_reco_Enu', 'kine_pio_mass','kine_pio_flag','kine_pio_vtx_dis',
                      'kine_pio_energy_1','kine_pio_energy_2','kine_pio_dis_1',
                      'kine_pio_dis_2','kine_pio_angle'] #pi0 selection
    
    #true vars
    kine_vars_true = []
    bdt_vars_true = []
    pfeval_vars_true = ['truth_corr_nuvtxX', 'truth_corr_nuvtxY', 'truth_corr_nuvtxZ', 
                        'truth_nuIntType', 'truth_pdg', 'truth_id', 'truth_NprimPio', 'truth_mother',
                        'truth_muonendX', 'truth_muonendY', 'truth_muonendZ', 'truth_startMomentum',
                        'truth_startXYZT','truth_endXYZT', 'truth_Ntrack', 'truth_vtxX',
                        'truth_vtxY', 'truth_vtxZ', 'truth_muonMomentum']
    eval_vars_true = ['weight_cv', 'weight_spline', 'truth_nuPdg', 'truth_isCC', 
                      'truth_nuEnergy', 'match_completeness_energy', 'truth_energyInside', 'truth_vtxInside']
    
    
    #————————————————————————————————————
    # import the trees from the root file
    #————————————————————————————————————
    
    
    T_eval = uproot.open(file)['wcpselection/T_eval']
    T_PFeval = uproot.open(file)['wcpselection/T_PFeval']
    T_BDTvars = uproot.open(file)['wcpselection/T_BDTvars']
    T_KINEvars = uproot.open(file)['wcpselection/T_KINEvars']

    
    #———————————————————————————————————————————————————
    # for each tree, create a reco and a true data frame
    #———————————————————————————————————————————————————
    
    
    #Reco dfs
    df_eval_reco = T_eval.pandas.df(eval_vars_reco, flatten=False)
    df_PFeval_reco = T_PFeval.pandas.df(pfeval_vars_reco, flatten=False)
    df_BDTeval_reco = T_BDTvars.pandas.df(bdt_vars_reco, flatten=False)
    df_KINEvars_reco =T_KINEvars.pandas.df(kine_vars_reco, flatten=False)
    
    #use concat to merge the individual reco dataframes
    df_ = pd.concat([df_KINEvars_reco, df_BDTeval_reco, df_PFeval_reco, 
                     df_eval_reco], axis=1)
    
    #true dfs
    if((fType=='MC') | (fType=='DIRT')):
        
        #for each tree, create a true data frame
        df_eval_true = T_eval.pandas.df(eval_vars_true, flatten=False)
        df_PFeval_true = T_PFeval.pandas.df(pfeval_vars_true, flatten=False)
        df_BDTeval_true = T_BDTvars.pandas.df(bdt_vars_true, flatten=False)
        df_KINEvars_true =T_KINEvars.pandas.df(kine_vars_true, flatten=False)
        
        #use concat to merge the individual dataframes
        df_ = pd.concat([df_, df_KINEvars_true, df_BDTeval_true, df_PFeval_true, 
                         df_eval_true], axis=1)
        
        #fix vars, make sure they have reasonable values (weight_cv & weight_spline)
        df_['weight_cv'] = np.where((df_.weight_cv < 0), 1, df_.weight_cv)
        df_['weight_cv'] = np.where((df_.weight_cv > 30), 1, df_.weight_cv)
        df_['weight_cv'] = np.where((df_.weight_cv == np.nan), 1, df_.weight_cv)
        df_['weight_cv'] = np.where((df_.weight_cv == np.inf), 1, df_.weight_cv)
        df_['weight_cv'] = np.where((df_['weight_cv'].isna()), 1, df_.weight_cv)
        df_['weight_spline'] = np.where((df_.weight_spline < 0), 1, df_.weight_spline)
        df_['weight_spline'] = np.where((df_.weight_spline > 30), 1, df_.weight_spline)
        df_['weight_spline'] = np.where((df_.weight_spline == np.nan), 1, df_.weight_spline)
        df_['weight_spline'] = np.where((df_.weight_spline == np.inf), 1, df_.weight_spline)
        df_['weight_spline'] = np.where((df_['weight_spline'].isna()), 1, df_.weight_spline)
        
    
    #———————————————————————————————————
    # before/after trigger config change
    #———————————————————————————————————
    
    
    #df_before = df_[df_.run<16880]
    #df_after = df_[df_.run>=16880]
    
    
    #————————————————————————
    # calculate event weights
    #————————————————————————
    
    if(fType=='MC'):
        W_ = pot_data/pot_mc 
        df_.loc[:,'weight_genie'] = df_['weight_cv'] * df_['weight_spline']
        df_.loc[:,'weight_ubtune'] = [W_]*df_.shape[0] * df_['weight_genie']
        df_.loc[:,'weight_no_ubtune'] = [W_]*df_.shape[0] * df_['weight_spline']
    elif(fType=='DATA'): #for beam-on sample
        W_ = 1
        df_.loc[:,'weight_ubtune'] = [W_]*df_.shape[0]
    elif(fType=='EXT'): #for beam-off sample
        W_ = (pot_data/pot_ext)*0.98
        df_.loc[:,'weight_ubtune'] = [W_]*df_.shape[0]
    elif(fType=='EXT_run1'):
        W_ = (2.59e20/pot_ext)*0.98
        df_.loc[:,'weight_ubtune'] = [W_]*df_.shape[0]
    elif(fType=='DIRT'):
        W_ = pot_data/pot_dirt
        df_.loc[:,'weight_genie'] = df_['weight_cv'] * df_['weight_spline']
        df_.loc[:,'weight_ubtune'] = [W_]*df_.shape[0] * df_['weight_genie']
        df_.loc[:,'weight_no_ubtune'] = [W_]*df_.shape[0] * df_['weight_spline']
        

    #—————————————————————#
    # add extra variables #
    #—————————————————————#
    
    if ( (fType=='DATA') | (fType=='MC') | (fType=='DIRT') | (fType=='EXT') | (fType=='EXT_run1') ):
        df_ = calc_costheta(df_)
        df_ = calc_particle_multiplicity(file, df_)
    
    if((fType=='MC') | (fType=='DIRT')):
        df_ = calc_particle_multiplicity_truth(file, df_)
        df_ = add_truth_proton_kinetic_energy_summaries(df_)
        df_ = calc_costheta_truth(df_)


    
    #————————————————————————————————————————————
    # For beam-on/off data, apply a quality check
    #————————————————————————————————————————————
    
    
    # if((fType=='DATA') | (fType=='EXT')):
    #     df_good_runs = pd.read_csv('goodruns_mcc9_run3_hardcoded.list', 
    #             sep=", ", header=None, engine='python')
    #     df_good_runs = df_good_runs.T
    #     df_good_runs.rename(columns={0:'run'}, inplace=True)
    #     list_good_runs = df_good_runs['run'].values.tolist()
    #     df_ = df_[df_['run'].isin(list_good_runs)]

        # df_run_subrun = df_.filter(['run', 'subrun'], axis=1)
        # df_run_subrun.drop_duplicates(subset=['run', 'subrun'], inplace=True)

        # if(fType=='DATA'):
    
        #     # for run 3b, separate out before and after trigger change
        #     df_run_subrun_before = df_run_subrun[df_run_subrun.run < 16880]
        #     df_run_subrun_after = df_run_subrun[df_run_subrun.run >= 16880]
        #     # these contains only good runs, save them into .txt for getting relevant POT calc values
        #     np.savetxt('calc_data_pot_run_3b_before_filtered.txt', df_run_subrun_before.values, fmt='%d')
        #     np.savetxt('calc_data_pot_run_3b_after_filtered.txt', df_run_subrun_after.values, fmt='%d')
    
    #—————————————————————————————————————————————————————————————————
    # fix neutrino energy for data
    # the neutrino reconstructed energy needs to be corrected for data
    # tot_energy = shower_energy + track_energy
    # and the shower_energy needs a 95% correction factor
    #————————————————————————————————————————————————————————————————— 
    
    
    if(fType=='DATA'):
        def get_data_nuEnergyCorrected(df_, file):
            
            df_in = df_.copy()

            # create dataframe with relevant information to 
            # calculate DATA neutrino energy
            tree = uproot.open(file)['wcpselection/T_KINEvars']
            var = ['kine_energy_particle', 'kine_energy_info', 'kine_particle_type', 
                   'kine_reco_add_energy', 'kine_reco_Enu']
            df_temp = tree.pandas.df(var, flatten = True)

            # apply cuts and define new dataframes
            df_temp = df_temp[ df_temp.kine_reco_Enu>0 ]
            df_EM = df_temp[ (df_temp.kine_energy_info==2) \
                            & (df_temp.kine_particle_type==11) ] # shower
            df_track = df_temp[ (df_temp.kine_energy_info!=2) \
                               | (df_temp.kine_particle_type!=11) ] # track

            # apply 95% correction on EM
            df_EM.loc[:,'kine_energy_particle'] = df_EM['kine_energy_particle']\
                                                .apply(lambda x: x*0.95)

            df_temp.loc[:,'energy_EM'] = df_EM.groupby(['entry'])['kine_energy_particle']\
                                        .transform('sum')
            df_temp.loc[:,'energy_track'] = df_track.groupby(['entry'])\
                                        ['kine_energy_particle'].transform('sum')

            df_temp['energy_EM'].fillna(0, inplace=True)
            df_temp['energy_track'].fillna(0, inplace=True)

            df_temp['energy_EM'] = df_temp.groupby(['entry'])['energy_EM'].transform(max)
            df_temp['energy_track'] = df_temp.groupby(['entry'])['energy_track']\
                                        .transform(max)

            df_temp = df_temp.groupby('entry').first()

            df_temp['kine_reco_Enu_corr'] = df_temp['energy_EM'] + df_temp['energy_track'] \
                                            + df_temp['kine_reco_add_energy']
            df_temp = df_temp.drop(['kine_energy_particle','kine_energy_info',
                                    'kine_particle_type','kine_reco_add_energy',
                                    'kine_reco_Enu','energy_EM','energy_track'], axis=1)
            df_temp = df_temp.rename(columns={"kine_reco_Enu_corr":"kine_reco_Enu"})

            df_in.update(df_temp)
            return df_in

        df_ = get_data_nuEnergyCorrected(df_.copy(),file)
    
    
    #—————————————
    # add RSE info
    #—————————————
    
    
    df_['RSE'] = df_["run"].astype(int).apply(str) + "_" \
                + df_["subrun"].astype(int).apply(str) + "_" \
                + df_["event"].astype(int).apply(str)
    
    
    #——————————————————————
    # Print POT and Entries
    #——————————————————————
    
    
    if(fType=='MC'):
        print('[%s] POT %5.2e      %7i entries,    %7.2f POT-scaled entries' % (fType,pot_mc,len(df_),sum(df_['weight_ubtune'])))
        print(pot_data)
        print(pot_mc)
        print(pot_data/pot_mc)
    elif(fType=='DATA'): #for beam-on sample
        print('[%s] POT %5.2e      %7i entries' % (fType,pot_data,len(df_)))
    elif(fType=='EXT'): #for beam-off sample 
        print('[%s] POT %5.2e      %7i entries,    %7.2f POT-scaled entries' % (fType,pot_ext,len(df_),sum(df_['weight_ubtune'])))
        print(pot_data)
        print(pot_ext)
        print(pot_data/pot_ext)
    elif(fType=='EXT_run1'):
        print('[%s] POT %5.2e      %7i entries,    %7.2f POT-scaled entries' % (fType,2.59e20,len(df_),sum(df_['weight_ubtune'])))
    elif(fType=='DIRT'):
        print('[%s] POT %5.2e      %7i entries,    %7.2f POT-scaled entries' % (fType,pot_dirt,len(df_),sum(df_['weight_ubtune'])))
        print(pot_data)
        print(pot_dirt)
        print(pot_data/pot_dirt)
    return df_

In [5]:
#=================================#
#  A Function used to create pot  #
#=================================#
def calc_pot(file, fType):
    
    
    #—————————————————————
    #create necessary vars
    #—————————————————————
    
    
    if(fType=='MC'):
        pot_vars = ['runNo','pot_tor875']
    elif(fType=='DIRT'):
        pot_vars = ['runNo','pot_tor875']
    
    
    #——————————————————————————————————————
    #import the pot tree from the root file
    #——————————————————————————————————————
    
    
    T_pot = uproot.open(file)['wcpselection/T_pot']
    
    
    #———————————————————————————
    #Create a pd df for pot tree
    #———————————————————————————
    
    
    df_pot = T_pot.pandas.df(pot_vars, flatten=False)
    
    
    #—————————————————
    #Calculate the pot
    #—————————————————
    
    
    pot = sum(df_pot.pot_tor875)
    return pot

In [6]:
#================================================================================#
# A Function for calculating the angle b/w numi beam target and muon from vertex #
#================================================================================#

def calc_costheta(df):
    
    # position of the NuMI target
    # those values are manually added to the lines below
    # if you want to change them, make sure you also update the lines below!!
    v_targ_uboone = [-31387.58422, -3316.402543, -60100.2414]

    # take the vector from the NuMI target to the neutrino vertex
    df.eval('vec_targ_vtx_X = reco_nuvtxX - (-31387.58422)', inplace=True)
    df.eval('vec_targ_vtx_Y = reco_nuvtxY - (-3316.402543)', inplace=True)
    df.eval('vec_targ_vtx_Z = reco_nuvtxZ - (-60100.2414)', inplace=True)
    
    # get reco shower vector (workaround that is working, for some reason I can't access it if I don't do this workaround, maybe it's somethind to do with the name containing [ ])
    df.loc[:,'muon_momentum_X'] = df["reco_muonMomentum[0]"]
    df.loc[:,'muon_momentum_Y'] = df["reco_muonMomentum[1]"]
    df.loc[:,'muon_momentum_Z'] = df["reco_muonMomentum[2]"]
    df.loc[:,'muon_energy'] = df["reco_muonMomentum[3]"] # MeV to GeV
    
    # calculate the norm of the vectors
    df.eval('norm_vec_targ_vtx = sqrt(vec_targ_vtx_X**2 + vec_targ_vtx_Y**2 + vec_targ_vtx_Z**2)', inplace=True)
    df.eval('norm_vec_shower = sqrt(muon_momentum_X**2 + muon_momentum_Y**2 + muon_momentum_Z**2)', inplace=True)
    
    # calculate cos_theta
    df.eval('cos_theta = ((vec_targ_vtx_X * muon_momentum_X) + (vec_targ_vtx_Y * muon_momentum_Y) + (vec_targ_vtx_Z * muon_momentum_Z) )/(norm_vec_targ_vtx * norm_vec_shower)', inplace=True)

    return df

In [7]:
# #================================================================================#
# # A Function for calculating the angle b/w numi beam target and muon from vertex #
# #================================================================================#

# def calc_costheta_truth(df):
    
#     # position of the NuMI target
#     # those values are manually added to the lines below
#     # if you want to change them, make sure you also update the lines below!!
#     v_targ_uboone = [-31387.58422, -3316.402543, -60100.2414]

#     # take the vector from the NuMI target to the neutrino vertex
#     df.eval('vec_targ_vtx_X_truth = truth_vtxX - (-31387.58422)', inplace=True)
#     df.eval('vec_targ_vtx_Y_truth = truth_vtxY - (-3316.402543)', inplace=True)
#     df.eval('vec_targ_vtx_Z_truth = truth_vtxZ - (-60100.2414)', inplace=True)
    
#     # get reco shower vector (workaround that is working, for some reason I can't access it if I don't do this workaround, maybe it's somethind to do with the name containing [ ])
#     df.loc[:,'muon_momentum_X_truth'] = df["truth_muonMomentum[0]"]
#     df.loc[:,'muon_momentum_Y_truth'] = df["truth_muonMomentum[1]"]
#     df.loc[:,'muon_momentum_Z_truth'] = df["truth_muonMomentum[2]"]
#     df.loc[:,'muon_energy_truth'] = df["truth_muonMomentum[3]"] # MeV to GeV
    
#     # calculate the norm of the vectors
#     df.eval('norm_vec_targ_vtx_truth = sqrt(vec_targ_vtx_X_truth**2 + vec_targ_vtx_Y_truth**2 + vec_targ_vtx_Z_truth**2)', inplace=True)
#     df.eval('norm_vec_shower_truth = sqrt(muon_momentum_X_truth**2 + muon_momentum_Y_truth**2 + muon_momentum_Z_truth**2)', inplace=True)
    
#     # calculate cos_theta
#     df.eval('truth_cos_theta = ((vec_targ_vtx_X_truth * muon_momentum_X_truth) + (vec_targ_vtx_Y_truth * muon_momentum_Y_truth) + (vec_targ_vtx_Z_truth * muon_momentum_Z_truth) )/(norm_vec_targ_vtx_truth * norm_vec_shower_truth)', inplace=True)

#     return df

In [8]:
def calc_costheta_truth(df):
    # --- NuMI target position ---
    v_targ_uboone = [-31387.58422, -3316.402543, -60100.2414]

    # Vector: target -> neutrino vertex
    df.eval('vec_targ_vtx_X_truth = truth_vtxX - (-31387.58422)', inplace=True)
    df.eval('vec_targ_vtx_Y_truth = truth_vtxY - (-3316.402543)', inplace=True)
    df.eval('vec_targ_vtx_Z_truth = truth_vtxZ - (-60100.2414)', inplace=True)

    # Output momentum columns (filled per event)
    df.loc[:, 'muon_momentum_X_truth'] = np.nan
    df.loc[:, 'muon_momentum_Y_truth'] = np.nan
    df.loc[:, 'muon_momentum_Z_truth'] = np.nan
    df.loc[:, 'muon_energy_truth']     = np.nan  # optional

    def valid_p3(p4):
        p3 = np.asarray(p4[:3], dtype=float)
        if p3.size != 3 or np.any(p3 == -1):
            return False
        return np.linalg.norm(p3) > 0

    for idx, row in df.iterrows():
        pdgs     = row['truth_pdg']            # jagged list/array
        mothers  = row['truth_mother']         # jagged list/array
        mom4_all = row['truth_startMomentum']  # list of [px,py,pz,E]

        chosen = None

        # 1) Primary muon from the neutrino: PDG ±13 AND mother==0
        for i, (pdg, mom) in enumerate(zip(pdgs, mothers)):
            if mom == 0 and pdg in (13, -13):
                if i < len(mom4_all) and valid_p3(mom4_all[i]):
                    chosen = mom4_all[i]
                    break

        # 2) Fallback: any primary (mother==0) that is NOT a muon
        #    (no extra neutrino exclusion needed here per your spec)
        if chosen is None:
            best_mag = -np.inf
            for i, (pdg, mom) in enumerate(zip(pdgs, mothers)):
                if mom != 0 or pdg in (13, -13):
                    continue
                if i >= len(mom4_all) or not valid_p3(mom4_all[i]):
                    continue
                p3 = np.asarray(mom4_all[i][:3], dtype=float)
                pmag = np.linalg.norm(p3)
                if pmag > best_mag:
                    best_mag = pmag
                    chosen = mom4_all[i]

        # Store the chosen momentum (if any)
        if chosen is not None:
            df.at[idx, 'muon_momentum_X_truth'] = float(chosen[0])
            df.at[idx, 'muon_momentum_Y_truth'] = float(chosen[1])
            df.at[idx, 'muon_momentum_Z_truth'] = float(chosen[2])
            if len(chosen) > 3:
                df.at[idx, 'muon_energy_truth'] = float(chosen[3])

    # Norms (unchanged)
    df.eval('norm_vec_targ_vtx_truth = sqrt(vec_targ_vtx_X_truth**2 + vec_targ_vtx_Y_truth**2 + vec_targ_vtx_Z_truth**2)', inplace=True)
    df.eval('norm_vec_shower_truth   = sqrt(muon_momentum_X_truth**2 + muon_momentum_Y_truth**2 + muon_momentum_Z_truth**2)', inplace=True)

    # cos(theta) (unchanged)
    df.eval(
        'truth_cos_theta = (vec_targ_vtx_X_truth * muon_momentum_X_truth + '
        '                   vec_targ_vtx_Y_truth * muon_momentum_Y_truth + '
        '                   vec_targ_vtx_Z_truth * muon_momentum_Z_truth) / '
        '                  (norm_vec_targ_vtx_truth * norm_vec_shower_truth)',
        inplace=True
    )

    return df


In [9]:
#=========================================#
# A Function for calculating multiplicity #
#=========================================#

def calc_particle_multiplicity(filename, df_original):

    # --- open file and variables
    pfeval_particle = ['reco_mother','reco_pdg','reco_Ntrack', 'run', 'subrun', 'event']
    T_PFeval = uproot.open(filename)['wcpselection/T_PFeval']
    DF = T_PFeval.pandas.df(pfeval_particle, flatten=True)

    # --- queries to split into different variables
    df_Neutron = DF.query('((reco_pdg==2112) | (reco_pdg==-2112)) & reco_mother==0')
    DF.loc[:,'countNeutron'] = df_Neutron.groupby(['entry'])['reco_pdg'].transform('count')

    df_Muon = DF.query('((reco_pdg==13) | (reco_pdg==-13)) & reco_mother==0')
    DF.loc[:,'countMuon'] = df_Muon.groupby(['entry'])['reco_pdg'].transform('count')

    df_Kaon = DF.query('((reco_pdg==321) | (reco_pdg==-321)) & reco_mother==0')
    DF.loc[:,'countKaon'] = df_Kaon.groupby(['entry'])['reco_pdg'].transform('count')

    df_Pion = DF.query('((reco_pdg==211) | (reco_pdg==-211)) & reco_mother==0')
    DF.loc[:,'countPion'] = df_Pion.groupby(['entry'])['reco_pdg'].transform('count')

    df_Proton = DF.query('((reco_pdg==2212) | (reco_pdg==-2212)) & reco_mother==0')
    DF.loc[:,'countProton'] = df_Proton.groupby(['entry'])['reco_pdg'].transform('count')

    df_Gamma = DF.query('((reco_pdg==22) | (reco_pdg==-22)) & reco_mother==0')
    DF.loc[:,'countGamma'] = df_Gamma.groupby(['entry'])['reco_pdg'].transform('count')

    df_Electron = DF.query('((reco_pdg==11) | (reco_pdg==-11)) & reco_mother==0')
    DF.loc[:,'countElectron'] = df_Electron.groupby(['entry'])['reco_pdg'].transform('count')

    df_concat = DF

    # --- create an extra column with the max number in the count... column
    df_concat.loc[:,'Num_Neutron'] = df_concat.groupby(['entry'])['countNeutron'].transform('max')
    df_concat.loc[:,'Num_Muon'] = df_concat.groupby(['entry'])['countMuon'].transform('max')
    df_concat.loc[:,'Num_Kaon'] = df_concat.groupby(['entry'])['countKaon'].transform('max')
    df_concat.loc[:,'Num_Proton'] = df_concat.groupby(['entry'])['countProton'].transform('max')
    df_concat.loc[:,'Num_Gamma'] = df_concat.groupby(['entry'])['countGamma'].transform('max')
    df_concat.loc[:,'Num_Electron'] = df_concat.groupby(['entry'])['countElectron'].transform('max')
    df_concat.loc[:,'Num_Pion'] = df_concat.groupby(['entry'])['countPion'].transform('max')

    # --- get the first subentry only --> the same as unflatten the dataframe
    df_concat.groupby('entry').first()

    # --- create the final dataframe with the columns that we want
    df_final = df_concat[['Num_Neutron','Num_Muon','Num_Kaon','Num_Proton','Num_Gamma','Num_Electron', 'Num_Pion', 'run', 'subrun', 'event']].groupby('entry').first()
    df_final = df_final.fillna(0)

    # --- merge with the original dataframe
    df_final = pd.merge(df_original, df_final, left_on=['run','subrun','event'], right_on=['run','subrun','event'], how='left')

    return df_final

In [10]:
#=========================================#
# A Function for calculating multiplicity #
#=========================================#

def calc_particle_multiplicity_truth(filename, df_original):

    # --- open file and variables
    pfeval_particle = ['truth_mother','truth_pdg','truth_Ntrack', 'run', 'subrun', 'event']
    T_PFeval = uproot.open(filename)['wcpselection/T_PFeval']
    DF = T_PFeval.pandas.df(pfeval_particle, flatten=True)

    # --- queries to split into different variables
    df_Neutron = DF.query('((truth_pdg==2112) | (truth_pdg==-2112)) & truth_mother==0')
    DF.loc[:,'countNeutron'] = df_Neutron.groupby(['entry'])['truth_pdg'].transform('count')

    df_Muon = DF.query('((truth_pdg==13) | (truth_pdg==-13)) & truth_mother==0')
    DF.loc[:,'countMuon'] = df_Muon.groupby(['entry'])['truth_pdg'].transform('count')

    df_Kaon = DF.query('((truth_pdg==321) | (truth_pdg==-321)) & truth_mother==0')
    DF.loc[:,'countKaon'] = df_Kaon.groupby(['entry'])['truth_pdg'].transform('count')

    df_Pion = DF.query('((truth_pdg==211) | (truth_pdg==-211)) & truth_mother==0')
    DF.loc[:,'countPion'] = df_Pion.groupby(['entry'])['truth_pdg'].transform('count')

    df_Proton = DF.query('((truth_pdg==2212) | (truth_pdg==-2212)) & truth_mother==0')
    DF.loc[:,'countProton'] = df_Proton.groupby(['entry'])['truth_pdg'].transform('count')

    df_Gamma = DF.query('((truth_pdg==22) | (truth_pdg==-22)) & truth_mother==0')
    DF.loc[:,'countGamma'] = df_Gamma.groupby(['entry'])['truth_pdg'].transform('count')

    df_Electron = DF.query('((truth_pdg==11) | (truth_pdg==-11)) & truth_mother==0')
    DF.loc[:,'countElectron'] = df_Electron.groupby(['entry'])['truth_pdg'].transform('count')

    df_concat = DF

    # --- create an extra column with the max number in the count... column
    df_concat.loc[:,'Truth_Num_Neutron'] = df_concat.groupby(['entry'])['countNeutron'].transform('max')
    df_concat.loc[:,'Truth_Num_Muon'] = df_concat.groupby(['entry'])['countMuon'].transform('max')
    df_concat.loc[:,'Truth_Num_Kaon'] = df_concat.groupby(['entry'])['countKaon'].transform('max')
    df_concat.loc[:,'Truth_Num_Proton'] = df_concat.groupby(['entry'])['countProton'].transform('max')
    df_concat.loc[:,'Truth_Num_Gamma'] = df_concat.groupby(['entry'])['countGamma'].transform('max')
    df_concat.loc[:,'Truth_Num_Electron'] = df_concat.groupby(['entry'])['countElectron'].transform('max')
    df_concat.loc[:,'Truth_Num_Pion'] = df_concat.groupby(['entry'])['countPion'].transform('max')

    # --- get the first subentry only --> the same as unflatten the dataframe
    df_concat.groupby('entry').first()

    # --- create the final dataframe with the columns that we want
    df_final = df_concat[['Truth_Num_Neutron','Truth_Num_Muon','Truth_Num_Kaon','Truth_Num_Proton','Truth_Num_Gamma','Truth_Num_Electron', 'Truth_Num_Pion', 'run', 'subrun', 'event']].groupby('entry').first()
    df_final = df_final.fillna(0)

    # --- merge with the original dataframe
    df_final = pd.merge(df_original, df_final, left_on=['run','subrun','event'], right_on=['run','subrun','event'], how='left')

    return df_final

In [11]:
#==================================================================================================#
# A Function for calculating kinetic energy of most energetic proton and sum of all proton energies #
#==================================================================================================#

def add_truth_proton_kinetic_energy_summaries(df):
    """
    For each event in df, compute:
      - Truth_Proton_Tmax : highest kinetic energy among primary protons (mother==0, |pdg|==2212), in MeV
      - Truth_Proton_Tsum : sum of kinetic energies of all such protons, in MeV

    Assumes truth_startMomentum[i][3] is in GeV (total energy).
    Subtracts proton rest mass (0.938272 GeV).
    """

    proton_mass_GeV = 0.938272
    Tmax_list = []
    Tsum_list = []

    for idx, row in df.iterrows():
        truth_pdg           = row['truth_pdg']
        truth_mother        = row['truth_mother']
        truth_startMomentum = row['truth_startMomentum']

        Tmax = 0.0
        Tsum = 0.0

        # Loop over particles in this event
        for i, pdg in enumerate(truth_pdg):
            # Primary proton: mother==0 and |pdg|==2212
            if truth_mother[i] == 0 and abs(pdg) == 2212:
                try:
                    E_GeV = float(truth_startMomentum[i][3])  # GeV
                except Exception:
                    continue

                if E_GeV == -1:
                    continue  # skip invalid

                # Kinetic energy in GeV
                T_GeV = E_GeV - proton_mass_GeV
                if T_GeV < 0:  # guard against rounding errors
                    T_GeV = 0.0

                # Convert to MeV
                T_MeV = T_GeV * 1000.0

                Tsum += T_MeV
                if T_MeV > Tmax:
                    Tmax = T_MeV

        Tmax_list.append(Tmax)
        Tsum_list.append(Tsum)

    df = df.copy()
    df['Truth_Proton_Tmax'] = Tmax_list
    df['Truth_Proton_Tsum'] = Tsum_list
    return df


### Event Selection Functions

In [12]:
#===============================================================#
# A Function for clearing non-physical values for reco vertices #
#===============================================================#


def reco_nu_vtx_val_clearing(df):
    df_ = df.query('reco_nuvtxX != -1 & reco_nuvtxY != -1 & reco_nuvtxZ != -1')
    return df_

In [13]:
#===========================================#
# A Function for Generic Neutrino Selection #
#===========================================#


def gen_nu_selec(df):
    selec_df = df[df.numu_cc_flag >= 0]
    return selec_df

In [14]:
#=================================#
# A Function for numuCC selection #
#=================================#
def numuCC(df):
    selec_df = df[df.numu_cc_flag >= 1]
    return selec_df

In [15]:
#======================================================#
# Add Geometry vars containing Fiducial Vol Boundaries #
#======================================================#
fv_xmin = tpc_xmin+7.55
fv_xmax = tpc_xmax-3.
fv_ymin = tpc_ymin+3.
fv_ymax = tpc_ymax-17.47
fv_zmin = tpc_zmin+15
fv_zmax = tpc_zmax-3.

In [16]:
#===========================================#
#  A Function for Fiducial Volume Selection #
#===========================================#


def apply_inFV(df):
    df_ = df[((df.reco_nuvtxX>(fv_xmin)) & (df.reco_nuvtxX<(fv_xmax))) &
             ((df.reco_nuvtxY>(fv_ymin)) & (df.reco_nuvtxY<(fv_ymax))) & 
             ((df.reco_nuvtxZ>(fv_zmin)) & (df.reco_nuvtxZ<(fv_zmax)))]    
    return df_

In [17]:
#=========================================================#
#  A Function for Containment Selection (Fully contained) #
#=========================================================#


def apply_isFC(df):
    selec_df = df[df.match_isFC==1]
    return selec_df

In [18]:
#============================================================#
# A Function for Containment Selection (Partially contained) #
#============================================================#


def apply_notFC(df):
    selec_df = df[df.match_isFC==0]
    return selec_df

In [19]:
#=======================================================================================================================#
# A Function for selecting at least a muon track in the evt, get rid of muon_E = -1, impossible, means muon not reco-ed #
#=======================================================================================================================#


def apply_muCut(df):
    df_ = df[df.muon_energy != -1]
    return df_

In [20]:
#============================================================#
# A Function for selecting problematic range for closer look #
#============================================================#

def problem_Enu_range(df):
    selec_df = df.query('kine_reco_Enu>=150. & kine_reco_Enu<=650.')
    return selec_df

In [21]:
#=======================================================#
# A Function for selecting events with bad angle values #
#=======================================================#

def problem_angle_val(df):
    df_ = df.query('cos_theta<-0.8 & cos_theta>-0.84')
    return df_

In [22]:
# #======================================================#
# # A Function for selecting reco Michel electron events #
# #======================================================#

# def select_reco_michel_with_mu(df):
#     selected_indices = []
#     bad_mother_count = 0  # Counter for bad mother indices

#     for idx, row in df.iterrows():
#         reco_pdg = row['reco_pdg']
#         reco_mother = row['reco_mother']
#         reco_id = row['reco_id']
#         reco_startMomentum = row['reco_startMomentum']

#         for i, pdg in enumerate(reco_pdg):
#             if pdg == 11 or pdg == -11:  # electron or positron
#                 mother_id = reco_mother[i]
#                 mother_index = np.where(reco_id == mother_id)[0]
#                 mother_pdg = reco_pdg[mother_index]

#                 # Enforce proper charge correlation and primary muon condition.
#                 if (
#                     ((pdg == 11 and mother_pdg == 13) or (pdg == -11 and mother_pdg == -13)) and
#                     reco_mother[mother_index] == 0
#                 ):
#                     energy = reco_startMomentum[i][3]
#                     if energy <= 0.07:  # 50 MeV = 0.05 GeV
#                         selected_indices.append(idx)
#                         break  # Found valid Michel for this event

#     print(f"Number of bad mother indices: {bad_mother_count}")
#     return df.loc[selected_indices].reset_index(drop=True)

In [23]:
# def select_reco_michel_with_mu(df):
#     selected_indices = []
#     bad_mother_count = 0  # Counter for bad mother indices

#     total_weight_initial = sum(df['weight_ubtune'])
#     print(f"Initial total weight: {total_weight_initial:.3f}")

#     step1_indices = []
#     step2_indices = []
#     step3_indices = []

#     for idx, row in df.iterrows():
#         reco_pdg = row['reco_pdg']
#         reco_mother = row['reco_mother']
#         reco_id = row['reco_id']
#         reco_startMomentum = row['reco_startMomentum']

#         for i, pdg in enumerate(reco_pdg):
#             if pdg == 11 or pdg == -11:
#                 step1_indices.append(idx)  # Passed e± selection

#                 mother_id = reco_mother[i]
#                 mother_index = np.where(reco_id == mother_id)[0]

#                 if len(mother_index) == 0:
#                     bad_mother_count += 1
#                     break

#                 mother_pdg = reco_pdg[mother_index]

#                 if (
#                     ((pdg == 11 and mother_pdg == 13) or (pdg == -11 and mother_pdg == -13)) and
#                     reco_mother[mother_index] == 0
#                 ):
#                     step2_indices.append(idx)  # Passed muon match and mother==0

#                     energy = reco_startMomentum[i][3]
#                     if energy <= 0.07:
#                         step3_indices.append(idx)  # Passed energy cut
#                         selected_indices.append(idx)
#                         break

#     # Print weights at each step
#     step1_indices = list(set(step1_indices))
#     step2_indices = list(set(step2_indices))
#     step3_indices = list(set(step3_indices))

#     print(f"After e± selection: {df.loc[step1_indices, 'weight_ubtune'].sum():.3f}")
#     print(f"After muon match + mother==0: {df.loc[step2_indices, 'weight_ubtune'].sum():.3f}")
#     print(f"After energy <= 70 MeV: {df.loc[step3_indices, 'weight_ubtune'].sum():.3f}")
#     print(f"Number of bad mother indices: {bad_mother_count}")

#     return df.loc[selected_indices].reset_index(drop=True)


In [24]:
def select_reco_michel_with_mu(df):
    selected_indices = []
    bad_mother_count = 0

    total_weight_initial = df['weight_ubtune'].sum()
    print(f"Initial total weight: {total_weight_initial:.3f}")

    step1_indices = []
    step2_indices = []
    step3_indices = []

    for idx, row in df.iterrows():
        reco_pdg = row['reco_pdg']
        reco_mother = row['reco_mother']
        reco_id = row['reco_id']
        reco_startMomentum = row['reco_startMomentum']
        reco_startXYZT = row['reco_startXYZT']
        reco_endXYZT = row['reco_endXYZT']

        for i, pdg in enumerate(reco_pdg):
            if pdg in (13, -13) and reco_mother[i] == 0:  # primary muon
                muon_end = reco_endXYZT[i][:3]  # x, y, z

                step1_indices.append(idx)

                # Loop over all other particles to find Michel candidate
                for j, cand_pdg in enumerate(reco_pdg):
                    if j == i:
                        continue

                    start = np.array(reco_startXYZT[j][:3])
                    end = np.array(reco_endXYZT[j][:3])
                    dist_to_muon_end = np.linalg.norm(np.array(muon_end) - start)
                    candidate_length = np.linalg.norm(end - start)
                    energy = reco_startMomentum[j][3]

                    if dist_to_muon_end <= 5.0:
                        step2_indices.append(idx)

                        if candidate_length <= 10.0 and energy <= 0.07:
                            step3_indices.append(idx)
                            selected_indices.append(idx)
                            break  # Only one Michel candidate per event

                break  # Only one primary muon per event

    # Drop duplicates
    step1_indices = list(set(step1_indices))
    step2_indices = list(set(step2_indices))
    step3_indices = list(set(step3_indices))

    print(f"After primary muon selection: {df.loc[step1_indices, 'weight_ubtune'].sum():.3f}")
    print(f"After spatial + energy selection: {df.loc[step2_indices, 'weight_ubtune'].sum():.3f}")
    print(f"After candidate length and energy cut: {df.loc[step3_indices, 'weight_ubtune'].sum():.3f}")
    print(f"Number of bad mother indices: {bad_mother_count}")

    return df.loc[selected_indices].reset_index(drop=True)


In [25]:
# #=======================================================#
# # A Function for selecting truth Michel electron events #
# #=======================================================#

# def select_truth_michel_with_mu(df):
#     selected_indices = []
#     bad_mother_count = 0  # Counter for bad mother indices

#     for idx, row in df.iterrows():
#         truth_pdg = row['truth_pdg']
#         truth_mother = row['truth_mother']
#         truth_id = row['truth_id']
#         truth_startMomentum = row['truth_startMomentum']

#         for i, pdg in enumerate(truth_pdg):
#             if pdg == 11 or pdg == -11:  # electron or positron
#                 mother_id = truth_mother[i]
#                 mother_index = np.where(truth_id == mother_id)[0]
#                 mother_pdg = truth_pdg[mother_index]

#                 # Enforce proper charge correlation and primary muon condition.
#                 if (
#                     ((pdg == 11 and mother_pdg == 13) or (pdg == -11 and mother_pdg == -13)) and
#                     truth_mother[mother_index] == 0
#                 ):
#                     energy = truth_startMomentum[i][3]
#                     if energy <= 0.07:  # 50 MeV = 0.05 GeV
#                         selected_indices.append(idx)
#                         break  # Found valid Michel for this event

#     print(f"Number of bad mother indices: {bad_mother_count}")
#     return df.loc[selected_indices].reset_index(drop=True)

In [26]:
def select_truth_michel_with_mu(df):
    selected_indices = []
    bad_mother_count = 0

    total_weight_initial = df['weight_ubtune'].sum()
    print(f"Initial total weight: {total_weight_initial:.3f}")

    step1_indices = []
    step2_indices = []
    step3_indices = []

    for idx, row in df.iterrows():
        truth_pdg = row['truth_pdg']
        truth_mother = row['truth_mother']
        truth_id = row['truth_id']
        truth_startMomentum = row['truth_startMomentum']
        truth_startXYZT = row['truth_startXYZT']
        truth_endXYZT = row['truth_endXYZT']

        for i, pdg in enumerate(truth_pdg):
            if pdg in (13, -13) and truth_mother[i] == 0:  # primary muon
                muon_end = truth_endXYZT[i][:3]

                step1_indices.append(idx)

                for j, cand_pdg in enumerate(truth_pdg):
                    if j == i:
                        continue

                    start = np.array(truth_startXYZT[j][:3])
                    end = np.array(truth_endXYZT[j][:3])
                    dist_to_muon_end = np.linalg.norm(np.array(muon_end) - start)
                    candidate_length = np.linalg.norm(end - start)
                    energy = truth_startMomentum[j][3]

                    if dist_to_muon_end <= 5.0:
                        step2_indices.append(idx)

                        if candidate_length <= 10.0 and energy <= 0.07:
                            step3_indices.append(idx)
                            selected_indices.append(idx)
                            break  # One Michel candidate per event

                break  # Only one primary muon per event

    # Drop duplicates
    step1_indices = list(set(step1_indices))
    step2_indices = list(set(step2_indices))
    step3_indices = list(set(step3_indices))

    print(f"After primary muon selection: {df.loc[step1_indices, 'weight_ubtune'].sum():.3f}")
    print(f"After spatial + energy selection: {df.loc[step2_indices, 'weight_ubtune'].sum():.3f}")
    print(f"After candidate length and energy cut: {df.loc[step3_indices, 'weight_ubtune'].sum():.3f}")
    print(f"Number of bad mother indices: {bad_mother_count}")

    return df.loc[selected_indices].reset_index(drop=True)


In [27]:
def select_truth_michel_positrons(df):
    selected_indices = []
    bad_mother_count = 0  # Counter for bad mother indices

    for idx, row in df.iterrows():
        truth_pdg = row['truth_pdg']
        truth_mother = row['truth_mother']
        truth_id = row['truth_id']
        truth_startMomentum = row['truth_startMomentum']

        for i, pdg in enumerate(truth_pdg):
            if pdg == -11:  # electron or positron
                mother_id = truth_mother[i]
                mother_index = np.where(truth_id == mother_id)[0]
                mother_pdg = truth_pdg[mother_index]

                # Enforce proper charge correlation and primary muon condition.
                if (
                    (pdg == -11 and mother_pdg == -13) and
                    truth_mother[mother_index] == 0
                ):
                    energy = truth_startMomentum[i][3]
                    if energy <= 0.07:  # 50 MeV = 0.05 GeV
                        selected_indices.append(idx)
                        break  # Found valid Michel for this event

    print(f"Number of bad mother indices: {bad_mother_count}")
    return df.loc[selected_indices].reset_index(drop=True)

In [28]:
def select_truth_michel_electrons(df):
    selected_indices = []
    bad_mother_count = 0  # Counter for bad mother indices

    for idx, row in df.iterrows():
        truth_pdg = row['truth_pdg']
        truth_mother = row['truth_mother']
        truth_id = row['truth_id']
        truth_startMomentum = row['truth_startMomentum']

        for i, pdg in enumerate(truth_pdg):
            if pdg == 11:  # electron or positron
                mother_id = truth_mother[i]
                mother_index = np.where(truth_id == mother_id)[0]
                mother_pdg = truth_pdg[mother_index]

                # Enforce proper charge correlation and primary muon condition.
                if (
                    (pdg == 11 and mother_pdg == 13) and
                    truth_mother[mother_index] == 0
                ):
                    energy = truth_startMomentum[i][3]
                    if energy <= 0.07:  # 50 MeV = 0.05 GeV
                        selected_indices.append(idx)
                        break  # Found valid Michel for this event

    print(f"Number of bad mother indices: {bad_mother_count}")
    return df.loc[selected_indices].reset_index(drop=True)

In [29]:
#========================================#
# A Function for applying selection cuts #
#========================================#


def applyCuts(label, dfData, dfEXT, dfOverlay, dfDirt, weight, print_lengths, POT_data, plotCondition='default'):
    print('\n---------- Applying: %s \n' % label)


    #—————————————#
    # Apply cuts  #
    #—————————————#


    if (label=='None'):                 # No selection applied
        df_data = dfData
        df_mc = dfOverlay
        df_ext = dfEXT
        df_dirt = dfDirt
    # elif (label=='recoVtxTrim'):         # Trim down badly reconstructed vertex values
    #     df_data = reco_nu_vtx_val_clearing(dfData)
    #     df_mc = reco_nu_vtx_val_clearing(dfOverlay)
    #     df_ext = reco_nu_vtx_val_clearing(dfEXT)
    #     df_dirt = reco_nu_vtx_val_clearing(dfDirt)
    elif (label=='genNuSelection'):      # Generic Neutrino Selection and numuCC selection
        df_data = gen_nu_selec(dfData)
        df_mc = gen_nu_selec(dfOverlay)
        df_ext = gen_nu_selec(dfEXT)
        df_dirt = gen_nu_selec(dfDirt)
    elif (label=='fiducialVol'):         # Fiducial volume selection
        df_data = apply_inFV(dfData)
        df_mc = apply_inFV(dfOverlay)
        df_ext = apply_inFV(dfEXT)
        df_dirt = apply_inFV(dfDirt)
    elif (label=='numuCCCut'):      # Generic Neutrino Selection and numuCC selection
        df_data = numuCC(dfData)
        df_mc = numuCC(dfOverlay)
        df_ext = numuCC(dfEXT)
        df_dirt = numuCC(dfDirt)
    elif (label=='muonCut'):      # Select events that has at least 1 reco-ed muon track
        df_data = apply_muCut(dfData)
        df_mc = apply_muCut(dfOverlay)
        df_ext = apply_muCut(dfEXT)
        df_dirt = apply_muCut(dfDirt)
    elif (label=='FC'):
        df_data = apply_isFC(dfData)
        df_mc = apply_isFC(dfOverlay)
        df_ext = apply_isFC(dfEXT)
        df_dirt = apply_isFC(dfDirt)
    elif (label=='PC'):
        df_data = apply_notFC(dfData)
        df_mc = apply_notFC(dfOverlay)
        df_ext = apply_notFC(dfEXT)
        df_dirt = apply_notFC(dfDirt)
    elif (label=='problematicEnu'):
        df_data = problem_Enu_range(dfData)
        df_mc = problem_Enu_range(dfOverlay)
        df_ext = problem_Enu_range(dfEXT)
        df_dirt = problem_Enu_range(dfDirt)
    elif (label=='truthMichel'):
        df_data = dfData
        df_ext = dfEXT
        df_mc = select_truth_michel_with_mu(dfOverlay)
        df_dirt = select_truth_michel_with_mu(dfDirt)
    elif (label=='truthMichelPositrons'):
        df_data = dfData
        df_ext = dfEXT
        df_mc = select_truth_michel_positrons(dfOverlay)
        df_dirt = select_truth_michel_positrons(dfDirt)

    elif (label=='truthMichelElectrons'):
        df_data = dfData
        df_ext = dfEXT
        df_mc = select_truth_michel_electrons(dfOverlay)
        df_dirt = select_truth_michel_electrons(dfDirt)

    elif (label=='recoMichel'):
        df_data = select_reco_michel_with_mu(dfData)
        df_ext = select_reco_michel_with_mu(dfEXT)
        df_mc = select_reco_michel_with_mu(dfOverlay)
        df_dirt = select_reco_michel_with_mu(dfDirt)


    if(print_lengths==True):
        print('[DATA] %7i entries' % sum(df_data[weight]))
        print('[EXT]  %7i entries' % sum(df_ext[weight]))
        print('[MC]   %7i entries' % sum(df_mc[weight]))
        print('[DIRT]   %7i entries' % sum(df_dirt[weight]))

    # make stacked histograms
    plots_for_SelectionCuts(df_data, df_ext, df_mc, df_dirt, weight, label, POT_data, plotCondition)
        
    # return the updated dataframes
    return df_data, df_ext, df_mc, df_dirt

### MC Topology Classification Functions

In [30]:
def isCosmic(df):
    df_ = df[((df.match_completeness_energy <= df.truth_energyInside * 0.1) \
               | (df.truth_energyInside <= 0))]
    # print('Number of rows BEFORE query: %s' % len(df))
    # df_ = df.query('match_completeness_energy <= truth_energyInside*0.1 \
    #                       | truth_energyInside <= 0')
    # print('Number of rows AFTER query: %s' % len(df_))
    return df_

def notCosmic(df):
    df_ = df[~((df.match_completeness_energy <= df.truth_energyInside * 0.1) \
               | (df.truth_energyInside <= 0))]
    # print('Number of rows BEFORE query: %s' % len(df))
    # df_ = df.query('match_completeness_energy > truth_energyInside*0.1 \
    #                & truth_energyInside > 0')
    # print('Number of rows AFTER query: %s' % len(df_))
    return df_

def isOutFV(df):
    df_ = df[(df.truth_vtxInside==0)]  
    # print('Number of rows BEFORE query: %s' % len(df))
    # df_ = df.query('truth_vtxInside==0')                          # vtx outFV
    # print('Number of rows AFTER query: %s' % len(df_))
    return df_

def notOutFV(df):
    df_ = df[(df.truth_vtxInside != 0)]
    return df_

def isNue_NuebarCC(df):
    df_ = df[((df.truth_nuPdg==12) | (df.truth_nuPdg==-12))    
             & (df.truth_isCC==1)] 
    # df_ = df.query('truth_nuPdg==12 | truth_nuPdg==-12')                            # nue/nue_barCC
    return df_

def isNumuCC(df):
    df_ = df[(df.truth_nuPdg==14) &                             
             (df.truth_isCC==1)] 
    # df_ = df.query('truth_nuPdg==14 & \
    #                truth_isCC==1')                               # numu CC
    return df_

def isNumu_barCC(df):
    df_ = df[(df.truth_nuPdg==-14) &                             
             (df.truth_isCC==1)]  
    # df_ = df.query('truth_nuPdg==-14 & \
    #                truth_isCC==1')                               # numu_barCC
    return df_

def isNCpi(df):
    mask_pi = np.array([211 in array for array in np.abs(np.array(df.truth_pdg))])
    mask_isNC = df.truth_isCC==0                                 # NC with charged pions
    df_ = df[mask_isNC & mask_pi]
    return df_
    
    return df_

def isNC(df):
    mask_no_pi = np.array([not 211 in array for array in np.abs(np.array(df.truth_pdg))])
    mask_isNC = df.truth_isCC==0
    df_ = df[mask_isNC & mask_no_pi]                             # NC with no pions
    return df_

In [31]:
#=============================================#
#  Functions setting vars to plot after cuts  #
#=============================================#

 #uneven reco nu E binning bins:
bins_reco_nuE = []
for i in range(12):
    bins_reco_nuE.append(150+94*i)
bins_reco_nuE.append(1280.)
bins_reco_nuE.append(1380.)
bins_reco_nuE.append(1500.)
bins_reco_nuE.append(1650.)
bins_reco_nuE.append(1840.)
bins_reco_nuE.append(2090.)
bins_reco_nuE.append(2500.)

#bins_reco_nuE = np.array(bins_reco_nuE, dtype=np.float32)   

def plots_for_SelectionCuts(df_data, df_ext, df_overlay, df_dirt, weight, label, POT_data, condition='default'):
    
    # this function makes all the plots that you want to plot throughout the pre-selection cuts
    
    if (condition=='default'):
        # --- MC/DATA comparison
        # x-dir reco neutrino vertex (also plot zoomed in edges)
        mc_data_stacked_hist(df_data, df_ext, df_overlay, df_dirt, "reco_nuvtxX", weight, \
                            "plots/%s_nuvtxX" % label, "Reco neutrino Vertex X [cm]", \
                            tpc_xmin, tpc_xmax, 25, POT_data)
        mc_data_stacked_hist(df_data, df_ext, df_overlay, df_dirt, "reco_nuvtxX", weight, \
                            "plots/%s_nuvtxX_zoom_anode" % label, "Reco neutrino Vertex X [cm]", \
                            tpc_xmin, tpc_xmin+20., 25, POT_data)
        mc_data_stacked_hist(df_data, df_ext, df_overlay, df_dirt, "reco_nuvtxX", weight, \
                            "plots/%s_nuvtxX_zoom_cathode" % label, "Reco neutrino Vertex X [cm]", \
                            tpc_xmax-20., tpc_xmax, 25, POT_data)
        
        # y-dir reco neutrino vertex (also plot zoomed in edges)
        mc_data_stacked_hist(df_data, df_ext, df_overlay, df_dirt, "reco_nuvtxY", weight, \
                            "plots/%s_nuvtxY" % label, "Reco neutrino Vertex Y [cm]", \
                            tpc_ymin, tpc_ymax, 25, POT_data)
        mc_data_stacked_hist(df_data, df_ext, df_overlay, df_dirt, "reco_nuvtxY", weight, \
                            "plots/%s_nuvtxY_zoom_bottom" % label, "Reco neutrino Vertex Y [cm]", \
                            tpc_ymin, tpc_ymin+15., 25, POT_data)
        mc_data_stacked_hist(df_data, df_ext, df_overlay, df_dirt, "reco_nuvtxY", weight, \
                            "plots/%s_nuvtxY_zoom_top" % label, "Reco neutrino Vertex Y [cm]", \
                            tpc_ymax-30., tpc_ymax, 25, POT_data)
        
        # z-dir reco neutrino vertex (also plot zoomed in edges)
        mc_data_stacked_hist(df_data, df_ext, df_overlay, df_dirt, "reco_nuvtxZ", weight, \
                            "plots/%s_nuvtxZ" % label, "Reco neutrino Vertex Z [cm]", \
                            tpc_zmin, tpc_zmax, 25, POT_data)
        mc_data_stacked_hist(df_data, df_ext, df_overlay, df_dirt, "reco_nuvtxZ", weight, \
                            "plots/%s_nuvtxZ_zoom_start" % label, "Reco neutrino Vertex Z [cm]", \
                            tpc_zmin, tpc_zmin+20., 25, POT_data)
        mc_data_stacked_hist(df_data, df_ext, df_overlay, df_dirt, "reco_nuvtxZ", weight, \
                            "plots/%s_nuvtxZ_zoom_end" % label, "Reco neutrino Vertex Z [cm]", \
                            tpc_zmax-20., tpc_zmax, 25, POT_data)

        # Flash Time (temporarily used as 1 bin histo)
        mc_data_stacked_hist(df_data, df_ext, df_overlay, df_dirt, "reco_nuvtxY", weight, \
                            "plots/%s_flash_time" % label, "1-Bin (A.U.)", \
                            -10000, 10000, 1, POT_data)

        # 1 bin histo with no category, data vs rest
        mc_data_stacked_hist_no_cat(df_data, df_ext, df_overlay, df_dirt, "reco_nuvtxY", weight, \
                            "plots/%s_kine_reco_Enu_no_cat" % label, "1-Bin (A.U.)", \
                            -10000, 10000, 1, POT_data)

        # reco neutrino energy

        mc_data_stacked_hist(df_data, df_ext, df_overlay, df_dirt, "kine_reco_Enu", weight, \
                            "plots/%s_kine_reco_Enu" % label, "Reco Neutrino Energy [MeV]",\
                            150, 2500, bins_reco_nuE, POT_data)
        

        # numuCC BDT var from WC
        mc_data_stacked_hist(df_data, df_ext, df_overlay, df_dirt, "numu_cc_flag", weight, \
                            "plots/%s_numu_cc_flag" % label, r"$\nu_{\mu}$ CC Score (AU)",\
                            -1, 1, 10, POT_data)
    
        # calculated theta angle b/w beam and muon
        mc_data_stacked_hist(df_data, df_ext, df_overlay, df_dirt, "cos_theta", weight, \
                            "plots/%s_cos_theta" % label, r"cos($\theta$)",\
                            -1.2, 1.2, 25, POT_data)
        
        # calculated theta angle b/w beam and muon
        mc_data_stacked_hist(df_data, df_ext, df_overlay, df_dirt, "numu_cc_3_max_muon_length", weight, \
                            "plots/%s_numu_cc_3_max_muon_length" % label, "Maximal Primary Muon Track Length (cm)",\
                            0, 500, 25, POT_data)
        
        mc_data_stacked_hist(df_data, df_ext, df_overlay, df_dirt, "cosmict_2_dQ_dx_end", weight, \
                            "plots/%s_cosmict_2_dQ_dx_end" % label, "dQ/dx at the end of the single track candidate (MeV/cm)",\
                            0, 5, 25, POT_data)
        
        # plot relevant WC LEM vars
        mc_data_stacked_hist(df_data, df_ext, df_overlay, df_dirt, "lem_shower_main_length", weight, \
                            "plots/%s_lem_shower_main_length" % label, "LEM Shower Main Length (cm)",\
                            0, 0.3, 25, POT_data)
        
        mc_data_stacked_hist(df_data, df_ext, df_overlay, df_dirt, "lem_n_3seg", weight, \
                            "plots/%s_lem_n_3seg" % label, "LEM - Number of Clusters in primary shower with at least 3 segs",\
                            0, 10, 10, POT_data)
        mc_data_stacked_hist(df_data, df_ext, df_overlay, df_dirt, "lem_e_charge", weight, \
                            "plots/%s_lem_e_charge" % label, "LEM Total Energy of the Primary Shower",\
                            0, 100, 10, POT_data)
        
        mc_data_stacked_hist(df_data, df_ext, df_overlay, df_dirt, "lem_e_dQdx", weight, \
                            "plots/%s_lem_e_dQdx" % label, "LEM dQ/dx of the Primary Shower",\
                            0, 0.4, 50, POT_data)
        
        mc_data_stacked_hist(df_data, df_ext, df_overlay, df_dirt, "lem_shower_num_main_segs", weight, \
                            "plots/%s_lem_shower_num_main_segs" % label, "LEM - Number of Segments in the Primary Shower",\
                            0, 8, 10, POT_data)


        
    if (condition=='truthMichel') or (condition=='truthMichelPositrons') or (condition=='truthMichelElectrons'):

        mc_data_stacked_hist_truth(df_overlay, df_dirt, "reco_nuvtxX", weight, "Reco neutrino Vertex X [cm]", \
                            tpc_xmin, tpc_xmax, 25)
        
        # y-dir reco neutrino vertex (also plot zoomed in edges)
        mc_data_stacked_hist_truth(df_overlay, df_dirt, "reco_nuvtxY", weight, "Reco neutrino Vertex Y [cm]", \
                            tpc_ymin, tpc_ymax, 25)
        
        # z-dir reco neutrino vertex (also plot zoomed in edges)
        mc_data_stacked_hist_truth(df_overlay, df_dirt, "reco_nuvtxZ", weight, "Reco neutrino Vertex Z [cm]", \
                            tpc_zmin, tpc_zmax, 25)


        # Flash Time (temporarily used as 1 bin histo)
        mc_data_stacked_hist_truth(df_overlay, df_dirt, "reco_nuvtxY", weight, "1-Bin", \
                            -10000, 10000, 1)
    
        # reco neutrino energy

        mc_data_stacked_hist_truth(df_overlay, df_dirt, "kine_reco_Enu", weight, "Reco Neutrino Energy [MeV]",\
                            150, 2500, bins_reco_nuE)
        
    
        # calculated theta angle b/w beam and muon
        mc_data_stacked_hist_truth(df_overlay, df_dirt, "cos_theta", weight, r"cos($\theta$)",\
                            -1.2, 1.2, 25)
        
        mc_data_stacked_hist_truth(df_overlay, df_dirt, "numu_cc_3_max_muon_length", weight, "Maximal Primary Muon Track Length (cm)",\
                            0, 500, 25)
        
        mc_data_stacked_hist_truth(df_overlay, df_dirt, "cosmict_2_dQ_dx_end", weight, "dQ/dx at the end of the single track candidate (MeV/cm)",\
                            0, 5, 25)
        
        
        # mc_data_stacked_hist(df_data, df_ext, df_overlay, df_dirt, "Num_Proton", \
        #                      weight, "plots/%s_Num_Proton" % label, r"Reco Proton Multiplicity",\
        #                     0, 10, 10, POT_data)
        
        # mc_data_stacked_hist(df_data, df_ext, df_overlay, df_dirt, "Num_Muon", \
        #                      weight, "plots/%s_Num_Muon" % label, r"Reco Muon Multiplicity",\
        #                     0, 10, 10, POT_data)
        
        # mc_data_stacked_hist(df_data, df_ext, df_overlay, df_dirt, "Num_Neutron", \
        #                      weight, "plots/%s_Num_Neutron" % label, r"Reco Neutron Multiplicity",\
        #                     0, 10, 10, POT_data)
        
        # mc_data_stacked_hist(df_data, df_ext, df_overlay, df_dirt, "Num_Gamma", \
        #                      weight, "plots/%s_Num_Gamma" % label, r"Reco Photon Multiplicity",\
        #                     0, 10, 10, POT_data)
        
        # mc_data_stacked_hist(df_data, df_ext, df_overlay, df_dirt, "Num_Kaon", \
        #                      weight, "plots/%s_Num_Kaon" % label, r"Reco Kaon Multiplicity",\
        #                     0, 10, 10, POT_data)
        
        # mc_data_stacked_hist(df_data, df_ext, df_overlay, df_dirt, "Num_Electron", \
        #                      weight, "plots/%s_Num_Electron" % label, r"Reco Electron Multiplicity",\
        #                     0, 10, 10, POT_data)
    



In [32]:
#=============================================#
#  A Function for Plotting Stacked Histograms #
#=============================================#


def mc_data_stacked_hist(df_data, df_ext, df_mc, df_dirt, var, weight, plot_png_name, xaxis, xmin, \
                         xmax, nbins, pot_data):
    #————————————————————————————————————————————————#
    # Restrict the range of df in the plotting range #
    #————————————————————————————————————————————————#
    
    
    # df_data = df_data[(df_data[var]>=xmin) & (df_data[var]<=xmax)]
    # df_ext = df_ext[(df_ext[var]>=xmin) & (df_ext[var]<=xmax)]
    # df_mc = df_mc[(df_mc[var]>=xmin) & (df_mc[var]<=xmax)]
    # df_dirt = df_dirt[(df_dirt[var]>=xmin) & (df_dirt[var]<=xmax)]
    
   
    
    #————————————————————————————————————————————#
    # Get the number of entries for each dataset #
    #————————————————————————————————————————————#
    
    
    n_data = sum(df_data[weight])
    n_ext = sum(df_ext[weight])
    n_mc = sum(df_mc[weight])

    #———————————————————————————————————#
    # Classify mc df into each topology #
    #———————————————————————————————————#
    
    
    # cosmic
    df_cosmic = isCosmic(df_mc)
    n_cosmic = sum(df_cosmic[weight])
    df_mc = notCosmic(df_mc)

    # outFV
    df_outFV = isOutFV(df_mc)
    n_outFV = sum(df_outFV[weight])
    df_mc = notOutFV(df_mc)

    # numuCC
    df_numuCC = isNumuCC(df_mc)
    n_numuCC = sum(df_numuCC[weight])

    # numubarCC
    df_numu_barCC = isNumu_barCC(df_mc)
    n_numu_barCC = sum(df_numu_barCC[weight])

    # NCpi+,-
    df_ncpi = isNCpi(df_mc)
    n_ncpi = sum(df_ncpi[weight])

    # NC
    df_nc = isNC(df_mc)
    n_nc = sum(df_nc[weight])
    
    # nue and nuebarCC
    df_nue_nuebarCC = isNue_NuebarCC(df_mc)
    n_nue_nuebarCC = sum(df_nue_nuebarCC[weight])

    # print('\nOverlay (%i entries)' % n_mc)
    # print('Cosmic    %10i' % n_cosmic)
    # print('outFV     %10i \n' % n_outFV)
    # print('NumuCC %10i' % n_numuCC)
    # print('NumubarCC    %10i' % n_numu_barCC)
    # print('NCpi      %10i' % n_ncpi)
    # print('NC        %10i' % n_nc)
    # print('Nue/NuebarCC %10i' % n_nue_nuebarCC)
    # print('Total     %10i' % (n_cosmic + n_outFV + n_numuCC + n_numu_barCC \
    #                           + n_ncpi + n_nc + n_nue_nuebarCC))


    #—————————————#
    # get entries #
    #—————————————#
    
    if (var=="kine_reco_Enu"):
        #—————————————————————#
        # .95*MC energy scale #
        #—————————————————————#
        hist_list = [df_cosmic[var]*0.95,
                    df_outFV[var]*0.95,
                    df_numuCC[var]*0.95,
                    df_numu_barCC[var]*0.95,
                    df_nc[var]*0.95,
                    df_ncpi[var]*0.95,
                    df_nue_nuebarCC[var]*0.95,
                    df_ext[var]*0.95,
                    df_dirt[var]*0.95]
    
    else:
        hist_list = [df_cosmic[var],
                    df_outFV[var],
                    df_numuCC[var],
                    df_numu_barCC[var],
                    df_nc[var],
                    df_ncpi[var],
                    df_nue_nuebarCC[var],
                    df_ext[var],
                    df_dirt[var]]

    hist_data = df_data[var]


    #—————————————#
    # get weight  #
    #—————————————#
    
    w_list = [df_cosmic[weight],
                df_outFV[weight],
                df_numuCC[weight],
                df_numu_barCC[weight],
                df_nc[weight],
                df_ncpi[weight],
                df_nue_nuebarCC[weight],
                df_ext[weight],
                df_dirt[weight]]

    w_data = df_data[weight]


    #—————————————#
    # Colors!!!!! #
    #—————————————#
    
    c_list = ['paleturquoise',
            'red',
            'limegreen',
            'aqua',
            'orange',
            'saddlebrown',
            'grey',
            'gold',
            'pink']

    c_data = 'black'
    
    
    #————————#
    # labels #
    #————————#
 
    label_list = ['In-time Cosmic (%1.1f)'%(sum(w_list[0])),
                'outFV (%1.1f)'%(sum(w_list[1])),
                r'$\nu_{\mu}$ CC (%1.1f)'%(sum(w_list[2])),
                r'$\bar{\nu}_{\mu}$ CC (%1.1f)'%(sum(w_list[3])),
                'NC (%1.1f)'%(sum(w_list[4])),
                r'NC $\pi$ (%1.1f)'%(sum(w_list[5])),
                r'$\nu_{e}$ CC (%1.1f)'%(sum(w_list[6])),
                'Other Cosmic (%1.1f)'%(sum(w_list[7])),
                'Out-of Cryo (%1.1f)'%(sum(w_list[8]))]

    label_data = 'Beam-On (%1.1f)'%(sum(w_data))

    # label_list = ['Cosmic (%1.1f)'%(len(df_cosmic)),
    #             'outFV (%1.1f)'%(len(df_outFV)),
    #             r'$\nu_{\mu}$ CC (%1.1f)'%(len(df_numuCC)),
    #             r'$\bar{\nu}_{\mu}$ CC (%1.1f)'%(len(df_numu_barCC)),
    #             'NC (%1.1f)'%(len(df_nc)),
    #             r'NC $\pi$ (%1.1f)'%(len(df_ncpi)),
    #             r'$\nu_{e}$ CC (%1.1f)'%(len(df_nue_nuebarCC)),
    #             'Beam Off (%1.1f)'%(len(df_ext)),
    #             'Out-of Cryo (%1.1f)'%(len(df_dirt))]

    # label_data = 'Beam-On (%1.1f)'%(sum(w_data))
    
    
    #————————————————————————————#
    # Calculate Stat Uncertainty #
    #————————————————————————————#

    # distributions without data pot normalisation
    h_cosmicB, b_cosmicB = np.histogram(hist_list[0], bins=nbins, range=(xmin,xmax))
    h_outFVB, b_outFVB = np.histogram(hist_list[1], bins=nbins, range=(xmin,xmax))
    h_numuCCB, b_numuCCB = np.histogram(hist_list[2], bins=nbins, range=(xmin,xmax))
    h_numuBarCCB, b_numuBarCCB = np.histogram(hist_list[3], bins=nbins, range=(xmin,xmax))
    h_ncB, b_ncB = np.histogram(hist_list[4], bins=nbins, range=(xmin,xmax))
    h_ncPiB, b_ncPiB = np.histogram(hist_list[5], bins=nbins, range=(xmin,xmax))
    h_nueCCB, b_nueCCB = np.histogram(hist_list[6], bins=nbins, range=(xmin,xmax))
    h_extB, b_extB = np.histogram(hist_list[7], bins=nbins, range=(xmin,xmax))
    h_dirtB, b_dirtB = np.histogram(hist_list[8], bins=nbins, range=(xmin,xmax))

    # distributions with data pot normalisation
    h_cosmic, b_cosmic = np.histogram(hist_list[0], bins=nbins, range=(xmin,xmax), weights=w_list[0])
    h_outFV, b_outFV = np.histogram(hist_list[1], bins=nbins, range=(xmin,xmax), weights=w_list[1])
    h_numuCC, b_numuCC = np.histogram(hist_list[2], bins=nbins, range=(xmin,xmax), weights=w_list[2])
    h_numuBarCC, b_numuBarCC = np.histogram(hist_list[3], bins=nbins, range=(xmin,xmax), weights=w_list[3])
    h_nc, b_nc = np.histogram(hist_list[4], bins=nbins, range=(xmin,xmax), weights=w_list[4])
    h_ncPi, b_ncPi = np.histogram(hist_list[5], bins=nbins, range=(xmin,xmax), weights=w_list[5])
    h_nueCC, b_nueCC = np.histogram(hist_list[6], bins=nbins, range=(xmin,xmax), weights=w_list[6])
    h_ext, b_ext = np.histogram(hist_list[7], bins=nbins, range=(xmin,xmax), weights=w_list[7])
    h_dirt, b_dirt = np.histogram(hist_list[8], bins=nbins, range=(xmin,xmax), weights=w_list[8])

    # sum mc distributions
    total_mc = np.array([h_cosmic, h_outFV, h_numuCC, h_numuBarCC, h_nc, h_ncPi, h_nueCC, h_ext, h_dirt])
    total_mc = total_mc.sum(axis=0)
    total_mc_max = np.max(total_mc)

    # calculate their individual stat uncert as sqrt(number of entries) for the distribution before data pot normalisation
    stat_cosmic = np.sqrt(h_cosmicB) 
    stat_outFV = np.sqrt(h_outFVB)
    stat_numuCC = np.sqrt(h_numuCCB)
    stat_numuBarCC = np.sqrt(h_numuBarCCB)
    stat_nc = np.sqrt(h_ncB)
    stat_ncPi = np.sqrt(h_ncPiB)
    stat_nueCC = np.sqrt(h_nueCCB)
    stat_ext = np.sqrt(h_extB)
    stat_dirt = np.sqrt(h_dirtB)


    # calculate ratio bin-by-bin of the distribution before/after data pot normalisation
    def divide_arrays(arr1, arr2):
        with np.errstate(divide='ignore', invalid='ignore'):
            arr3 = np.true_divide(arr1, arr2)
            arr3[arr3 == np.inf] = 0
            arr3 = np.nan_to_num(arr3)
        return arr3
    ratio_cosmic = divide_arrays(h_cosmic, h_cosmicB)
    ratio_outFV = divide_arrays(h_outFV, h_outFVB)
    ratio_numuCC = divide_arrays(h_numuCC, h_numuCCB)
    ratio_numuBarCC = divide_arrays(h_numuBarCC, h_numuBarCCB)
    ratio_nc = divide_arrays(h_nc, h_ncB)
    ratio_ncPi = divide_arrays(h_ncPi, h_ncPiB)
    ratio_nueCC = divide_arrays(h_nueCC, h_nueCCB)
    ratio_ext = divide_arrays(h_ext, h_extB)
    ratio_dirt = divide_arrays(h_dirt, h_dirtB)

    # update stat uncertainty by multiplying it by the ratio we've just calculated
    stat_cosmic = np.multiply(stat_cosmic, ratio_cosmic)
    stat_outFV = np.multiply(stat_outFV, ratio_outFV)
    stat_numuCC = np.multiply(stat_numuCC, ratio_numuCC)
    stat_numuBarCC = np.multiply(stat_numuBarCC, ratio_numuBarCC)
    stat_nc = np.multiply(stat_nc, ratio_nc)
    stat_ncPi = np.multiply(stat_ncPi, ratio_ncPi)
    stat_nueCC = np.multiply(stat_nueCC, ratio_nueCC)
    stat_ext = np.multiply(stat_ext, ratio_ext)
    stat_dirt = np.multiply(stat_dirt, ratio_dirt)

    # calculate total uncertainty
    stat_cosmic2 = np.multiply(stat_cosmic, ratio_cosmic)
    stat_outFV2 = np.multiply(stat_outFV, ratio_outFV)
    stat_numuCC2 = np.multiply(stat_numuCC, ratio_numuCC)
    stat_numuBarCC2 = np.multiply(stat_numuBarCC, ratio_numuBarCC)
    stat_nc2 = np.multiply(stat_nc, ratio_nc)
    stat_ncPi2 = np.multiply(stat_ncPi, ratio_ncPi)
    stat_nueCC2 = np.multiply(stat_nueCC, ratio_nueCC)
    stat_ext2 = np.multiply(stat_ext, ratio_ext)
    stat_dirt2 = np.multiply(stat_dirt, ratio_dirt)

    total_stat = np.array([stat_cosmic2, stat_outFV2, stat_numuCC2, stat_numuBarCC2, 
                        stat_nc2, stat_ncPi2,  stat_nueCC2, stat_ext2, stat_dirt2])
    total_stat = np.sqrt(total_stat.sum(axis=0))


    #——————————————————————#
    # Create Stacked Histo #
    #——————————————————————#
    #xerr for uneven histo reco nu E
    if (not isinstance(nbins, int)):
        xerr_reco_nuE = []
        for i in range(len(nbins)):
            print(len(nbins), i)
            if (i<len(nbins)-1):
                xerr_reco_nuE.append(0.5*(nbins[i+1]-nbins[i]))              
    
    fig, axs = plt.subplots(1, 1, figsize=(8,8))
    plt.subplots_adjust(left=0.125, bottom=0.1, right=0.9, top=0.9, wspace=0.2, hspace=0.1)
    axs.hist(hist_list, bins=nbins, range=(xmin,xmax), weights=w_list, color=c_list, label=label_list, stacked=True)
    h_mc_max = np.max(total_mc_max)
    h_data, b_data = np.histogram(hist_data, weights=w_data, bins=nbins, range=(xmin,xmax))
    h_data_max = np.max(h_data)
    # err_bar = [np.sqrt(x) for x in h_data] # h_err = sqrt(bin)
    # mid = 0.5*(b_data[1:] + b_data[:-1])
    # if (not isinstance(nbins, int)):
    #     axs[0].errorbar(mid, h_data, xerr=xerr_reco_nuE, yerr=err_bar, color='black', label=label_data, fmt='o')
    # else:
    #     axs[0].errorbar(mid, h_data, xerr=0.5*(xmax-xmin)/nbins, yerr=err_bar, color='black', label=label_data, fmt='o')
    # upvals = np.append((np.array(total_mc)+np.array(total_stat)),(np.array(total_mc)+np.array(total_stat))[-1])
    # lowvals = np.append((np.array(total_mc)-np.array(total_stat)),(np.array(total_mc)-np.array(total_stat))[-1])
    # axs[0].fill_between(b_data, lowvals, upvals, step='post', color='gray', hatch='///', alpha=0.3, zorder=2)
    axs.legend(ncol=2, fontsize=12, frameon=False, loc='best')
    axs.set_ylabel('Entries', size=17.5)
    axs.set_xlabel('%s' % xaxis, size=17.5)
    axs.set_ylim([0, 1.5*h_mc_max])
    #axs[0].set_tick_params(labelsize=15)
    # axs[0].set_title("MicroBooNE NuMI Data: 4.58e+20 POT", size=15, loc='left')
    # axs[0].set_title("MicroBooNE NuMI Data: 4.97e+20 POT", size=15, loc='left')
    # #axs[0].xaxis.set_tick_params(labelsize=15)
    # axs[0].yaxis.set_tick_params(labelsize=15)
    
    # axs[1].set_ylabel('Data/MC', size=15)
    # axs[1].set_xlabel('%s' % xaxis, size=15)
    # axs[1].xaxis.set_tick_params(labelsize=15)
    # axs[1].yaxis.set_tick_params(labelsize=15)
    # axs[1].set_ylim([0.5, 1.5])
    
    # --- bottom pad
    
    # calculate data/mc ratio and the uncertainty bar
    #ratio_data_mc = divide_arrays(h_data, total_mc)
    #axs[1].plot(mid, ratio_data_mc, marker='o', color='black', linewidth=0) # plot points
    # draw data stat uncertainty
    #with np.errstate(divide='ignore', invalid='ignore'):
        #num = np.sqrt(h_data)
        #den = total_mc
        #ratio_err_bar = np.true_divide(num, den)
        #ratio_err_bar[ratio_err_bar == np.inf] = 0
        #ratio_err_bar = np.nan_to_num(ratio_err_bar)
        #if (not isinstance(nbins, int)):
        #    axs[1].errorbar(mid, ratio_data_mc, xerr=xerr_reco_nuE, yerr=ratio_err_bar, color='black', label=label_data, fmt='o')
        #else:
        #    axs[1].errorbar(mid, ratio_data_mc, xerr=0.5*(xmax-xmin)/nbins, yerr=ratio_err_bar, color='black', label=label_data, fmt='o')
    
    # draw gray area error
    #if (not isinstance(nbins, int)):
       # ratio_center = np.ones(len(nbins)-1)
    #else:
        #ratio_center = np.ones(nbins)
    # with np.errstate(divide='ignore', invalid='ignore'):
    #     num = total_stat
    #     den = total_mc
    #     ratio_err_area = np.true_divide(num, den)
    #     ratio_err_area[ratio_err_area == np.inf] = 0
    #     ratio_err_area = np.nan_to_num(ratio_err_area)
    # upvals = np.append((np.array(ratio_center)+np.array(ratio_err_area)),(np.array(ratio_center)+np.array(ratio_err_area))[-1])
    # lowvals = np.append((np.array(ratio_center)-np.array(ratio_err_area)),(np.array(ratio_center)-np.array(ratio_err_area))[-1])
    # axs[1].fill_between(b_data, lowvals, upvals, step='post', color='gray', alpha=0.3, zorder=2)
    
    plt.savefig('%s.png' % plot_png_name, dpi=600)
    plt.tight_layout()
    plt.show()

In [33]:
#=============================================#
#  A Function for Plotting Stacked Histograms # (MC ONLY)
#=============================================#


def mc_data_stacked_hist_truth(df_mc, df_dirt, var, weight, xaxis, xmin, \
                         xmax, nbins):
    #————————————————————————————————————————————————#
    # Restrict the range of df in the plotting range #
    #————————————————————————————————————————————————#
    
    
    # df_data = df_data[(df_data[var]>=xmin) & (df_data[var]<=xmax)]
    # df_ext = df_ext[(df_ext[var]>=xmin) & (df_ext[var]<=xmax)]
    # df_mc = df_mc[(df_mc[var]>=xmin) & (df_mc[var]<=xmax)]
    # df_dirt = df_dirt[(df_dirt[var]>=xmin) & (df_dirt[var]<=xmax)]
    
   
    
    #————————————————————————————————————————————#
    # Get the number of entries for each dataset #
    #————————————————————————————————————————————#
    
    n_mc = sum(df_mc[weight])

    #———————————————————————————————————#
    # Classify mc df into each topology #
    #———————————————————————————————————#
    
    
    # cosmic
    df_cosmic = isCosmic(df_mc)
    n_cosmic = sum(df_cosmic[weight])
    df_mc = notCosmic(df_mc)

    # outFV
    df_outFV = isOutFV(df_mc)
    n_outFV = sum(df_outFV[weight])
    df_mc = notOutFV(df_mc)

    # numuCC
    df_numuCC = isNumuCC(df_mc)
    n_numuCC = sum(df_numuCC[weight])

    # numubarCC
    df_numu_barCC = isNumu_barCC(df_mc)
    n_numu_barCC = sum(df_numu_barCC[weight])

    # NCpi+,-
    df_ncpi = isNCpi(df_mc)
    n_ncpi = sum(df_ncpi[weight])

    # NC
    df_nc = isNC(df_mc)
    n_nc = sum(df_nc[weight])
    
    # nue and nuebarCC
    df_nue_nuebarCC = isNue_NuebarCC(df_mc)
    n_nue_nuebarCC = sum(df_nue_nuebarCC[weight])

    # print('\nOverlay (%i entries)' % n_mc)
    # print('Cosmic    %10i' % n_cosmic)
    # print('outFV     %10i \n' % n_outFV)
    # print('NumuCC %10i' % n_numuCC)
    # print('NumubarCC    %10i' % n_numu_barCC)
    # print('NCpi      %10i' % n_ncpi)
    # print('NC        %10i' % n_nc)
    # print('Nue/NuebarCC %10i' % n_nue_nuebarCC)
    # print('Total     %10i' % (n_cosmic + n_outFV + n_numuCC + n_numu_barCC \
    #                           + n_ncpi + n_nc + n_nue_nuebarCC))


    #—————————————#
    # get entries #
    #—————————————#
    
    # if (var=="kine_reco_Enu"):
    #     #—————————————————————#
    #     # .95*MC energy scale #
    #     #—————————————————————#
    #     hist_list = [df_cosmic[var]*0.95,
    #                 df_outFV[var]*0.95,
    #                 df_numuCC[var]*0.95,
    #                 df_numu_barCC[var]*0.95,
    #                 df_nc[var]*0.95,
    #                 df_ncpi[var]*0.95,
    #                 df_nue_nuebarCC[var]*0.95,
    #                 df_dirt[var]*0.95]
    
    # else:
    hist_list = [df_cosmic[var],
                df_outFV[var],
                df_numuCC[var],
                df_numu_barCC[var],
                df_nc[var],
                df_ncpi[var],
                df_nue_nuebarCC[var],
                df_dirt[var]]


    #—————————————#
    # get weight  #
    #—————————————#
    
    w_list = [df_cosmic[weight],
                df_outFV[weight],
                df_numuCC[weight],
                df_numu_barCC[weight],
                df_nc[weight],
                df_ncpi[weight],
                df_nue_nuebarCC[weight],
                df_dirt[weight]]


    #—————————————#
    # Colors!!!!! #
    #—————————————#
    
    c_list = ['paleturquoise',
            'red',
            'limegreen',
            'aqua',
            'orange',
            'saddlebrown',
            'grey',
            'pink']

    
    #————————#
    # labels #
    #————————#
 
    label_list = ['In-Time Cosmic (%1.1f)'%(sum(w_list[0])),
                'outFV (%1.1f)'%(sum(w_list[1])),
                r'$\nu_{\mu}$ CC (%1.1f)'%(sum(w_list[2])),
                r'$\bar{\nu}_{\mu}$ CC (%1.1f)'%(sum(w_list[3])),
                'NC (%1.1f)'%(sum(w_list[4])),
                r'NC $\pi$ (%1.1f)'%(sum(w_list[5])),
                r'$\nu_{e}$ CC (%1.1f)'%(sum(w_list[6])),
                'Out-of Cryo (%1.1f)'%(sum(w_list[7]))]

    # label_list = ['Cosmic (%1.1f)'%(len(df_cosmic)),
    #             'outFV (%1.1f)'%(len(df_outFV)),
    #             r'$\nu_{\mu}$ CC (%1.1f)'%(len(df_numuCC)),
    #             r'$\bar{\nu}_{\mu}$ CC (%1.1f)'%(len(df_numu_barCC)),
    #             'NC (%1.1f)'%(len(df_nc)),
    #             r'NC $\pi$ (%1.1f)'%(len(df_ncpi)),
    #             r'$\nu_{e}$ CC (%1.1f)'%(len(df_nue_nuebarCC)),
    #             'Beam Off (%1.1f)'%(len(df_ext)),
    #             'Out-of Cryo (%1.1f)'%(len(df_dirt))]

    # label_data = 'Beam-On (%1.1f)'%(sum(w_data))
    
    
    #————————————————————————————#
    # Calculate Stat Uncertainty #
    #————————————————————————————#

    # distributions without data pot normalisation
    h_cosmicB, b_cosmicB = np.histogram(hist_list[0], bins=nbins, range=(xmin,xmax))
    h_outFVB, b_outFVB = np.histogram(hist_list[1], bins=nbins, range=(xmin,xmax))
    h_numuCCB, b_numuCCB = np.histogram(hist_list[2], bins=nbins, range=(xmin,xmax))
    h_numuBarCCB, b_numuBarCCB = np.histogram(hist_list[3], bins=nbins, range=(xmin,xmax))
    h_ncB, b_ncB = np.histogram(hist_list[4], bins=nbins, range=(xmin,xmax))
    h_ncPiB, b_ncPiB = np.histogram(hist_list[5], bins=nbins, range=(xmin,xmax))
    h_nueCCB, b_nueCCB = np.histogram(hist_list[6], bins=nbins, range=(xmin,xmax))
    h_dirtB, b_dirtB = np.histogram(hist_list[7], bins=nbins, range=(xmin,xmax))

    # distributions with data pot normalisation
    h_cosmic, b_cosmic = np.histogram(hist_list[0], bins=nbins, range=(xmin,xmax), weights=w_list[0])
    h_outFV, b_outFV = np.histogram(hist_list[1], bins=nbins, range=(xmin,xmax), weights=w_list[1])
    h_numuCC, b_numuCC = np.histogram(hist_list[2], bins=nbins, range=(xmin,xmax), weights=w_list[2])
    h_numuBarCC, b_numuBarCC = np.histogram(hist_list[3], bins=nbins, range=(xmin,xmax), weights=w_list[3])
    h_nc, b_nc = np.histogram(hist_list[4], bins=nbins, range=(xmin,xmax), weights=w_list[4])
    h_ncPi, b_ncPi = np.histogram(hist_list[5], bins=nbins, range=(xmin,xmax), weights=w_list[5])
    h_nueCC, b_nueCC = np.histogram(hist_list[6], bins=nbins, range=(xmin,xmax), weights=w_list[6])
    h_dirt, b_dirt = np.histogram(hist_list[7], bins=nbins, range=(xmin,xmax), weights=w_list[7])

    # sum mc distributions
    total_mc = np.array([h_cosmic, h_outFV, h_numuCC, h_numuBarCC, h_nc, h_ncPi, h_nueCC, h_dirt])
    total_mc = total_mc.sum(axis=0)
    total_mc_max = np.max(total_mc)

    # calculate their individual stat uncert as sqrt(number of entries) for the distribution before data pot normalisation
    stat_cosmic = np.sqrt(h_cosmicB) 
    stat_outFV = np.sqrt(h_outFVB)
    stat_numuCC = np.sqrt(h_numuCCB)
    stat_numuBarCC = np.sqrt(h_numuBarCCB)
    stat_nc = np.sqrt(h_ncB)
    stat_ncPi = np.sqrt(h_ncPiB)
    stat_nueCC = np.sqrt(h_nueCCB)
    stat_dirt = np.sqrt(h_dirtB)


    # calculate ratio bin-by-bin of the distribution before/after data pot normalisation
    def divide_arrays(arr1, arr2):
        with np.errstate(divide='ignore', invalid='ignore'):
            arr3 = np.true_divide(arr1, arr2)
            arr3[arr3 == np.inf] = 0
            arr3 = np.nan_to_num(arr3)
        return arr3
    ratio_cosmic = divide_arrays(h_cosmic, h_cosmicB)
    ratio_outFV = divide_arrays(h_outFV, h_outFVB)
    ratio_numuCC = divide_arrays(h_numuCC, h_numuCCB)
    ratio_numuBarCC = divide_arrays(h_numuBarCC, h_numuBarCCB)
    ratio_nc = divide_arrays(h_nc, h_ncB)
    ratio_ncPi = divide_arrays(h_ncPi, h_ncPiB)
    ratio_nueCC = divide_arrays(h_nueCC, h_nueCCB)
    ratio_dirt = divide_arrays(h_dirt, h_dirtB)

    # update stat uncertainty by multiplying it by the ratio we've just calculated
    stat_cosmic = np.multiply(stat_cosmic, ratio_cosmic)
    stat_outFV = np.multiply(stat_outFV, ratio_outFV)
    stat_numuCC = np.multiply(stat_numuCC, ratio_numuCC)
    stat_numuBarCC = np.multiply(stat_numuBarCC, ratio_numuBarCC)
    stat_nc = np.multiply(stat_nc, ratio_nc)
    stat_ncPi = np.multiply(stat_ncPi, ratio_ncPi)
    stat_nueCC = np.multiply(stat_nueCC, ratio_nueCC)
    stat_dirt = np.multiply(stat_dirt, ratio_dirt)

    # calculate total uncertainty
    stat_cosmic2 = np.multiply(stat_cosmic, ratio_cosmic)
    stat_outFV2 = np.multiply(stat_outFV, ratio_outFV)
    stat_numuCC2 = np.multiply(stat_numuCC, ratio_numuCC)
    stat_numuBarCC2 = np.multiply(stat_numuBarCC, ratio_numuBarCC)
    stat_nc2 = np.multiply(stat_nc, ratio_nc)
    stat_ncPi2 = np.multiply(stat_ncPi, ratio_ncPi)
    stat_nueCC2 = np.multiply(stat_nueCC, ratio_nueCC)
    stat_dirt2 = np.multiply(stat_dirt, ratio_dirt)

    total_stat = np.array([stat_cosmic2, stat_outFV2, stat_numuCC2, stat_numuBarCC2, 
                        stat_nc2, stat_ncPi2,  stat_nueCC2, stat_dirt2])
    total_stat = np.sqrt(total_stat.sum(axis=0))


    #——————————————————————#
    # Create Stacked Histo #
    #——————————————————————#
    #xerr for uneven histo reco nu E
    if (not isinstance(nbins, int)):
        xerr_reco_nuE = []
        for i in range(len(nbins)):
            print(len(nbins), i)
            if (i<len(nbins)-1):
                xerr_reco_nuE.append(0.5*(nbins[i+1]-nbins[i]))              
    
    fig, axs = plt.subplots(1, 1, figsize=(8,8))
    plt.subplots_adjust(left=0.125, bottom=0.1, right=0.9, top=0.9, wspace=0.2, hspace=0.1)
    axs.hist(hist_list, bins=nbins, range=(xmin,xmax), weights=w_list, color=c_list, label=label_list, stacked=True)
    # h_data, b_data = np.histogram(hist_data, weights=w_data, bins=nbins, range=(xmin,xmax))
    # h_data_max = np.max(h_data)
    # err_bar = [np.sqrt(x) for x in h_data] # h_err = sqrt(bin)
    # mid = 0.5*(b_data[1:] + b_data[:-1])
    # if (not isinstance(nbins, int)):
    #     axs[0].errorbar(mid, h_data, xerr=xerr_reco_nuE, yerr=err_bar, color='black', label=label_data, fmt='o')
    # else:
    #     axs[0].errorbar(mid, h_data, xerr=0.5*(xmax-xmin)/nbins, yerr=err_bar, color='black', label=label_data, fmt='o')
    # upvals = np.append((np.array(total_mc)+np.array(total_stat)),(np.array(total_mc)+np.array(total_stat))[-1])
    # lowvals = np.append((np.array(total_mc)-np.array(total_stat)),(np.array(total_mc)-np.array(total_stat))[-1])
    # axs[0].fill_between(b_data, lowvals, upvals, step='post', color='gray', hatch='///', alpha=0.3, zorder=2)
    axs.legend(ncol=2, fontsize=14.5, frameon=False, loc='best')
    axs.set_ylabel('Entries', size=17.5)
    axs.set_xlabel('%s' % xaxis, size=17.5)
    #axs[0].set_tick_params(labelsize=15)
    # axs[0].set_title("MicroBooNE NuMI Data: 4.58e+20 POT", size=15, loc='left')
    # axs[0].set_title("MicroBooNE NuMI Data: 4.97e+20 POT", size=15, loc='left')
    # #axs[0].xaxis.set_tick_params(labelsize=15)
    # axs[0].yaxis.set_tick_params(labelsize=15)
    axs.set_ylim([0, 2*total_mc_max])
    
    # axs[1].set_ylabel('Data/MC', size=15)
    # axs[1].set_xlabel('%s' % xaxis, size=15)
    # axs[1].xaxis.set_tick_params(labelsize=15)
    # axs[1].yaxis.set_tick_params(labelsize=15)
    # axs[1].set_ylim([0.5, 1.5])
    
    # --- bottom pad
    
    # calculate data/mc ratio and the uncertainty bar
    #ratio_data_mc = divide_arrays(h_data, total_mc)
    #axs[1].plot(mid, ratio_data_mc, marker='o', color='black', linewidth=0) # plot points
    # draw data stat uncertainty
    #with np.errstate(divide='ignore', invalid='ignore'):
        #num = np.sqrt(h_data)
        #den = total_mc
        #ratio_err_bar = np.true_divide(num, den)
        #ratio_err_bar[ratio_err_bar == np.inf] = 0
        #ratio_err_bar = np.nan_to_num(ratio_err_bar)
        #if (not isinstance(nbins, int)):
        #    axs[1].errorbar(mid, ratio_data_mc, xerr=xerr_reco_nuE, yerr=ratio_err_bar, color='black', label=label_data, fmt='o')
        #else:
        #    axs[1].errorbar(mid, ratio_data_mc, xerr=0.5*(xmax-xmin)/nbins, yerr=ratio_err_bar, color='black', label=label_data, fmt='o')
    
    # draw gray area error
    #if (not isinstance(nbins, int)):
       # ratio_center = np.ones(len(nbins)-1)
    #else:
        #ratio_center = np.ones(nbins)
    # with np.errstate(divide='ignore', invalid='ignore'):
    #     num = total_stat
    #     den = total_mc
    #     ratio_err_area = np.true_divide(num, den)
    #     ratio_err_area[ratio_err_area == np.inf] = 0
    #     ratio_err_area = np.nan_to_num(ratio_err_area)
    # upvals = np.append((np.array(ratio_center)+np.array(ratio_err_area)),(np.array(ratio_center)+np.array(ratio_err_area))[-1])
    # lowvals = np.append((np.array(ratio_center)-np.array(ratio_err_area)),(np.array(ratio_center)-np.array(ratio_err_area))[-1])
    # axs[1].fill_between(b_data, lowvals, upvals, step='post', color='gray', alpha=0.3, zorder=2)

    plt.tight_layout()
    plt.show()

In [34]:
# #=============================================#
# #  A Function for Plotting Stacked Histograms #
# #=============================================#


# def mc_data_stacked_hist(df_data, df_ext, df_mc, df_dirt, var, weight, plot_png_name, xaxis, xmin, \
#                          xmax, nbins, pot_data):
#     #————————————————————————————————————————————————#
#     # Restrict the range of df in the plotting range #
#     #————————————————————————————————————————————————#
    
    
#     # df_data = df_data[(df_data[var]>=xmin) & (df_data[var]<=xmax)]
#     # df_ext = df_ext[(df_ext[var]>=xmin) & (df_ext[var]<=xmax)]
#     # df_mc = df_mc[(df_mc[var]>=xmin) & (df_mc[var]<=xmax)]
#     # df_dirt = df_dirt[(df_dirt[var]>=xmin) & (df_dirt[var]<=xmax)]
    
   
    
#     #————————————————————————————————————————————#
#     # Get the number of entries for each dataset #
#     #————————————————————————————————————————————#
    
    
#     n_data = sum(df_data[weight])
#     n_ext = sum(df_ext[weight])
#     n_mc = sum(df_mc[weight])

#     #———————————————————————————————————#
#     # Classify mc df into each topology #
#     #———————————————————————————————————#
    
    
#     # cosmic
#     df_cosmic = isCosmic(df_mc)
#     n_cosmic = sum(df_cosmic[weight])
#     df_mc = notCosmic(df_mc)

#     # outFV
#     df_outFV = isOutFV(df_mc)
#     n_outFV = sum(df_outFV[weight])
#     df_mc = notOutFV(df_mc)

#     # numuCC
#     df_numuCC = isNumuCC(df_mc)
#     n_numuCC = sum(df_numuCC[weight])

#     # numubarCC
#     df_numu_barCC = isNumu_barCC(df_mc)
#     n_numu_barCC = sum(df_numu_barCC[weight])

#     # NCpi+,-
#     df_ncpi = isNCpi(df_mc)
#     n_ncpi = sum(df_ncpi[weight])

#     # NC
#     df_nc = isNC(df_mc)
#     n_nc = sum(df_nc[weight])
    
#     # nue and nuebarCC
#     df_nue_nuebarCC = isNue_NuebarCC(df_mc)
#     n_nue_nuebarCC = sum(df_nue_nuebarCC[weight])

#     print('\nOverlay (%i entries)' % n_mc)
#     print('Cosmic    %10i' % n_cosmic)
#     print('outFV     %10i \n' % n_outFV)
#     print('NumuCC %10i' % n_numuCC)
#     print('NumubarCC    %10i' % n_numu_barCC)
#     print('NCpi      %10i' % n_ncpi)
#     print('NC        %10i' % n_nc)
#     print('Nue/NuebarCC %10i' % n_nue_nuebarCC)
#     print('Total     %10i' % (n_cosmic + n_outFV + n_numuCC + n_numu_barCC \
#                               + n_ncpi + n_nc + n_nue_nuebarCC))


#     #—————————————#
#     # get entries #
#     #—————————————#
    
#     if (var=="kine_reco_Enu"):
#         #—————————————————————#
#         # .95*MC energy scale #
#         #—————————————————————#
#         hist_list = [df_cosmic[var]*0.95,
#                     df_outFV[var]*0.95,
#                     df_numuCC[var]*0.95,
#                     df_numu_barCC[var]*0.95,
#                     df_nc[var]*0.95,
#                     df_ncpi[var]*0.95,
#                     df_nue_nuebarCC[var]*0.95,
#                     df_ext[var]*0.95,
#                     df_dirt[var]*0.95]
    
#     else:
#         hist_list = [df_cosmic[var],
#                     df_outFV[var],
#                     df_numuCC[var],
#                     df_numu_barCC[var],
#                     df_nc[var],
#                     df_ncpi[var],
#                     df_nue_nuebarCC[var],
#                     df_ext[var],
#                     df_dirt[var]]

#     hist_data = df_data[var]


#     #—————————————#
#     # get weight  #
#     #—————————————#
    
#     w_list = [df_cosmic[weight],
#                 df_outFV[weight],
#                 df_numuCC[weight],
#                 df_numu_barCC[weight],
#                 df_nc[weight],
#                 df_ncpi[weight],
#                 df_nue_nuebarCC[weight],
#                 df_ext[weight],
#                 df_dirt[weight]]

#     w_data = df_data[weight]


#     #—————————————#
#     # Colors!!!!! #
#     #—————————————#
    
#     c_list = ['paleturquoise',
#             'red',
#             'limegreen',
#             'aqua',
#             'orange',
#             'saddlebrown',
#             'grey',
#             'gold',
#             'pink']

#     c_data = 'black'
    
    
#     #————————#
#     # labels #
#     #————————#
 
#     label_list = ['In-Time Cosmic (%1.1f)'%(sum(w_list[0])),
#                 'outFV (%1.1f)'%(sum(w_list[1])),
#                 r'$\nu_{\mu}$ CC (%1.1f)'%(sum(w_list[2])),
#                 r'$\bar{\nu}_{\mu}$ CC (%1.1f)'%(sum(w_list[3])),
#                 'NC (%1.1f)'%(sum(w_list[4])),
#                 r'NC $\pi$ (%1.1f)'%(sum(w_list[5])),
#                 r'$\nu_{e}$ CC (%1.1f)'%(sum(w_list[6])),
#                 'Other Cosmics (%1.1f)'%(sum(w_list[7])),
#                 'Out-of Cryo (%1.1f)'%(sum(w_list[8]))]

#     label_data = 'Beam-On (%1.1f)'%(sum(w_data))
    
    
#     #————————————————————————————#
#     # Calculate Stat Uncertainty #
#     #————————————————————————————#

#     # distributions without data pot normalisation
#     h_cosmicB, b_cosmicB = np.histogram(hist_list[0], bins=nbins, range=(xmin,xmax))
#     h_outFVB, b_outFVB = np.histogram(hist_list[1], bins=nbins, range=(xmin,xmax))
#     h_numuCCB, b_numuCCB = np.histogram(hist_list[2], bins=nbins, range=(xmin,xmax))
#     h_numuBarCCB, b_numuBarCCB = np.histogram(hist_list[3], bins=nbins, range=(xmin,xmax))
#     h_ncB, b_ncB = np.histogram(hist_list[4], bins=nbins, range=(xmin,xmax))
#     h_ncPiB, b_ncPiB = np.histogram(hist_list[5], bins=nbins, range=(xmin,xmax))
#     h_nueCCB, b_nueCCB = np.histogram(hist_list[6], bins=nbins, range=(xmin,xmax))
#     h_extB, b_extB = np.histogram(hist_list[7], bins=nbins, range=(xmin,xmax))
#     h_dirtB, b_dirtB = np.histogram(hist_list[8], bins=nbins, range=(xmin,xmax))

#     # distributions with data pot normalisation
#     h_cosmic, b_cosmic = np.histogram(hist_list[0], bins=nbins, range=(xmin,xmax), weights=w_list[0])
#     h_outFV, b_outFV = np.histogram(hist_list[1], bins=nbins, range=(xmin,xmax), weights=w_list[1])
#     h_numuCC, b_numuCC = np.histogram(hist_list[2], bins=nbins, range=(xmin,xmax), weights=w_list[2])
#     h_numuBarCC, b_numuBarCC = np.histogram(hist_list[3], bins=nbins, range=(xmin,xmax), weights=w_list[3])
#     h_nc, b_nc = np.histogram(hist_list[4], bins=nbins, range=(xmin,xmax), weights=w_list[4])
#     h_ncPi, b_ncPi = np.histogram(hist_list[5], bins=nbins, range=(xmin,xmax), weights=w_list[5])
#     h_nueCC, b_nueCC = np.histogram(hist_list[6], bins=nbins, range=(xmin,xmax), weights=w_list[6])
#     h_ext, b_ext = np.histogram(hist_list[7], bins=nbins, range=(xmin,xmax), weights=w_list[7])
#     h_dirt, b_dirt = np.histogram(hist_list[8], bins=nbins, range=(xmin,xmax), weights=w_list[8])

#     # sum mc distributions
#     total_mc = np.array([h_cosmic, h_outFV, h_numuCC, h_numuBarCC, h_nc, h_ncPi, h_nueCC, h_ext, h_dirt])
#     total_mc = total_mc.sum(axis=0)
#     total_mc_max = np.max(total_mc)

#     # calculate their individual stat uncert as sqrt(number of entries) for the distribution before data pot normalisation
#     stat_cosmic = np.sqrt(h_cosmicB) 
#     stat_outFV = np.sqrt(h_outFVB)
#     stat_numuCC = np.sqrt(h_numuCCB)
#     stat_numuBarCC = np.sqrt(h_numuBarCCB)
#     stat_nc = np.sqrt(h_ncB)
#     stat_ncPi = np.sqrt(h_ncPiB)
#     stat_nueCC = np.sqrt(h_nueCCB)
#     stat_ext = np.sqrt(h_extB)
#     stat_dirt = np.sqrt(h_dirtB)


#     # calculate ratio bin-by-bin of the distribution before/after data pot normalisation
#     def divide_arrays(arr1, arr2):
#         with np.errstate(divide='ignore', invalid='ignore'):
#             arr3 = np.true_divide(arr1, arr2)
#             arr3[arr3 == np.inf] = 0
#             arr3 = np.nan_to_num(arr3)
#         return arr3
#     ratio_cosmic = divide_arrays(h_cosmic, h_cosmicB)
#     ratio_outFV = divide_arrays(h_outFV, h_outFVB)
#     ratio_numuCC = divide_arrays(h_numuCC, h_numuCCB)
#     ratio_numuBarCC = divide_arrays(h_numuBarCC, h_numuBarCCB)
#     ratio_nc = divide_arrays(h_nc, h_ncB)
#     ratio_ncPi = divide_arrays(h_ncPi, h_ncPiB)
#     ratio_nueCC = divide_arrays(h_nueCC, h_nueCCB)
#     ratio_ext = divide_arrays(h_ext, h_extB)
#     ratio_dirt = divide_arrays(h_dirt, h_dirtB)

#     # update stat uncertainty by multiplying it by the ratio we've just calculated
#     stat_cosmic = np.multiply(stat_cosmic, ratio_cosmic)
#     stat_outFV = np.multiply(stat_outFV, ratio_outFV)
#     stat_numuCC = np.multiply(stat_numuCC, ratio_numuCC)
#     stat_numuBarCC = np.multiply(stat_numuBarCC, ratio_numuBarCC)
#     stat_nc = np.multiply(stat_nc, ratio_nc)
#     stat_ncPi = np.multiply(stat_ncPi, ratio_ncPi)
#     stat_nueCC = np.multiply(stat_nueCC, ratio_nueCC)
#     stat_ext = np.multiply(stat_ext, ratio_ext)
#     stat_dirt = np.multiply(stat_dirt, ratio_dirt)

#     # calculate total uncertainty
#     stat_cosmic2 = np.multiply(stat_cosmic, ratio_cosmic)
#     stat_outFV2 = np.multiply(stat_outFV, ratio_outFV)
#     stat_numuCC2 = np.multiply(stat_numuCC, ratio_numuCC)
#     stat_numuBarCC2 = np.multiply(stat_numuBarCC, ratio_numuBarCC)
#     stat_nc2 = np.multiply(stat_nc, ratio_nc)
#     stat_ncPi2 = np.multiply(stat_ncPi, ratio_ncPi)
#     stat_nueCC2 = np.multiply(stat_nueCC, ratio_nueCC)
#     stat_ext2 = np.multiply(stat_ext, ratio_ext)
#     stat_dirt2 = np.multiply(stat_dirt, ratio_dirt)

#     total_stat = np.array([stat_cosmic2, stat_outFV2, stat_numuCC2, stat_numuBarCC2, 
#                         stat_nc2, stat_ncPi2,  stat_nueCC2, stat_ext2, stat_dirt2])
#     total_stat = np.sqrt(total_stat.sum(axis=0))


#     #——————————————————————#
#     # Create Stacked Histo #
#     #——————————————————————#
#     #xerr for uneven histo reco nu E
#     if (not isinstance(nbins, int)):
#         xerr_reco_nuE = []
#         for i in range(len(nbins)):
#             print(len(nbins), i)
#             if (i<len(nbins)-1):
#                 xerr_reco_nuE.append(0.5*(nbins[i+1]-nbins[i]))              
    
#     fig, axs = plt.subplots(2, 1, figsize=(8,10), gridspec_kw=dict(height_ratios=[4,1]), sharex=True)
#     plt.subplots_adjust(left=0.125, bottom=0.1, right=0.9, top=0.9, wspace=0.2, hspace=0.1)
#     axs[0].hist(hist_list, bins=nbins, range=(xmin,xmax), weights=w_list, color=c_list, label=label_list, stacked=True)
#     h_data, b_data = np.histogram(hist_data, weights=w_data, bins=nbins, range=(xmin,xmax))
#     h_data_max = np.max(h_data)
#     err_bar = [np.sqrt(x) for x in h_data] # h_err = sqrt(bin)
#     mid = 0.5*(b_data[1:] + b_data[:-1])
#     if (not isinstance(nbins, int)):
#         axs[0].errorbar(mid, h_data, xerr=xerr_reco_nuE, yerr=err_bar, color='black', label=label_data, fmt='o')
#     else:
#         axs[0].errorbar(mid, h_data, xerr=0.5*(xmax-xmin)/nbins, yerr=err_bar, color='black', label=label_data, fmt='o')
#     upvals = np.append((np.array(total_mc)+np.array(total_stat)),(np.array(total_mc)+np.array(total_stat))[-1])
#     lowvals = np.append((np.array(total_mc)-np.array(total_stat)),(np.array(total_mc)-np.array(total_stat))[-1])
#     axs[0].fill_between(b_data, lowvals, upvals, step='post', color='gray', hatch='///', alpha=0.3, zorder=2)
#     axs[0].legend(ncol=2, fontsize=12, frameon=False, loc='best')
#     axs[0].set_ylabel('Entries', size=17.5)
#     axs[0].set_xlabel('%s' % xaxis, size=17.5)
#     axs[0].set_title("MicroBooNE NuMI Data: %5.2e POT" % pot_data, size=15, loc='left')
#     #axs[0].xaxis.set_tick_params(labelsize=15)
#     axs[0].yaxis.set_tick_params(labelsize=15)
#     axs[0].set_ylim([0, 2*total_mc_max])
    
#     axs[1].set_ylabel('Data/MC', size=15)
#     axs[1].set_xlabel('%s' % xaxis, size=15)
#     axs[1].xaxis.set_tick_params(labelsize=15)
#     axs[1].yaxis.set_tick_params(labelsize=15)
#     axs[1].set_ylim([0.5, 1.5])
    
#     # --- bottom pad
    
#     # calculate data/mc ratio and the uncertainty bar
#     ratio_data_mc = divide_arrays(h_data, total_mc)
#     axs[1].plot(mid, ratio_data_mc, marker='o', color='black', linewidth=0) # plot points
#     # draw data stat uncertainty
#     with np.errstate(divide='ignore', invalid='ignore'):
#         num = np.sqrt(h_data)
#         den = total_mc
#         ratio_err_bar = np.true_divide(num, den)
#         ratio_err_bar[ratio_err_bar == np.inf] = 0
#         ratio_err_bar = np.nan_to_num(ratio_err_bar)
#         if (not isinstance(nbins, int)):
#             axs[1].errorbar(mid, ratio_data_mc, xerr=xerr_reco_nuE, yerr=ratio_err_bar, color='black', label=label_data, fmt='o')
#         else:
#             axs[1].errorbar(mid, ratio_data_mc, xerr=0.5*(xmax-xmin)/nbins, yerr=ratio_err_bar, color='black', label=label_data, fmt='o')
    
#     # draw gray area error
#     if (not isinstance(nbins, int)):
#         ratio_center = np.ones(len(nbins)-1)
#     else:
#         ratio_center = np.ones(nbins)
#     with np.errstate(divide='ignore', invalid='ignore'):
#         num = total_stat
#         den = total_mc
#         ratio_err_area = np.true_divide(num, den)
#         ratio_err_area[ratio_err_area == np.inf] = 0
#         ratio_err_area = np.nan_to_num(ratio_err_area)
#     upvals = np.append((np.array(ratio_center)+np.array(ratio_err_area)),(np.array(ratio_center)+np.array(ratio_err_area))[-1])
#     lowvals = np.append((np.array(ratio_center)-np.array(ratio_err_area)),(np.array(ratio_center)-np.array(ratio_err_area))[-1])
#     axs[1].fill_between(b_data, lowvals, upvals, step='post', color='gray', alpha=0.3, zorder=2)
    
#     plt.savefig('%s.png' % plot_png_name, dpi=600)
#     plt.tight_layout()
#     plt.show()

In [35]:
#=============================================#
#  A Function for Plotting Stacked Histograms # (no category, data vs rest)
#=============================================#


def mc_data_stacked_hist_no_cat(df_data, df_ext, df_mc, df_dirt, var, weight, plot_png_name, xaxis, xmin, \
                         xmax, nbins, pot_data):
    #————————————————————————————————————————————————#
    # Restrict the range of df in the plotting range #
    #————————————————————————————————————————————————#
    
    
    # df_data = df_data[(df_data[var]>=xmin) & (df_data[var]<=xmax)]
    # df_ext = df_ext[(df_ext[var]>=xmin) & (df_ext[var]<=xmax)]
    # df_mc = df_mc[(df_mc[var]>=xmin) & (df_mc[var]<=xmax)]
    # df_dirt = df_dirt[(df_dirt[var]>=xmin) & (df_dirt[var]<=xmax)]
    
   
    
    #————————————————————————————————————————————#
    # Get the number of entries for each dataset #
    #————————————————————————————————————————————#
    
    
    n_data = sum(df_data[weight])
    n_ext = sum(df_ext[weight])
    n_mc = sum(df_mc[weight])

    #———————————————————————————————————#
    # Classify mc df into each topology #
    #———————————————————————————————————#
    
    
    # cosmic
    df_cosmic = isCosmic(df_mc)
    n_cosmic = sum(df_cosmic[weight])
    df_mc = notCosmic(df_mc)

    # outFV
    df_outFV = isOutFV(df_mc)
    n_outFV = sum(df_outFV[weight])
    df_mc = notOutFV(df_mc)

    # numuCC
    df_numuCC = isNumuCC(df_mc)
    n_numuCC = sum(df_numuCC[weight])

    # numubarCC
    df_numu_barCC = isNumu_barCC(df_mc)
    n_numu_barCC = sum(df_numu_barCC[weight])

    # NCpi+,-
    df_ncpi = isNCpi(df_mc)
    n_ncpi = sum(df_ncpi[weight])

    # NC
    df_nc = isNC(df_mc)
    n_nc = sum(df_nc[weight])
    
    # nue and nuebarCC
    df_nue_nuebarCC = isNue_NuebarCC(df_mc)
    n_nue_nuebarCC = sum(df_nue_nuebarCC[weight])

    # print('\nOverlay (%i entries)' % n_mc)
    # print('Cosmic    %10i' % n_cosmic)
    # print('outFV     %10i \n' % n_outFV)
    # print('NumuCC %10i' % n_numuCC)
    # print('NumubarCC    %10i' % n_numu_barCC)
    # print('NCpi      %10i' % n_ncpi)
    # print('NC        %10i' % n_nc)
    # print('Nue/NuebarCC %10i' % n_nue_nuebarCC)
    # print('Total     %10i' % (n_cosmic + n_outFV + n_numuCC + n_numu_barCC \
    #                           + n_ncpi + n_nc + n_nue_nuebarCC))


    #—————————————#
    # get entries #
    #—————————————#
    
    if (var=="kine_reco_Enu"):
        #—————————————————————#
        # .95*MC energy scale #
        #—————————————————————#
        hist_list = [df_mc[var]*0.95,
                    df_ext[var]*0.95,
                    df_dirt[var]*0.95]
    
    else:
        hist_list = [df_mc[var],
                    df_ext[var],
                    df_dirt[var]]

    hist_data = df_data[var]


    #—————————————#
    # get weight  #
    #—————————————#
    
    w_list = [df_mc[weight],
            df_ext[weight],
            df_dirt[weight]]

    w_data = df_data[weight]


    #—————————————#
    # Colors!!!!! #
    #—————————————#
    
    c_list = ['paleturquoise',
            'gold',
            'pink']

    c_data = 'black'
    
    
    #————————#
    # labels #
    #————————#
 
    label_list = ['MC (%1.1f)'%(sum(w_list[0])),
                'Beam Off (%1.1f)'%(sum(w_list[1])),
                'Out-of Cryo (%1.1f)'%(sum(w_list[2]))]

    label_data = 'Beam-On (%1.1f)'%(sum(w_data))
    
    
    #————————————————————————————#
    # Calculate Stat Uncertainty #
    #————————————————————————————#

    # distributions without data pot normalisation
    h_mcB, b_mcB = np.histogram(hist_list[0], bins=nbins, range=(xmin,xmax))
    h_extB, b_extB = np.histogram(hist_list[1], bins=nbins, range=(xmin,xmax))
    h_dirtB, b_dirtB = np.histogram(hist_list[2], bins=nbins, range=(xmin,xmax))

    # distributions with data pot normalisation
    h_mc, b_mc = np.histogram(hist_list[0], bins=nbins, range=(xmin,xmax), weights=w_list[0])
    h_ext, b_ext = np.histogram(hist_list[1], bins=nbins, range=(xmin,xmax), weights=w_list[1])
    h_dirt, b_dirt = np.histogram(hist_list[2], bins=nbins, range=(xmin,xmax), weights=w_list[2])

    # sum mc distributions
    total_mc = np.array([h_mc, h_ext, h_dirt])
    total_mc = total_mc.sum(axis=0)

    # calculate their individual stat uncert as sqrt(number of entries) for the distribution before data pot normalisation
    stat_mc = np.sqrt(h_mcB)
    stat_ext = np.sqrt(h_extB)
    stat_dirt = np.sqrt(h_dirtB)


    # calculate ratio bin-by-bin of the distribution before/after data pot normalisation
    def divide_arrays(arr1, arr2):
        with np.errstate(divide='ignore', invalid='ignore'):
            arr3 = np.true_divide(arr1, arr2)
            arr3[arr3 == np.inf] = 0
            arr3 = np.nan_to_num(arr3)
        return arr3
    ratio_mc = divide_arrays(h_mc, h_mcB)
    ratio_ext = divide_arrays(h_ext, h_extB)
    ratio_dirt = divide_arrays(h_dirt, h_dirtB)

    # update stat uncertainty by multiplying it by the ratio we've just calculated
    stat_mc = np.multiply(stat_mc, ratio_mc)
    stat_ext = np.multiply(stat_ext, ratio_ext)
    stat_dirt = np.multiply(stat_dirt, ratio_dirt)

    # calculate total uncertainty
    stat_mc2 = np.multiply(stat_mc, ratio_mc)
    stat_ext2 = np.multiply(stat_ext, ratio_ext)
    stat_dirt2 = np.multiply(stat_dirt, ratio_dirt)

    total_stat = np.array([stat_mc2, stat_ext2, stat_dirt2])
    total_stat = np.sqrt(total_stat.sum(axis=0))


    #——————————————————————#
    # Create Stacked Histo #
    #——————————————————————#
    #xerr for uneven histo reco nu E
    if (not isinstance(nbins, int)):
        xerr_reco_nuE = []
        for i in range(len(nbins)):
            print(len(nbins), i)
            if (i<len(nbins)-1):
                xerr_reco_nuE.append(0.5*(nbins[i+1]-nbins[i]))              
    
    fig, axs = plt.subplots(2, 1, figsize=(8,10), gridspec_kw=dict(height_ratios=[4,1]), sharex=True)
    plt.subplots_adjust(left=0.125, bottom=0.1, right=0.9, top=0.9, wspace=0.2, hspace=0.1)
    axs[0].hist(hist_list, bins=nbins, range=(xmin,xmax), weights=w_list, color=c_list, label=label_list, stacked=True)
    h_data, b_data = np.histogram(hist_data, weights=w_data, bins=nbins, range=(xmin,xmax))
    h_data_max = np.max(h_data)
    err_bar = [np.sqrt(x) for x in h_data] # h_err = sqrt(bin)
    mid = 0.5*(b_data[1:] + b_data[:-1])
    if (not isinstance(nbins, int)):
        axs[0].errorbar(mid, h_data, xerr=xerr_reco_nuE, yerr=err_bar, color='black', label=label_data, fmt='o')
    else:
        axs[0].errorbar(mid, h_data, xerr=0.5*(xmax-xmin)/nbins, yerr=err_bar, color='black', label=label_data, fmt='o')
    upvals = np.append((np.array(total_mc)+np.array(total_stat)),(np.array(total_mc)+np.array(total_stat))[-1])
    lowvals = np.append((np.array(total_mc)-np.array(total_stat)),(np.array(total_mc)-np.array(total_stat))[-1])
    axs[0].fill_between(b_data, lowvals, upvals, step='post', color='gray', hatch='///', alpha=0.3, zorder=2)
    axs[0].legend(ncol=2, fontsize=12, frameon=False, loc='best')
    axs[0].set_ylabel('Entries', size=15)
    axs[0].set_title("MicroBooNE NuMI Data: %5.2e POT" % pot_data, size=15, loc='left')
    #axs[0].xaxis.set_tick_params(labelsize=15)
    axs[0].yaxis.set_tick_params(labelsize=15)
    axs[0].set_ylim([0, 2*h_data_max])
    
    axs[1].set_ylabel('Data/MC', size=15)
    axs[1].set_xlabel('%s' % xaxis, size=15)
    axs[1].xaxis.set_tick_params(labelsize=15)
    axs[1].yaxis.set_tick_params(labelsize=15)
    axs[1].set_ylim([0.5, 1.5])
    
    # --- bottom pad
    
    # calculate data/mc ratio and the uncertainty bar
    ratio_data_mc = divide_arrays(h_data, total_mc)
    axs[1].plot(mid, ratio_data_mc, marker='o', color='black', linewidth=0) # plot points
    # draw data stat uncertainty
    with np.errstate(divide='ignore', invalid='ignore'):
        num = np.sqrt(h_data)
        den = total_mc
        ratio_err_bar = np.true_divide(num, den)
        ratio_err_bar[ratio_err_bar == np.inf] = 0
        ratio_err_bar = np.nan_to_num(ratio_err_bar)
        if (not isinstance(nbins, int)):
            axs[1].errorbar(mid, ratio_data_mc, xerr=xerr_reco_nuE, yerr=ratio_err_bar, color='black', label=label_data, fmt='o')
        else:
            axs[1].errorbar(mid, ratio_data_mc, xerr=0.5*(xmax-xmin)/nbins, yerr=ratio_err_bar, color='black', label=label_data, fmt='o')
    
    # draw gray area error
    if (not isinstance(nbins, int)):
        ratio_center = np.ones(len(nbins)-1)
    else:
        ratio_center = np.ones(nbins)
    with np.errstate(divide='ignore', invalid='ignore'):
        num = total_stat
        den = total_mc
        ratio_err_area = np.true_divide(num, den)
        ratio_err_area[ratio_err_area == np.inf] = 0
        ratio_err_area = np.nan_to_num(ratio_err_area)
    upvals = np.append((np.array(ratio_center)+np.array(ratio_err_area)),(np.array(ratio_center)+np.array(ratio_err_area))[-1])
    lowvals = np.append((np.array(ratio_center)-np.array(ratio_err_area)),(np.array(ratio_center)-np.array(ratio_err_area))[-1])
    axs[1].fill_between(b_data, lowvals, upvals, step='post', color='gray', alpha=0.3, zorder=2)
    
    plt.savefig('%s.png' % plot_png_name, dpi=600)
    plt.tight_layout()
    plt.show()

## Importing files for the runs

In [36]:
file_beam_on_run1_RHC = '/scratch/wwang/ROOT_files/run1/checkout_data_numi_run1_RHC.root'
file_mc_run1_RHC = '/scratch/wwang/ROOT_files/run1/checkout_prodgenie_numi_newflux_run1_rhc_nu_overlay.root'
file_mc_run1_FHC = '/scratch/wwang/ROOT_files/run1/checkout_prodgenie_numi_newflux_run1_fhc_nu_overlay.root'
file_beam_on_run1_FHC = '/scratch/wwang/ROOT_files/run1/checkout_data_numi_run1_FHC.root'
file_dirt_run1_FHC = '/scratch/wwang/ROOT_files/run1/checkout_prodgenie_run1_fhc_dirt.root'
file_dirt_run1_RHC = '/scratch/wwang/ROOT_files/run1/checkout_prodgenie_run3_rhc_dirt.root'

file_ext_run1 = '/scratch/wwang/ROOT_files/run1/checkout_data_extnumi_run1.root'

In [37]:
file_beam_on_run2_RHC = '/scratch/wwang/ROOT_files/run2/checkout_data_numi_run2_RHC.root'
file_mc_run2_RHC = '/scratch/wwang/ROOT_files/run2/checkout_prodgenie_numi_newflux_run2_rhc_nu_overlay.root'
file_mc_run2_FHC = '/scratch/wwang/ROOT_files/run2/checkout_prodgenie_numi_newflux_run2_fhc_nu_overlay.root'
file_beam_on_run2_FHC = '/scratch/wwang/ROOT_files/run2/checkout_data_numi_run2_FHC.root'
file_dirt_run2_FHC = '/scratch/wwang/ROOT_files/run2/checkout_prodgenie_run2_fhc_dirt.root'
file_dirt_run2_RHC = '/scratch/wwang/ROOT_files/run2/checkout_prodgenie_run2_rhc_dirt.root'

file_ext_run2 = '/scratch/wwang/ROOT_files/run2/checkout_data_extnumi_run2.root'

In [38]:
# Run 3 Only RHC available

file_mc_run3 = '/scratch/wwang/ROOT_files/run3/checkout_prodgenie_numi_newflux_run3_rhc_nu_overlay.root'
file_beam_on_run3 = '/scratch/wwang/ROOT_files/run3/checkout_data_numi_run3_RHC.root'
file_dirt_run3 = '/scratch/wwang/ROOT_files/run3/checkout_prodgenie_run3_rhc_dirt.root'

file_ext_run3 = '/scratch/wwang/ROOT_files/run3/checkout_data_extnumi_run3.root'

In [39]:
# Run 4a Only RHC available

file_beam_on_run4a = '/scratch/wwang/ROOT_files/run4a/checkout_run4a_rhc_beamon.root.root'
file_mc_run4a = '/scratch/wwang/ROOT_files/run4a/checkout_prodgenie_numi_newflux_run4a_rhc_nu_overlay.root'
file_dirt_run4a = '/scratch/wwang/ROOT_files/run4a/checkout_prodgenie_run3_rhc_dirt.root'

file_ext_run4a = '/scratch/wwang/ROOT_files/run4a/checkout_run4a_rhc_beamoff.root.root'

In [40]:
# Run 4b Only RHC available

file_beam_on_run4b = '/scratch/wwang/ROOT_files/run4b/checkout_run4b_rhc_beamon.root.root'
file_mc_run4b = '/scratch/wwang/ROOT_files/run4b/checkout_prodgenie_numi_newflux_run4b_rhc_nu_overlay.root'
file_dirt_run4b = '/scratch/wwang/ROOT_files/run4b/checkout_prodgenie_run3_rhc_dirt.root'

file_ext_run4b = '/scratch/wwang/ROOT_files/run4b/checkout_run4b_rhc_beamoff.root.root'

In [41]:
file_beam_on_run4c_RHC = '/scratch/wwang/ROOT_files/run4c/checkout_run4c_rhc_beamon.root.root'
file_mc_run4c_RHC = '/scratch/wwang/ROOT_files/run4c/checkout_prodgenie_numi_newflux_run4c_rhc_nu_overlay.root'
file_dirt_run4c_RHC = '/scratch/wwang/ROOT_files/run4c/checkout_prodgenie_run3_rhc_dirt.root'
file_ext_run4c_RHC = '/scratch/wwang/ROOT_files/run4c/checkout_run4c_rhc_beamoff.root.root'

file_beam_on_run4c_FHC = '/scratch/wwang/ROOT_files/run4c/checkout_run4c_fhc_beamon.root.root'
file_mc_run4c_FHC = '/scratch/wwang/ROOT_files/run4c/checkout_prodgenie_numi_newflux_run4c_fhc_nu_overlay.root'
file_dirt_run4c_FHC = '/scratch/wwang/ROOT_files/run4c/checkout_prodgenie_run1_fhc_dirt.root'
file_ext_run4c_FHC = '/scratch/wwang/ROOT_files/run4c/checkout_run4c_fhc_beamoff.root.root'

In [42]:
# Run 4d Only FHC available
file_beam_on_run4d = '/scratch/wwang/ROOT_files/run4d/checkout_run4d_fhc_beamon.root.root'
file_mc_run4d = '/scratch/wwang/ROOT_files/run4d/checkout_prodgenie_numi_newflux_run4d_fhc_nu_overlay.root'
file_dirt_run4d = '/scratch/wwang/ROOT_files/run4d/checkout_prodgenie_run1_fhc_dirt.root'
file_ext_run4d = '/scratch/wwang/ROOT_files/run4d/checkout_run4d_fhc_beamoff.root.root'

In [43]:
# Run 5 Only FHC available
file_beam_on_run5 = '/scratch/wwang/ROOT_files/run5/checkout_run5_fhc_beamon.root.root'
file_mc_run5 = '/scratch/wwang/ROOT_files/run5/checkout_prodgenie_numi_newflux_run5_fhc_nu_overlay.root'
file_dirt_run5 = '/scratch/wwang/ROOT_files/run5/checkout_prodgenie_run1_fhc_dirt.root'
file_ext_run5 = '/scratch/wwang/ROOT_files/run5/checkout_run5_fhc_beamoff.root.root'

## Getting the POT for each runs

In [44]:
# Run 1 FHC
run1_FHC_pot = 1.988e+20  # tortgt_wcut

# Run 1 RHC
run1_RHC_pot = 5.973e+19  # tortgt_wcut 

# Total Run 1 POT
run1_pot_beam_on = run1_FHC_pot + run1_RHC_pot

# EXT information
run1_EXT_EA9CNT_wcut = 6820803.0
run1_EXT_NUMIwin_FEMBeamTriggerAlgo = 7642688.005
run1_ext_pot = run1_pot_beam_on / (run1_EXT_EA9CNT_wcut / run1_EXT_NUMIwin_FEMBeamTriggerAlgo)

run1_pot_mc_FHC = calc_pot(file_mc_run1_FHC, 'MC')
run1_pot_mc_RHC = calc_pot(file_mc_run1_RHC, 'MC')
run1_pot_mc = run1_pot_mc_FHC + run1_pot_mc_RHC
run1_pot_dirt_FHC = calc_pot(file_dirt_run1_FHC, 'DIRT')
run1_pot_dirt_RHC = calc_pot(file_dirt_run1_RHC, 'DIRT')
run1_pot_dirt = run1_pot_dirt_FHC + run1_pot_dirt_RHC

print("Run 1 FHC DATA POT = %.2e" % run1_FHC_pot)
print("Run 1 MC POT FHC = %.2e" % run1_pot_mc_FHC)
print("Run 1 DIRT POT FHC = %.2e" % run1_pot_dirt_FHC)


print("Run 1 RHC DATA POT = %.2e" % run1_RHC_pot) 
print("Run 1 MC POT RHC = %.2e" % run1_pot_mc_RHC)
print("Run 1 DIRT POT RHC = %.2e" % run1_pot_dirt_RHC)

print("Run 1 EXT POT = %.2e" % run1_ext_pot)

Run 1 FHC DATA POT = 1.99e+20
Run 1 MC POT FHC = 2.28e+21
Run 1 DIRT POT FHC = 6.15e+20
Run 1 RHC DATA POT = 5.97e+19
Run 1 MC POT RHC = 8.66e+20
Run 1 DIRT POT RHC = 9.05e+20
Run 1 EXT POT = 2.90e+20


In [45]:
# Run 2 FHC
run2_FHC_pot = 1.29e+20  # tortgt_wcut

# Run 2 RHC
run2_RHC_pot = 2.306e+20  # tortgt_wcut 
# Print individual POTs
print("Run 2 FHC POT = %.2e" % run2_FHC_pot)
print("Run 2 RHC POT = %.2e" % run2_RHC_pot)

# Total Run 2 POT
run2_total_pot = run2_FHC_pot + run2_RHC_pot
print("Run 2 TOTAL POT = %.2e" % run2_total_pot)


run2_EXT_EA9CNT_wcut = 8796279.0
run2_EXT_NUMIwin_FEMBeamTriggerAlgo = 25847010.350000
run2_ext_pot = run2_total_pot / (run2_EXT_EA9CNT_wcut / run2_EXT_NUMIwin_FEMBeamTriggerAlgo)
print("Run 2 EXT POT (updated) = %.2e" % run2_ext_pot)

run2_pot_mc_FHC = calc_pot(file_mc_run2_FHC, 'MC')
run2_pot_mc_RHC = calc_pot(file_mc_run2_RHC, 'MC')
run2_pot_mc = run2_pot_mc_FHC + run2_pot_mc_RHC
run2_pot_dirt_FHC = calc_pot(file_dirt_run2_FHC, 'DIRT')
run2_pot_dirt_RHC = calc_pot(file_dirt_run2_RHC, 'DIRT')
run2_pot_dirt = run2_pot_dirt_FHC + run2_pot_dirt_RHC

print("Run 2 FHC DATA POT = %.2e" % run2_FHC_pot)
print("Run 2 MC POT FHC = %.2e" % run2_pot_mc_FHC)
print("Run 2 DIRT POT FHC = %.2e" % run2_pot_dirt_FHC)


print("Run 2 RHC DATA POT = %.2e" % run2_RHC_pot) 
print("Run 2 MC POT RHC = %.2e" % run2_pot_mc_RHC)
print("Run 2 DIRT POT RHC = %.2e" % run2_pot_dirt_RHC)

print("Run 2 EXT POT = %.2e" % run2_ext_pot)

Run 2 FHC POT = 1.29e+20
Run 2 RHC POT = 2.31e+20
Run 2 TOTAL POT = 3.60e+20
Run 2 EXT POT (updated) = 1.06e+21
Run 2 FHC DATA POT = 1.29e+20
Run 2 MC POT FHC = 2.42e+21
Run 2 DIRT POT FHC = 1.37e+20
Run 2 RHC DATA POT = 2.31e+20
Run 2 MC POT RHC = 5.59e+21
Run 2 DIRT POT RHC = 1.07e+20
Run 2 EXT POT = 1.06e+21


In [46]:
#———————————————————————————————#
#Calculate pot for each dataset # RUN 3
#———————————————————————————————#
    

#From the result of run3 beam-on POT calculation
run3b_pot_beam_on = 4.562e+20

#EXT info
run3b_beam_off_hw = 30445265.025000 # EXT_NUMIwin_FEMBeamTriggerAlgo
run3b_EXT_EA9CNT_wcut = 9334387.0

run3b_pot_ext = run3b_pot_beam_on/(run3b_EXT_EA9CNT_wcut/run3b_beam_off_hw)

run3b_pot_mc = calc_pot(file_mc_run3, 'MC')
run3b_pot_dirt = calc_pot(file_dirt_run3, 'DIRT')

# Sanity check cell
print("Run 3 DATA POT = %.2e" % run3b_pot_beam_on)
print("Run 3 EXT POT = %.2e" % run3b_pot_ext)
print("Run 3 MC POT = %.2e" % run3b_pot_mc)
print("Run 3 DIRT POT = %.2e" % run3b_pot_dirt)

Run 3 DATA POT = 4.56e+20
Run 3 EXT POT = 1.49e+21
Run 3 MC POT = 5.45e+21
Run 3 DIRT POT = 9.05e+20


In [47]:
#———————————————————————————————#
#Calculate pot for each dataset # RUN 4a
#———————————————————————————————#
    

#From the result of run4a RHC beam-on POT calculation
run4a_RHC_pot = 8.034e+18

run4a_EXT_EA9CNT_wcut = 330138.0
run4a_EXT_NUMIwin_FEMBeamTriggerAlgo = 4923095.45
run4a_ext_pot = run4a_RHC_pot / (run4a_EXT_EA9CNT_wcut / run4a_EXT_NUMIwin_FEMBeamTriggerAlgo)

run4a_pot_mc = calc_pot(file_mc_run4a, 'MC')
run4a_pot_dirt = calc_pot(file_dirt_run4a, 'DIRT')

# Sanity check cell
print("Run 4a RHC DATA POT = %.2e" % run4a_RHC_pot)
print("Run 4a EXT POT = %.2e" % run4a_ext_pot)
print("Run 4a MC POT = %.2e" % run4a_pot_mc)
print("Run 4a DIRT POT = %.2e" % run4a_pot_dirt)

Run 4a RHC DATA POT = 8.03e+18
Run 4a EXT POT = 1.20e+20
Run 4a MC POT = 7.37e+20
Run 4a DIRT POT = 9.05e+20


In [48]:
#———————————————————————————————#
#Calculate pot for each dataset # RUN 4a
#———————————————————————————————#
    

#From the result of run4b RHC beam-on POT calculation
run4b_RHC_pot = 2.615e+20

run4b_EXT_EA9CNT_wcut = 5480981.0
run4b_EXT_NUMIwin_FEMBeamTriggerAlgo = 15479433.9
run4b_ext_pot = run4b_RHC_pot / (run4b_EXT_EA9CNT_wcut / run4b_EXT_NUMIwin_FEMBeamTriggerAlgo)
run4b_pot_mc = calc_pot(file_mc_run4b, 'MC')
run4b_pot_dirt = calc_pot(file_dirt_run4b, 'DIRT')

# Sanity check cell
print("Run 4b RHC DATA POT = %.2e" % run4b_RHC_pot)
print("Run 4b EXT POT = %.2e" % run4b_ext_pot)
print("Run 4b MC POT = %.2e" % run4b_pot_mc)
print("Run 4b DIRT POT = %.2e" % run4b_pot_dirt)

Run 4b RHC DATA POT = 2.62e+20
Run 4b EXT POT = 7.39e+20
Run 4b MC POT = 2.30e+21
Run 4b DIRT POT = 9.05e+20


In [49]:
# Run 4c FHC
run4c_FHC_pot = 1.012e+20  # tortgt_wcut

# Run 4c RHC
run4c_RHC_pot = 1.365e+19  # tortgt_wcut 


run4c_EXT_EA9CNT_wcut_FHC = 1933296.0
run4c_EXT_NUMIwin_FEMBeamTriggerAlgo_FHC = 6532201.55
run4c_ext_pot_fhc = run4c_FHC_pot / (run4c_EXT_EA9CNT_wcut_FHC / run4c_EXT_NUMIwin_FEMBeamTriggerAlgo_FHC)

run4c_EXT_EA9CNT_wcut_RHC = 271478.0
run4c_EXT_NUMIwin_FEMBeamTriggerAlgo_RHC = 848948.625
run4c_ext_pot_rhc = run4c_RHC_pot / (run4c_EXT_EA9CNT_wcut_RHC / run4c_EXT_NUMIwin_FEMBeamTriggerAlgo_RHC)

run4c_pot_mc_FHC = calc_pot(file_mc_run4c_FHC, 'MC')
run4c_pot_mc_RHC = calc_pot(file_mc_run4c_RHC, 'MC')
run4c_pot_mc = run4c_pot_mc_FHC + run4c_pot_mc_RHC
run4c_pot_dirt_FHC = calc_pot(file_dirt_run4c_FHC, 'DIRT')
run4c_pot_dirt_RHC = calc_pot(file_dirt_run4c_RHC, 'DIRT')
run4c_pot_dirt = run4c_pot_dirt_FHC + run4c_pot_dirt_RHC

print("Run 4c FHC DATA POT = %.2e" % run4c_FHC_pot)
print("Run 4c MC POT FHC = %.2e" % run4c_pot_mc_FHC)
print("Run 4c DIRT POT FHC = %.2e" % run4c_pot_dirt_FHC)
print("Run 4c EXT FHC POT = %.2e" % run4c_ext_pot_fhc)

print("Run 4c RHC DATA POT = %.2e" % run4c_RHC_pot) 
print("Run 4c MC POT RHC = %.2e" % run4c_pot_mc_RHC)
print("Run 4c DIRT POT RHC = %.2e" % run4c_pot_dirt_RHC)
print("Run 4c EXT RHC POT = %.2e" % run4c_ext_pot_rhc)

Run 4c FHC DATA POT = 1.01e+20
Run 4c MC POT FHC = 1.09e+21
Run 4c DIRT POT FHC = 6.15e+20
Run 4c EXT FHC POT = 3.42e+20
Run 4c RHC DATA POT = 1.36e+19
Run 4c MC POT RHC = 1.46e+20
Run 4c DIRT POT RHC = 9.05e+20
Run 4c EXT RHC POT = 4.27e+19


In [50]:
#———————————————————————————————#
#Calculate pot for each dataset # RUN 4a
#———————————————————————————————#
    

#From the result of run4d FHC beam-on POT calculation
run4d_FHC_pot = 8.012e+19

run4d_EXT_EA9CNT_wcut = 1449858.0
run4d_EXT_NUMIwin_FEMBeamTriggerAlgo = 28343856.625
run4d_ext_pot = run4d_FHC_pot / (run4d_EXT_EA9CNT_wcut / run4d_EXT_NUMIwin_FEMBeamTriggerAlgo)
run4d_pot_mc = calc_pot(file_mc_run4d, 'MC')
run4d_pot_dirt = calc_pot(file_dirt_run4d, 'DIRT')

# Sanity check cell
print("Run 4d FHC DATA POT = %.2e" % run4d_FHC_pot)
print("Run 4d EXT POT = %.2e" % run4d_ext_pot)
print("Run 4d MC POT = %.2e" % run4d_pot_mc)
print("Run 4d DIRT POT = %.2e" % run4d_pot_dirt)

Run 4d FHC DATA POT = 8.01e+19
Run 4d EXT POT = 1.57e+21
Run 4d MC POT = 1.74e+21
Run 4d DIRT POT = 6.15e+20


In [51]:
#———————————————————————————————#
#Calculate pot for each dataset # RUN 4a
#———————————————————————————————#
    

#From the result of run5 FHC beam-on POT calculation
run5_FHC_pot = 2.14e+20

run5_EXT_EA9CNT_wcut = 4906558.0
run5_EXT_NUMIwin_FEMBeamTriggerAlgo = 18907129.15
run5_ext_pot = run5_FHC_pot / (run5_EXT_EA9CNT_wcut / run5_EXT_NUMIwin_FEMBeamTriggerAlgo)
run5_pot_mc = calc_pot(file_mc_run5, 'MC')
run5_pot_dirt = calc_pot(file_dirt_run5, 'DIRT')
# Sanity check cell
print("Run 5 FHC DATA POT = %.2e" % run5_FHC_pot)
print("Run 5 EXT POT = %.2e" % run5_ext_pot)
print("Run 5 MC POT = %.2e" % run5_pot_mc)
print("Run 5 DIRT POT = %.2e" % run5_pot_dirt)

Run 5 FHC DATA POT = 2.14e+20
Run 5 EXT POT = 8.25e+20
Run 5 MC POT = 2.73e+21
Run 5 DIRT POT = 6.15e+20


In [52]:
FHC_full_POT = run1_FHC_pot + run2_FHC_pot + run4c_FHC_pot + run4d_FHC_pot + run5_FHC_pot
RHC_full_POT = run1_RHC_pot + run2_RHC_pot + run3b_pot_beam_on + run4a_RHC_pot + run4b_RHC_pot + run4c_RHC_pot

print("Total FHC POT = %.2e" % FHC_full_POT)
print("Total RHC POT = %.2e" % RHC_full_POT)

Total FHC POT = 7.23e+20
Total RHC POT = 1.03e+21


## Creating dataframes for the runs

In [53]:
if lock1==True: 
    df_beam_on_run1_RHC = new_df(file_beam_on_run1_RHC, 'DATA', run1_RHC_pot, run1_ext_pot, run1_pot_mc_RHC, run1_pot_dirt_RHC)
    df_ext_run1_RHC = new_df(file_ext_run1, 'EXT', run1_RHC_pot, run1_ext_pot, run1_pot_mc_RHC, run1_pot_dirt_RHC)
    df_mc_run1_RHC = new_df(file_mc_run1_RHC, 'MC', run1_RHC_pot, run1_ext_pot, run1_pot_mc_RHC, run1_pot_dirt_RHC)
    df_dirt_run1_RHC = new_df(file_dirt_run1_RHC, 'DIRT', run1_RHC_pot, run1_ext_pot, run1_pot_mc_RHC, run1_pot_dirt_RHC)

    df_beam_on_run1_FHC = new_df(file_beam_on_run1_FHC, 'DATA', run1_FHC_pot, run1_ext_pot, run1_pot_mc_FHC, run1_pot_dirt_FHC)
    df_ext_run1_FHC = new_df(file_ext_run1, 'EXT', run1_FHC_pot, run1_ext_pot, run1_pot_mc_FHC, run1_pot_dirt_FHC)
    df_mc_run1_FHC = new_df(file_mc_run1_FHC, 'MC', run1_FHC_pot, run1_ext_pot, run1_pot_mc_FHC, run1_pot_dirt_FHC)
    df_dirt_run1_FHC = new_df(file_dirt_run1_FHC, 'DIRT', run1_FHC_pot, run1_ext_pot, run1_pot_mc_FHC, run1_pot_dirt_FHC)
    print('\nDataframes created.')

In [54]:
if lock1==True: 
    # Concatenate RHC and FHC dataframes for each sample type (DATA, EXT, MC, DIRT) and reset index
    df_beam_on_run1 = pd.concat([df_beam_on_run1_RHC, df_beam_on_run1_FHC], ignore_index=True)
    df_ext_run1 = pd.concat([df_ext_run1_RHC, df_ext_run1_FHC], ignore_index=True)
    df_mc_run1 = pd.concat([df_mc_run1_RHC, df_mc_run1_FHC], ignore_index=True)
    df_dirt_run1 = pd.concat([df_dirt_run1_RHC, df_dirt_run1_FHC], ignore_index=True)
    # EXT is only one file for run1, no need to concat

In [55]:
if lock1==True:
    df_beam_on_run2_RHC = new_df(file_beam_on_run2_RHC, 'DATA', run2_RHC_pot, run2_ext_pot, run2_pot_mc_RHC, run2_pot_dirt_RHC)
    df_ext_run2_RHC = new_df(file_ext_run2, 'EXT', run2_RHC_pot, run2_ext_pot, run2_pot_mc_RHC, run2_pot_dirt_RHC)
    df_mc_run2_RHC = new_df(file_mc_run2_RHC, 'MC', run2_RHC_pot, run2_ext_pot, run2_pot_mc_RHC, run2_pot_dirt_RHC)
    df_dirt_run2_RHC = new_df(file_dirt_run2_RHC, 'DIRT', run2_RHC_pot, run2_ext_pot, run2_pot_mc_RHC, run2_pot_dirt_RHC)

    df_beam_on_run2_FHC = new_df(file_beam_on_run2_FHC, 'DATA', run2_FHC_pot, run2_ext_pot, run2_pot_mc_FHC, run2_pot_dirt_FHC)
    df_ext_run2_FHC = new_df(file_ext_run2, 'EXT', run2_FHC_pot, run2_ext_pot, run2_pot_mc_FHC, run2_pot_dirt_FHC)
    df_mc_run2_FHC = new_df(file_mc_run2_FHC, 'MC', run2_FHC_pot, run2_ext_pot, run2_pot_mc_FHC, run2_pot_dirt_FHC)
    df_dirt_run2_FHC = new_df(file_dirt_run2_FHC, 'DIRT', run2_FHC_pot, run2_ext_pot, run2_pot_mc_FHC, run2_pot_dirt_FHC)
    print('\nDataframes created.')

In [56]:
if lock1==True:
    # Concatenate RHC and FHC dataframes for each sample type (DATA, EXT, MC, DIRT) and reset index
    df_beam_on_run2 = pd.concat([df_beam_on_run2_RHC, df_beam_on_run2_FHC], ignore_index=True)
    df_ext_run2 = pd.concat([df_ext_run2_RHC, df_ext_run2_FHC], ignore_index=True)
    df_mc_run2 = pd.concat([df_mc_run2_RHC, df_mc_run2_FHC], ignore_index=True)
    df_dirt_run2 = pd.concat([df_dirt_run2_RHC, df_dirt_run2_FHC], ignore_index=True)

In [57]:
#——————————————————————————————————————#
# Create dataframes containing NTuples #
#——————————————————————————————————————#

if lock1==True:    
    
    df_beam_on_run3 = new_df(file_beam_on_run3, 'DATA', run3b_pot_beam_on, run3b_pot_ext, run3b_pot_mc, run3b_pot_dirt)
    df_ext_run3 = new_df(file_ext_run3, 'EXT', run3b_pot_beam_on, run3b_pot_ext, run3b_pot_mc, run3b_pot_dirt)
    df_mc_run3 = new_df(file_mc_run3, 'MC', run3b_pot_beam_on, run3b_pot_ext, run3b_pot_mc, run3b_pot_dirt)
    df_dirt_run3 = new_df(file_dirt_run3, 'DIRT', run3b_pot_beam_on, run3b_pot_ext, run3b_pot_mc, run3b_pot_dirt)
    print('\nDataframes created.')

In [58]:
if lock1==True:    
    df_beam_on_run4a = new_df(file_beam_on_run4a, 'DATA', run4a_RHC_pot, run4a_ext_pot, run4a_pot_mc, run4a_pot_dirt)
    df_ext_run4a = new_df(file_ext_run4a, 'EXT', run4a_RHC_pot, run4a_ext_pot, run4a_pot_mc, run4a_pot_dirt)
    df_mc_run4a = new_df(file_mc_run4a, 'MC', run4a_RHC_pot, run4a_ext_pot, run4a_pot_mc, run4a_pot_dirt)
    df_dirt_run4a = new_df(file_dirt_run4a, 'DIRT', run4a_RHC_pot, run4a_ext_pot, run4a_pot_mc, run4a_pot_dirt)
    print('\nDataframes created.')

In [59]:
if lock1==True:
    df_beam_on_run4b = new_df(file_beam_on_run4b, 'DATA', run4b_RHC_pot, run4b_ext_pot, run4b_pot_mc, run4b_pot_dirt)
    df_ext_run4b = new_df(file_ext_run4b, 'EXT', run4b_RHC_pot, run4b_ext_pot, run4b_pot_mc, run4b_pot_dirt)
    df_mc_run4b = new_df(file_mc_run4b, 'MC', run4b_RHC_pot, run4b_ext_pot, run4b_pot_mc, run4b_pot_dirt)
    df_dirt_run4b = new_df(file_dirt_run4b, 'DIRT', run4b_RHC_pot, run4b_ext_pot, run4b_pot_mc, run4b_pot_dirt)
    print('\nDataframes created.')

In [60]:
if lock1==True:
    df_beam_on_run4c_RHC = new_df(file_beam_on_run4c_RHC, 'DATA', run4c_RHC_pot, run4c_ext_pot_rhc, run4c_pot_mc_RHC, run4c_pot_dirt_RHC)
    df_ext_run4c_RHC = new_df(file_ext_run4c_RHC, 'EXT', run4c_RHC_pot, run4c_ext_pot_rhc, run4c_pot_mc_RHC, run4c_pot_dirt_RHC)
    df_mc_run4c_RHC = new_df(file_mc_run4c_RHC, 'MC', run4c_RHC_pot, run4c_ext_pot_rhc, run4c_pot_mc_RHC, run4c_pot_dirt_RHC)
    df_dirt_run4c_RHC = new_df(file_dirt_run4c_RHC, 'DIRT', run4c_RHC_pot, run4c_ext_pot_rhc, run4c_pot_mc_RHC, run4c_pot_dirt_RHC)

    df_beam_on_run4c_FHC = new_df(file_beam_on_run4c_FHC, 'DATA', run4c_FHC_pot, run4c_ext_pot_fhc, run4c_pot_mc_FHC, run4c_pot_dirt_FHC)
    df_ext_run4c_FHC = new_df(file_ext_run4c_FHC, 'EXT', run4c_FHC_pot, run4c_ext_pot_fhc, run4c_pot_mc_FHC, run4c_pot_dirt_FHC)
    df_mc_run4c_FHC = new_df(file_mc_run4c_FHC, 'MC', run4c_FHC_pot, run4c_ext_pot_fhc, run4c_pot_mc_FHC, run4c_pot_dirt_FHC)
    df_dirt_run4c_FHC = new_df(file_dirt_run4c_FHC, 'DIRT', run4c_FHC_pot, run4c_ext_pot_fhc, run4c_pot_mc_FHC, run4c_pot_dirt_FHC)
    print('\nDataframes created.')

In [61]:
if lock1==True:
    # Concatenate RHC and FHC dataframes for each sample type (DATA, EXT, MC, DIRT) and reset index
    df_beam_on_run4c = pd.concat([df_beam_on_run4c_RHC, df_beam_on_run4c_FHC], ignore_index=True)
    df_ext_run4c = pd.concat([df_ext_run4c_RHC, df_ext_run4c_FHC], ignore_index=True)
    df_mc_run4c = pd.concat([df_mc_run4c_RHC, df_mc_run4c_FHC], ignore_index=True)
    df_dirt_run4c = pd.concat([df_dirt_run4c_RHC, df_dirt_run4c_FHC], ignore_index=True)

In [62]:
if lock1==True:
    df_beam_on_run4d = new_df(file_beam_on_run4d, 'DATA', run4d_FHC_pot, run4d_ext_pot, run4d_pot_mc, run4d_pot_dirt)
    df_ext_run4d = new_df(file_ext_run4d, 'EXT', run4d_FHC_pot, run4d_ext_pot, run4d_pot_mc, run4d_pot_dirt)
    df_mc_run4d = new_df(file_mc_run4d, 'MC', run4d_FHC_pot, run4d_ext_pot, run4d_pot_mc, run4d_pot_dirt)
    df_dirt_run4d = new_df(file_dirt_run4d, 'DIRT', run4d_FHC_pot, run4d_ext_pot, run4d_pot_mc, run4d_pot_dirt)
    print('\nDataframes created.')

In [63]:
if lock1==True:
    df_beam_on_run5 = new_df(file_beam_on_run5, 'DATA', run5_FHC_pot, run5_ext_pot, run5_pot_mc, run5_pot_dirt)
    df_ext_run5 = new_df(file_ext_run5, 'EXT', run5_FHC_pot, run5_ext_pot, run5_pot_mc, run5_pot_dirt)
    df_mc_run5 = new_df(file_mc_run5, 'MC', run5_FHC_pot, run5_ext_pot, run5_pot_mc, run5_pot_dirt)
    df_dirt_run5 = new_df(file_dirt_run5, 'DIRT', run5_FHC_pot, run5_ext_pot, run5_pot_mc, run5_pot_dirt)
    print('\nDataframes created.')

## Save the created dfs into pickle files

In [64]:
# ============================================================
# Save all individual per-run dataframes to pickle
# Match filenames used by the load block exactly
# ============================================================

if lock1:

    # -----------------
    # Run 1
    # -----------------
    df_beam_on_run1_RHC.to_pickle(os.path.join(dfs_dir, 'df_beam_on_run1_RHC.pkl'))
    df_ext_run1_RHC.to_pickle(os.path.join(dfs_dir, 'df_ext_run1_RHC.pkl'))
    df_mc_run1_RHC.to_pickle(os.path.join(dfs_dir, 'df_mc_run1_RHC.pkl'))
    df_dirt_run1_RHC.to_pickle(os.path.join(dfs_dir, 'df_dirt_run1_RHC.pkl'))

    df_beam_on_run1_FHC.to_pickle(os.path.join(dfs_dir, 'df_beam_on_run1_FHC.pkl'))
    df_ext_run1_FHC.to_pickle(os.path.join(dfs_dir, 'df_ext_run1_FHC.pkl'))
    df_mc_run1_FHC.to_pickle(os.path.join(dfs_dir, 'df_mc_run1_FHC.pkl'))
    df_dirt_run1_FHC.to_pickle(os.path.join(dfs_dir, 'df_dirt_run1_FHC.pkl'))

    print("Run 1 dataframes saved.")

    # -----------------
    # Run 2
    # -----------------
    df_beam_on_run2_RHC.to_pickle(os.path.join(dfs_dir, 'df_beam_on_run2_RHC.pkl'))
    df_ext_run2_RHC.to_pickle(os.path.join(dfs_dir, 'df_ext_run2_RHC.pkl'))
    df_mc_run2_RHC.to_pickle(os.path.join(dfs_dir, 'df_mc_run2_RHC.pkl'))
    df_dirt_run2_RHC.to_pickle(os.path.join(dfs_dir, 'df_dirt_run2_RHC.pkl'))

    df_beam_on_run2_FHC.to_pickle(os.path.join(dfs_dir, 'df_beam_on_run2_FHC.pkl'))
    df_ext_run2_FHC.to_pickle(os.path.join(dfs_dir, 'df_ext_run2_FHC.pkl'))
    df_mc_run2_FHC.to_pickle(os.path.join(dfs_dir, 'df_mc_run2_FHC.pkl'))
    df_dirt_run2_FHC.to_pickle(os.path.join(dfs_dir, 'df_dirt_run2_FHC.pkl'))

    print("Run 2 dataframes saved.")

    # -----------------
    # Run 3
    # -----------------
    df_beam_on_run3.to_pickle(os.path.join(dfs_dir, 'df_beam_on_run3_RHC.pkl'))
    df_ext_run3.to_pickle(os.path.join(dfs_dir, 'df_ext_run3_RHC.pkl'))
    df_mc_run3.to_pickle(os.path.join(dfs_dir, 'df_mc_run3_RHC.pkl'))
    df_dirt_run3.to_pickle(os.path.join(dfs_dir, 'df_dirt_run3_RHC.pkl'))

    print("Run 3 dataframes saved.")

    # -----------------
    # Run 4a
    # -----------------
    df_beam_on_run4a.to_pickle(os.path.join(dfs_dir, 'df_beam_on_run4a_RHC.pkl'))
    df_ext_run4a.to_pickle(os.path.join(dfs_dir, 'df_ext_run4a_RHC.pkl'))
    df_mc_run4a.to_pickle(os.path.join(dfs_dir, 'df_mc_run4a_RHC.pkl'))
    df_dirt_run4a.to_pickle(os.path.join(dfs_dir, 'df_dirt_run4a_RHC.pkl'))

    print("Run 4a dataframes saved.")

    # -----------------
    # Run 4b
    # -----------------
    df_beam_on_run4b.to_pickle(os.path.join(dfs_dir, 'df_beam_on_run4b_RHC.pkl'))
    df_ext_run4b.to_pickle(os.path.join(dfs_dir, 'df_ext_run4b_RHC.pkl'))
    df_mc_run4b.to_pickle(os.path.join(dfs_dir, 'df_mc_run4b_RHC.pkl'))
    df_dirt_run4b.to_pickle(os.path.join(dfs_dir, 'df_dirt_run4b_RHC.pkl'))

    print("Run 4b dataframes saved.")

    # -----------------
    # Run 4c
    # -----------------
    df_beam_on_run4c_RHC.to_pickle(os.path.join(dfs_dir, 'df_beam_on_run4c_RHC.pkl'))
    df_ext_run4c_RHC.to_pickle(os.path.join(dfs_dir, 'df_ext_run4c_RHC.pkl'))
    df_mc_run4c_RHC.to_pickle(os.path.join(dfs_dir, 'df_mc_run4c_RHC.pkl'))
    df_dirt_run4c_RHC.to_pickle(os.path.join(dfs_dir, 'df_dirt_run4c_RHC.pkl'))

    df_beam_on_run4c_FHC.to_pickle(os.path.join(dfs_dir, 'df_beam_on_run4c_FHC.pkl'))
    df_ext_run4c_FHC.to_pickle(os.path.join(dfs_dir, 'df_ext_run4c_FHC.pkl'))
    df_mc_run4c_FHC.to_pickle(os.path.join(dfs_dir, 'df_mc_run4c_FHC.pkl'))
    df_dirt_run4c_FHC.to_pickle(os.path.join(dfs_dir, 'df_dirt_run4c_FHC.pkl'))

    print("Run 4c dataframes saved.")

    # -----------------
    # Run 4d
    # -----------------
    df_beam_on_run4d.to_pickle(os.path.join(dfs_dir, 'df_beam_on_run4d_FHC.pkl'))
    df_ext_run4d.to_pickle(os.path.join(dfs_dir, 'df_ext_run4d_FHC.pkl'))
    df_mc_run4d.to_pickle(os.path.join(dfs_dir, 'df_mc_run4d_FHC.pkl'))
    df_dirt_run4d.to_pickle(os.path.join(dfs_dir, 'df_dirt_run4d_FHC.pkl'))

    print("Run 4d dataframes saved.")

    # -----------------
    # Run 5
    # -----------------
    df_beam_on_run5.to_pickle(os.path.join(dfs_dir, 'df_beam_on_run5_FHC.pkl'))
    df_ext_run5.to_pickle(os.path.join(dfs_dir, 'df_ext_run5_FHC.pkl'))
    df_mc_run5.to_pickle(os.path.join(dfs_dir, 'df_mc_run5_FHC.pkl'))
    df_dirt_run5.to_pickle(os.path.join(dfs_dir, 'df_dirt_run5_FHC.pkl'))

    print("Run 5 dataframes saved.")
    print("\nAll individual per-run dataframes saved to pickle.")

## Load the dataframes from pickle files

In [65]:
# ============================================================
# Load all individual per-run dataframes from pickle
# Do NOT rebuild combined run dataframes
# ============================================================

if lock2:

    # -----------------
    # Run 1
    # -----------------
    df_beam_on_run1_RHC = pd.read_pickle(os.path.join(dfs_dir, 'df_beam_on_run1_RHC.pkl'))
    df_ext_run1_RHC     = pd.read_pickle(os.path.join(dfs_dir, 'df_ext_run1_RHC.pkl'))
    df_mc_run1_RHC      = pd.read_pickle(os.path.join(dfs_dir, 'df_mc_run1_RHC.pkl'))
    df_dirt_run1_RHC    = pd.read_pickle(os.path.join(dfs_dir, 'df_dirt_run1_RHC.pkl'))

    df_beam_on_run1_FHC = pd.read_pickle(os.path.join(dfs_dir, 'df_beam_on_run1_FHC.pkl'))
    df_ext_run1_FHC     = pd.read_pickle(os.path.join(dfs_dir, 'df_ext_run1_FHC.pkl'))
    df_mc_run1_FHC      = pd.read_pickle(os.path.join(dfs_dir, 'df_mc_run1_FHC.pkl'))
    df_dirt_run1_FHC    = pd.read_pickle(os.path.join(dfs_dir, 'df_dirt_run1_FHC.pkl'))

    print("Run 1 dataframes loaded.")

    # -----------------
    # Run 2
    # -----------------
    df_beam_on_run2_RHC = pd.read_pickle(os.path.join(dfs_dir, 'df_beam_on_run2_RHC.pkl'))
    df_ext_run2_RHC     = pd.read_pickle(os.path.join(dfs_dir, 'df_ext_run2_RHC.pkl'))
    df_mc_run2_RHC      = pd.read_pickle(os.path.join(dfs_dir, 'df_mc_run2_RHC.pkl'))
    df_dirt_run2_RHC    = pd.read_pickle(os.path.join(dfs_dir, 'df_dirt_run2_RHC.pkl'))

    df_beam_on_run2_FHC = pd.read_pickle(os.path.join(dfs_dir, 'df_beam_on_run2_FHC.pkl'))
    df_ext_run2_FHC     = pd.read_pickle(os.path.join(dfs_dir, 'df_ext_run2_FHC.pkl'))
    df_mc_run2_FHC      = pd.read_pickle(os.path.join(dfs_dir, 'df_mc_run2_FHC.pkl'))
    df_dirt_run2_FHC    = pd.read_pickle(os.path.join(dfs_dir, 'df_dirt_run2_FHC.pkl'))

    print("Run 2 dataframes loaded.")

    # -----------------
    # Run 3
    # -----------------
    df_beam_on_run3 = pd.read_pickle(os.path.join(dfs_dir, 'df_beam_on_run3_RHC.pkl'))
    df_ext_run3     = pd.read_pickle(os.path.join(dfs_dir, 'df_ext_run3_RHC.pkl'))
    df_mc_run3      = pd.read_pickle(os.path.join(dfs_dir, 'df_mc_run3_RHC.pkl'))
    df_dirt_run3    = pd.read_pickle(os.path.join(dfs_dir, 'df_dirt_run3_RHC.pkl'))

    print("Run 3 dataframes loaded.")

    # -----------------
    # Run 4a
    # -----------------
    df_beam_on_run4a = pd.read_pickle(os.path.join(dfs_dir, 'df_beam_on_run4a_RHC.pkl'))
    df_ext_run4a     = pd.read_pickle(os.path.join(dfs_dir, 'df_ext_run4a_RHC.pkl'))
    df_mc_run4a      = pd.read_pickle(os.path.join(dfs_dir, 'df_mc_run4a_RHC.pkl'))
    df_dirt_run4a    = pd.read_pickle(os.path.join(dfs_dir, 'df_dirt_run4a_RHC.pkl'))

    print("Run 4a dataframes loaded.")

    # -----------------
    # Run 4b
    # -----------------
    df_beam_on_run4b = pd.read_pickle(os.path.join(dfs_dir, 'df_beam_on_run4b_RHC.pkl'))
    df_ext_run4b     = pd.read_pickle(os.path.join(dfs_dir, 'df_ext_run4b_RHC.pkl'))
    df_mc_run4b      = pd.read_pickle(os.path.join(dfs_dir, 'df_mc_run4b_RHC.pkl'))
    df_dirt_run4b    = pd.read_pickle(os.path.join(dfs_dir, 'df_dirt_run4b_RHC.pkl'))

    print("Run 4b dataframes loaded.")

    # -----------------
    # Run 4c
    # -----------------
    df_beam_on_run4c_RHC = pd.read_pickle(os.path.join(dfs_dir, 'df_beam_on_run4c_RHC.pkl'))
    df_ext_run4c_RHC     = pd.read_pickle(os.path.join(dfs_dir, 'df_ext_run4c_RHC.pkl'))
    df_mc_run4c_RHC      = pd.read_pickle(os.path.join(dfs_dir, 'df_mc_run4c_RHC.pkl'))
    df_dirt_run4c_RHC    = pd.read_pickle(os.path.join(dfs_dir, 'df_dirt_run4c_RHC.pkl'))

    df_beam_on_run4c_FHC = pd.read_pickle(os.path.join(dfs_dir, 'df_beam_on_run4c_FHC.pkl'))
    df_ext_run4c_FHC     = pd.read_pickle(os.path.join(dfs_dir, 'df_ext_run4c_FHC.pkl'))
    df_mc_run4c_FHC      = pd.read_pickle(os.path.join(dfs_dir, 'df_mc_run4c_FHC.pkl'))
    df_dirt_run4c_FHC    = pd.read_pickle(os.path.join(dfs_dir, 'df_dirt_run4c_FHC.pkl'))

    print("Run 4c dataframes loaded.")

    # -----------------
    # Run 4d
    # -----------------
    df_beam_on_run4d = pd.read_pickle(os.path.join(dfs_dir, 'df_beam_on_run4d_FHC.pkl'))
    df_ext_run4d     = pd.read_pickle(os.path.join(dfs_dir, 'df_ext_run4d_FHC.pkl'))
    df_mc_run4d      = pd.read_pickle(os.path.join(dfs_dir, 'df_mc_run4d_FHC.pkl'))
    df_dirt_run4d    = pd.read_pickle(os.path.join(dfs_dir, 'df_dirt_run4d_FHC.pkl'))

    print("Run 4d dataframes loaded.")

    # -----------------
    # Run 5
    # -----------------
    df_beam_on_run5 = pd.read_pickle(os.path.join(dfs_dir, 'df_beam_on_run5_FHC.pkl'))
    df_ext_run5     = pd.read_pickle(os.path.join(dfs_dir, 'df_ext_run5_FHC.pkl'))
    df_mc_run5      = pd.read_pickle(os.path.join(dfs_dir, 'df_mc_run5_FHC.pkl'))
    df_dirt_run5    = pd.read_pickle(os.path.join(dfs_dir, 'df_dirt_run5_FHC.pkl'))

    print("Run 5 dataframes loaded.")
    print("\nAll individual per-run dataframes loaded from pickle.")

KeyboardInterrupt: 

## Applying 1cm shift in reco neutrino vertex in x direction for all runs

In [ ]:
#===============================================================#
# Apply reco_nuvtxX shift to EXT/MC/DIRT before any cuts
#===============================================================#

def apply_vertex_x_shift_once(dfs, shift=-1.0):
    for df in dfs:
        if df is None:
            continue
        if df.attrs.get("reco_nuvtxX_shift_applied", False):
            continue

        df.loc[:, "reco_nuvtxX"] = df["reco_nuvtxX"] + shift
        df.attrs["reco_nuvtxX_shift_applied"] = True


apply_vertex_x_shift_once([df_mc_run1_FHC, df_ext_run1_FHC, df_dirt_run1_FHC])
apply_vertex_x_shift_once([df_mc_run1_RHC, df_ext_run1_RHC, df_dirt_run1_RHC])

apply_vertex_x_shift_once([df_mc_run2_FHC, df_ext_run2_FHC, df_dirt_run2_FHC])
apply_vertex_x_shift_once([df_mc_run2_RHC, df_ext_run2_RHC, df_dirt_run2_RHC])

apply_vertex_x_shift_once([df_mc_run3, df_ext_run3, df_dirt_run3])

apply_vertex_x_shift_once([df_mc_run4a, df_ext_run4a, df_dirt_run4a])
apply_vertex_x_shift_once([df_mc_run4b, df_ext_run4b, df_dirt_run4b])

apply_vertex_x_shift_once([df_mc_run4c_FHC, df_ext_run4c_FHC, df_dirt_run4c_FHC])
apply_vertex_x_shift_once([df_mc_run4c_RHC, df_ext_run4c_RHC, df_dirt_run4c_RHC])

apply_vertex_x_shift_once([df_mc_run4d, df_ext_run4d, df_dirt_run4d])
apply_vertex_x_shift_once([df_mc_run5, df_ext_run5, df_dirt_run5])

In [ ]:
# #————————————————————————————————————————————————————————#
# # Apply 1cm shift in reco neutrino vertex in x direction #
# #————————————————————————————————————————————————————————#
# def apply_vertex_x_shift(dfs, shift_cm=1.0):
#     """
#     Apply a shift in the reco neutrino vertex X position for a list of dataframes.

#     Parameters
#     ----------
#     dfs : list of pd.DataFrame
#         DataFrames to be shifted in-place.
#     shift_cm : float
#         Amount to subtract from reco_nuvtxX (default: 1.0 cm).
#     """
#     for df in dfs:
#         df["reco_nuvtxX"] = df["reco_nuvtxX"] - shift_cm

# if lock1==True:
#     # For run 1:
#     apply_vertex_x_shift([df_mc_run1_RHC, df_ext_run1_RHC, df_dirt_run1_RHC])
#     apply_vertex_x_shift([df_mc_run1_FHC, df_ext_run1_FHC, df_dirt_run1_FHC])
#     apply_vertex_x_shift([df_mc_run2_FHC, df_ext_run2_FHC, df_dirt_run2_FHC])
#     apply_vertex_x_shift([df_mc_run2_RHC, df_ext_run2_RHC, df_dirt_run2_RHC])
#     # For run 3:
#     apply_vertex_x_shift([df_mc_run3, df_ext_run3, df_dirt_run3])

#     apply_vertex_x_shift([df_mc_run4a, df_ext_run4a, df_dirt_run4a])
#     apply_vertex_x_shift([df_mc_run4b, df_ext_run4b, df_dirt_run4b])
#     apply_vertex_x_shift([df_mc_run4c_RHC, df_ext_run4c_RHC, df_dirt_run4c_RHC])
#     apply_vertex_x_shift([df_mc_run4c_FHC, df_ext_run4c_FHC, df_dirt_run4c_FHC])
#     apply_vertex_x_shift([df_mc_run4d, df_ext_run4d, df_dirt_run4d])
#     apply_vertex_x_shift([df_mc_run5, df_ext_run5, df_dirt_run5])

## Applying cuts and plotting/performing analysis for all runs

In [ ]:
df_data_run1_rhc, df_ext_run1_sel_rhc, df_mc_run1_sel_rhc, df_dirt_run1_sel_rhc = applyCuts('None', df_beam_on_run1_RHC, df_ext_run1_RHC, df_mc_run1_RHC, df_dirt_run1_RHC, 'weight_ubtune', True, POT_data=run1_RHC_pot)
df_data1_run1_rhc, df_ext1_run1_rhc, df_mc1_run1_rhc, df_dirt1_run1_rhc = applyCuts('genNuSelection', df_data_run1_rhc, df_ext_run1_sel_rhc, df_mc_run1_sel_rhc, df_dirt_run1_sel_rhc, 'weight_ubtune', True, POT_data=run1_RHC_pot)
df_data2_run1_rhc, df_ext2_run1_rhc, df_mc2_run1_rhc, df_dirt2_run1_rhc = applyCuts('fiducialVol', df_data1_run1_rhc, df_ext1_run1_rhc, df_mc1_run1_rhc, df_dirt1_run1_rhc, 'weight_ubtune', True, POT_data=run1_RHC_pot)
df_data4_run1_rhc, df_ext4_run1_rhc, df_mc4_run1_rhc, df_dirt4_run1_rhc = applyCuts('muonCut', df_data2_run1_rhc, df_ext2_run1_rhc, df_mc2_run1_rhc, df_dirt2_run1_rhc, 'weight_ubtune', True, POT_data=run1_RHC_pot)
df_dataFC_run1_rhc, df_extFC_run1_rhc, df_mcFC_run1_rhc, df_dirtFC_run1_rhc = applyCuts('FC', df_data4_run1_rhc, df_ext4_run1_rhc, df_mc4_run1_rhc, df_dirt4_run1_rhc, 'weight_ubtune', True, POT_data=run1_RHC_pot)
df_dataPC_run1_rhc, df_extPC_run1_rhc, df_mcPC_run1_rhc, df_dirtPC_run1_rhc = applyCuts('PC', df_data4_run1_rhc, df_ext4_run1_rhc, df_mc4_run1_rhc, df_dirt4_run1_rhc, 'weight_ubtune', True, POT_data=run1_RHC_pot)

In [ ]:
df_data_run1_fhc, df_ext_run1_sel_fhc, df_mc_run1_sel_fhc, df_dirt_run1_sel_fhc = applyCuts('None', df_beam_on_run1_FHC, df_ext_run1_FHC, df_mc_run1_FHC, df_dirt_run1_FHC, 'weight_ubtune', True, POT_data=run1_FHC_pot)
df_data1_run1_fhc, df_ext1_run1_fhc, df_mc1_run1_fhc, df_dirt1_run1_fhc = applyCuts('genNuSelection', df_data_run1_fhc, df_ext_run1_sel_fhc, df_mc_run1_sel_fhc, df_dirt_run1_sel_fhc, 'weight_ubtune', True, POT_data=run1_FHC_pot)
df_data2_run1_fhc, df_ext2_run1_fhc, df_mc2_run1_fhc, df_dirt2_run1_fhc = applyCuts('fiducialVol', df_data1_run1_fhc, df_ext1_run1_fhc, df_mc1_run1_fhc, df_dirt1_run1_fhc, 'weight_ubtune', True, POT_data=run1_FHC_pot)
df_data4_run1_fhc, df_ext4_run1_fhc, df_mc4_run1_fhc, df_dirt4_run1_fhc = applyCuts('muonCut', df_data2_run1_fhc, df_ext2_run1_fhc, df_mc2_run1_fhc, df_dirt2_run1_fhc, 'weight_ubtune', True, POT_data=run1_FHC_pot)
df_dataFC_run1_fhc, df_extFC_run1_fhc, df_mcFC_run1_fhc, df_dirtFC_run1_fhc = applyCuts('FC', df_data4_run1_fhc, df_ext4_run1_fhc, df_mc4_run1_fhc, df_dirt4_run1_fhc, 'weight_ubtune', True, POT_data=run1_FHC_pot)
df_dataPC_run1_fhc, df_extPC_run1_fhc, df_mcPC_run1_fhc, df_dirtPC_run1_fhc = applyCuts('PC', df_data4_run1_fhc, df_ext4_run1_fhc, df_mc4_run1_fhc, df_dirt4_run1_fhc, 'weight_ubtune', True, POT_data=run1_FHC_pot)

In [ ]:
df_data_run2_rhc, df_ext_run2_sel_rhc, df_mc_run2_sel_rhc, df_dirt_run2_sel_rhc = applyCuts('None', df_beam_on_run2_RHC, df_ext_run2_RHC, df_mc_run2_RHC, df_dirt_run2_RHC, 'weight_ubtune', True, POT_data=run2_RHC_pot)
df_data1_run2_rhc, df_ext1_run2_rhc, df_mc1_run2_rhc, df_dirt1_run2_rhc = applyCuts('genNuSelection', df_data_run2_rhc, df_ext_run2_sel_rhc, df_mc_run2_sel_rhc, df_dirt_run2_sel_rhc, 'weight_ubtune', True, POT_data=run2_RHC_pot)
df_data2_run2_rhc, df_ext2_run2_rhc, df_mc2_run2_rhc, df_dirt2_run2_rhc = applyCuts('fiducialVol', df_data1_run2_rhc, df_ext1_run2_rhc, df_mc1_run2_rhc, df_dirt1_run2_rhc, 'weight_ubtune', True, POT_data=run2_RHC_pot)
df_data4_run2_rhc, df_ext4_run2_rhc, df_mc4_run2_rhc, df_dirt4_run2_rhc = applyCuts('muonCut', df_data2_run2_rhc, df_ext2_run2_rhc, df_mc2_run2_rhc, df_dirt2_run2_rhc, 'weight_ubtune', True, POT_data=run2_RHC_pot)
df_dataFC_run2_rhc, df_extFC_run2_rhc, df_mcFC_run2_rhc, df_dirtFC_run2_rhc = applyCuts('FC', df_data4_run2_rhc, df_ext4_run2_rhc, df_mc4_run2_rhc, df_dirt4_run2_rhc, 'weight_ubtune', True, POT_data=run2_RHC_pot)
df_dataPC_run2_rhc, df_extPC_run2_rhc, df_mcPC_run2_rhc, df_dirtPC_run2_rhc = applyCuts('PC', df_data4_run2_rhc, df_ext4_run2_rhc, df_mc4_run2_rhc, df_dirt4_run2_rhc, 'weight_ubtune', True, POT_data=run2_RHC_pot)

In [ ]:
df_data_run2_fhc, df_ext_run2_sel_fhc, df_mc_run2_sel_fhc, df_dirt_run2_sel_fhc = applyCuts('None', df_beam_on_run2_FHC, df_ext_run2_FHC, df_mc_run2_FHC, df_dirt_run2_FHC, 'weight_ubtune', True, POT_data=run2_FHC_pot)
df_data1_run2_fhc, df_ext1_run2_fhc, df_mc1_run2_fhc, df_dirt1_run2_fhc = applyCuts('genNuSelection', df_data_run2_fhc, df_ext_run2_sel_fhc, df_mc_run2_sel_fhc, df_dirt_run2_sel_fhc, 'weight_ubtune', True, POT_data=run2_FHC_pot)
df_data2_run2_fhc, df_ext2_run2_fhc, df_mc2_run2_fhc, df_dirt2_run2_fhc = applyCuts('fiducialVol', df_data1_run2_fhc, df_ext1_run2_fhc, df_mc1_run2_fhc, df_dirt1_run2_fhc, 'weight_ubtune', True, POT_data=run2_FHC_pot)
df_data4_run2_fhc, df_ext4_run2_fhc, df_mc4_run2_fhc, df_dirt4_run2_fhc = applyCuts('muonCut', df_data2_run2_fhc, df_ext2_run2_fhc, df_mc2_run2_fhc, df_dirt2_run2_fhc, 'weight_ubtune', True, POT_data=run2_FHC_pot)
df_dataFC_run2_fhc, df_extFC_run2_fhc, df_mcFC_run2_fhc, df_dirtFC_run2_fhc = applyCuts('FC', df_data4_run2_fhc, df_ext4_run2_fhc, df_mc4_run2_fhc, df_dirt4_run2_fhc, 'weight_ubtune', True, POT_data=run2_FHC_pot)
df_dataPC_run2_fhc, df_extPC_run2_fhc, df_mcPC_run2_fhc, df_dirtPC_run2_fhc = applyCuts('PC', df_data4_run2_fhc, df_ext4_run2_fhc, df_mc4_run2_fhc, df_dirt4_run2_fhc, 'weight_ubtune', True, POT_data=run2_FHC_pot)

In [ ]:
#%%capture
#—————————————————————————————————————————————————————#
# Now apply selection and draw histo and show entries for RUN 1 #
#—————————————————————————————————————————————————————#

df_data_run3, df_ext_run3_sel, df_mc_run3_sel, df_dirt_run3_sel = applyCuts('None', df_beam_on_run3, df_ext_run3, df_mc_run3, df_dirt_run3, 'weight_ubtune', True, POT_data=run3b_pot_beam_on)
df_data1_run3, df_ext1_run3, df_mc1_run3, df_dirt1_run3 = applyCuts('genNuSelection', df_data_run3, df_ext_run3_sel, df_mc_run3_sel, df_dirt_run3_sel, 'weight_ubtune', True, POT_data=run3b_pot_beam_on)
df_data2_run3, df_ext2_run3, df_mc2_run3, df_dirt2_run3 = applyCuts('fiducialVol', df_data1_run3, df_ext1_run3, df_mc1_run3, df_dirt1_run3, 'weight_ubtune', True, POT_data=run3b_pot_beam_on)
df_data4_run3, df_ext4_run3, df_mc4_run3, df_dirt4_run3 = applyCuts('muonCut', df_data2_run3, df_ext2_run3, df_mc2_run3, df_dirt2_run3, 'weight_ubtune', True, POT_data=run3b_pot_beam_on)
df_dataFC_run3, df_extFC_run3, df_mcFC_run3, df_dirtFC_run3 = applyCuts('FC', df_data4_run3, df_ext4_run3, df_mc4_run3, df_dirt4_run3, 'weight_ubtune', True, POT_data=run3b_pot_beam_on)
df_dataPC_run3, df_extPC_run3, df_mcPC_run3, df_dirtPC_run3 = applyCuts('PC', df_data4_run3, df_ext4_run3, df_mc4_run3, df_dirt4_run3, 'weight_ubtune', True, POT_data=run3b_pot_beam_on)

In [ ]:
df_data_run4a, df_ext_run4a_sel, df_mc_run4a_sel, df_dirt_run4a_sel = applyCuts('None', df_beam_on_run4a, df_ext_run4a, df_mc_run4a, df_dirt_run4a, 'weight_ubtune', True, POT_data=run4a_RHC_pot)
df_data1_run4a, df_ext1_run4a, df_mc1_run4a, df_dirt1_run4a = applyCuts('genNuSelection', df_data_run4a, df_ext_run4a_sel, df_mc_run4a_sel, df_dirt_run4a_sel, 'weight_ubtune', True, POT_data=run4a_RHC_pot)
df_data2_run4a, df_ext2_run4a, df_mc2_run4a, df_dirt2_run4a = applyCuts('fiducialVol', df_data1_run4a, df_ext1_run4a, df_mc1_run4a, df_dirt1_run4a, 'weight_ubtune', True, POT_data=run4a_RHC_pot)
df_data4_run4a, df_ext4_run4a, df_mc4_run4a, df_dirt4_run4a = applyCuts('muonCut', df_data2_run4a, df_ext2_run4a, df_mc2_run4a, df_dirt2_run4a, 'weight_ubtune', True, POT_data=run4a_RHC_pot)
df_dataFC_run4a, df_extFC_run4a, df_mcFC_run4a, df_dirtFC_run4a = applyCuts('FC', df_data4_run4a, df_ext4_run4a, df_mc4_run4a, df_dirt4_run4a, 'weight_ubtune', True, POT_data=run4a_RHC_pot)
df_dataPC_run4a, df_extPC_run4a, df_mcPC_run4a, df_dirtPC_run4a = applyCuts('PC', df_data4_run4a, df_ext4_run4a, df_mc4_run4a, df_dirt4_run4a, 'weight_ubtune', True, POT_data=run4a_RHC_pot)

In [ ]:
df_data_run4b, df_ext_run4b_sel, df_mc_run4b_sel, df_dirt_run4b_sel = applyCuts('None', df_beam_on_run4b, df_ext_run4b, df_mc_run4b, df_dirt_run4b, 'weight_ubtune', True, POT_data=run4b_RHC_pot)
df_data1_run4b, df_ext1_run4b, df_mc1_run4b, df_dirt1_run4b = applyCuts('genNuSelection', df_data_run4b, df_ext_run4b_sel, df_mc_run4b_sel, df_dirt_run4b_sel, 'weight_ubtune', True, POT_data=run4b_RHC_pot)
df_data2_run4b, df_ext2_run4b, df_mc2_run4b, df_dirt2_run4b = applyCuts('fiducialVol', df_data1_run4b, df_ext1_run4b, df_mc1_run4b, df_dirt1_run4b, 'weight_ubtune', True, POT_data=run4b_RHC_pot)
df_data4_run4b, df_ext4_run4b, df_mc4_run4b, df_dirt4_run4b = applyCuts('muonCut', df_data2_run4b, df_ext2_run4b, df_mc2_run4b, df_dirt2_run4b, 'weight_ubtune', True, POT_data=run4b_RHC_pot)
df_dataFC_run4b, df_extFC_run4b, df_mcFC_run4b, df_dirtFC_run4b = applyCuts('FC', df_data4_run4b, df_ext4_run4b, df_mc4_run4b, df_dirt4_run4b, 'weight_ubtune', True, POT_data=run4b_RHC_pot)
df_dataPC_run4b, df_extPC_run4b, df_mcPC_run4b, df_dirtPC_run4b = applyCuts('PC', df_data4_run4b, df_ext4_run4b, df_mc4_run4b, df_dirt4_run4b, 'weight_ubtune', True, POT_data=run4b_RHC_pot)

In [ ]:
df_data_run4c_rhc, df_ext_run4c_sel_rhc, df_mc_run4c_sel_rhc, df_dirt_run4c_sel_rhc = applyCuts('None', df_beam_on_run4c_RHC, df_ext_run4c_RHC, df_mc_run4c_RHC, df_dirt_run4c_RHC, 'weight_ubtune', True, POT_data=run4c_RHC_pot)
df_data1_run4c_rhc, df_ext1_run4c_rhc, df_mc1_run4c_rhc, df_dirt1_run4c_rhc = applyCuts('genNuSelection', df_data_run4c_rhc, df_ext_run4c_sel_rhc, df_mc_run4c_sel_rhc, df_dirt_run4c_sel_rhc, 'weight_ubtune', True, POT_data=run4c_RHC_pot)
df_data2_run4c_rhc, df_ext2_run4c_rhc, df_mc2_run4c_rhc, df_dirt2_run4c_rhc = applyCuts('fiducialVol', df_data1_run4c_rhc, df_ext1_run4c_rhc, df_mc1_run4c_rhc, df_dirt1_run4c_rhc, 'weight_ubtune', True, POT_data=run4c_RHC_pot)
df_data4_run4c_rhc, df_ext4_run4c_rhc, df_mc4_run4c_rhc, df_dirt4_run4c_rhc = applyCuts('muonCut', df_data2_run4c_rhc, df_ext2_run4c_rhc, df_mc2_run4c_rhc, df_dirt2_run4c_rhc, 'weight_ubtune', True, POT_data=run4c_RHC_pot)
df_dataFC_run4c_rhc, df_extFC_run4c_rhc, df_mcFC_run4c_rhc, df_dirtFC_run4c_rhc = applyCuts('FC', df_data4_run4c_rhc, df_ext4_run4c_rhc, df_mc4_run4c_rhc, df_dirt4_run4c_rhc, 'weight_ubtune', True, POT_data=run4c_RHC_pot)
df_dataPC_run4c_rhc, df_extPC_run4c_rhc, df_mcPC_run4c_rhc, df_dirtPC_run4c_rhc = applyCuts('PC', df_data4_run4c_rhc, df_ext4_run4c_rhc, df_mc4_run4c_rhc, df_dirt4_run4c_rhc, 'weight_ubtune', True, POT_data=run4c_RHC_pot)   

In [ ]:
df_data_run4c_fhc, df_ext_run4c_sel_fhc, df_mc_run4c_sel_fhc, df_dirt_run4c_sel_fhc = applyCuts('None', df_beam_on_run4c_FHC, df_ext_run4c_FHC, df_mc_run4c_FHC, df_dirt_run4c_FHC, 'weight_ubtune', True, POT_data=run4c_FHC_pot)
df_data1_run4c_fhc, df_ext1_run4c_fhc, df_mc1_run4c_fhc, df_dirt1_run4c_fhc = applyCuts('genNuSelection', df_data_run4c_fhc, df_ext_run4c_sel_fhc, df_mc_run4c_sel_fhc, df_dirt_run4c_sel_fhc, 'weight_ubtune', True, POT_data=run4c_FHC_pot)
df_data2_run4c_fhc, df_ext2_run4c_fhc, df_mc2_run4c_fhc, df_dirt2_run4c_fhc = applyCuts('fiducialVol', df_data1_run4c_fhc, df_ext1_run4c_fhc, df_mc1_run4c_fhc, df_dirt1_run4c_fhc, 'weight_ubtune', True, POT_data=run4c_FHC_pot)
df_data4_run4c_fhc, df_ext4_run4c_fhc, df_mc4_run4c_fhc, df_dirt4_run4c_fhc = applyCuts('muonCut', df_data2_run4c_fhc, df_ext2_run4c_fhc, df_mc2_run4c_fhc, df_dirt2_run4c_fhc, 'weight_ubtune', True, POT_data=run4c_FHC_pot)
df_dataFC_run4c_fhc, df_extFC_run4c_fhc, df_mcFC_run4c_fhc, df_dirtFC_run4c_fhc = applyCuts('FC', df_data4_run4c_fhc, df_ext4_run4c_fhc, df_mc4_run4c_fhc, df_dirt4_run4c_fhc, 'weight_ubtune', True, POT_data=run4c_FHC_pot)
df_dataPC_run4c_fhc, df_extPC_run4c_fhc, df_mcPC_run4c_fhc, df_dirtPC_run4c_fhc = applyCuts('PC', df_data4_run4c_fhc, df_ext4_run4c_fhc, df_mc4_run4c_fhc, df_dirt4_run4c_fhc, 'weight_ubtune', True, POT_data=run4c_FHC_pot)

In [ ]:
df_data_run4d, df_ext_run4d_sel, df_mc_run4d_sel, df_dirt_run4d_sel = applyCuts('None', df_beam_on_run4d, df_ext_run4d, df_mc_run4d, df_dirt_run4d, 'weight_ubtune', True, POT_data=run4d_FHC_pot)
df_data1_run4d, df_ext1_run4d, df_mc1_run4d, df_dirt1_run4d = applyCuts('genNuSelection', df_data_run4d, df_ext_run4d_sel, df_mc_run4d_sel, df_dirt_run4d_sel, 'weight_ubtune', True, POT_data=run4d_FHC_pot)
df_data2_run4d, df_ext2_run4d, df_mc2_run4d, df_dirt2_run4d = applyCuts('fiducialVol', df_data1_run4d, df_ext1_run4d, df_mc1_run4d, df_dirt1_run4d, 'weight_ubtune', True, POT_data=run4d_FHC_pot)
df_data4_run4d, df_ext4_run4d, df_mc4_run4d, df_dirt4_run4d = applyCuts('muonCut', df_data2_run4d, df_ext2_run4d, df_mc2_run4d, df_dirt2_run4d, 'weight_ubtune', True, POT_data=run4d_FHC_pot)
df_dataFC_run4d, df_extFC_run4d, df_mcFC_run4d, df_dirtFC_run4d = applyCuts('FC', df_data4_run4d, df_ext4_run4d, df_mc4_run4d, df_dirt4_run4d, 'weight_ubtune', True, POT_data=run4d_FHC_pot)
df_dataPC_run4d, df_extPC_run4d, df_mcPC_run4d, df_dirtPC_run4d = applyCuts('PC', df_data4_run4d, df_ext4_run4d, df_mc4_run4d, df_dirt4_run4d, 'weight_ubtune', True, POT_data=run4d_FHC_pot)

In [ ]:
df_data_run5, df_ext_run5_sel, df_mc_run5_sel, df_dirt_run5_sel = applyCuts('None', df_beam_on_run5, df_ext_run5, df_mc_run5, df_dirt_run5, 'weight_ubtune', True, POT_data=run5_FHC_pot)
df_data1_run5, df_ext1_run5, df_mc1_run5, df_dirt1_run5 = applyCuts('genNuSelection', df_data_run5, df_ext_run5_sel, df_mc_run5_sel, df_dirt_run5_sel, 'weight_ubtune', True, POT_data=run5_FHC_pot)
df_data2_run5, df_ext2_run5, df_mc2_run5, df_dirt2_run5 = applyCuts('fiducialVol', df_data1_run5, df_ext1_run5, df_mc1_run5, df_dirt1_run5, 'weight_ubtune', True, POT_data=run5_FHC_pot)
df_data4_run5, df_ext4_run5, df_mc4_run5, df_dirt4_run5 = applyCuts('muonCut', df_data2_run5, df_ext2_run5, df_mc2_run5, df_dirt2_run5, 'weight_ubtune', True, POT_data=run5_FHC_pot)
df_dataFC_Run5, df_extFC_Run5, df_mcFC_Run5, df_dirtFC_Run5 = applyCuts('FC', df_data4_run5, df_ext4_run5, df_mc4_run5, df_dirt4_run5, 'weight_ubtune', True, POT_data=run5_FHC_pot)
df_dataPC_Run5, df_extPC_Run5, df_mcPC_Run5, df_dirtPC_Run5 = applyCuts('PC', df_data4_run5, df_ext4_run5, df_mc4_run5, df_dirt4_run5, 'weight_ubtune', True, POT_data=run5_FHC_pot)

## COMBINE TO HAVE 1 FHC and 1 RHC dataframes

In [ ]:
# ============================================================
# Merge loaded individual run dataframes into FHC-only and RHC-only
# ============================================================

# -----------------
# RHC merged
# -----------------
df_beam_on_RHC = pd.concat([
    df_beam_on_run1_RHC,
    df_beam_on_run2_RHC,
    df_beam_on_run3,
    df_beam_on_run4a,
    df_beam_on_run4b,
    df_beam_on_run4c_RHC
], ignore_index=True)

df_ext_RHC = pd.concat([
    df_ext_run1_RHC,
    df_ext_run2_RHC,
    df_ext_run3,
    df_ext_run4a,
    df_ext_run4b,
    df_ext_run4c_RHC
], ignore_index=True)

df_mc_RHC = pd.concat([
    df_mc_run1_RHC,
    df_mc_run2_RHC,
    df_mc_run3,
    df_mc_run4a,
    df_mc_run4b,
    df_mc_run4c_RHC
], ignore_index=True)

df_dirt_RHC = pd.concat([
    df_dirt_run1_RHC,
    df_dirt_run2_RHC,
    df_dirt_run3,
    df_dirt_run4a,
    df_dirt_run4b,
    df_dirt_run4c_RHC
], ignore_index=True)

# -----------------
# FHC merged
# -----------------
df_beam_on_FHC = pd.concat([
    df_beam_on_run1_FHC,
    df_beam_on_run2_FHC,
    df_beam_on_run4c_FHC,
    df_beam_on_run4d,
    df_beam_on_run5
], ignore_index=True)

df_ext_FHC = pd.concat([
    df_ext_run1_FHC,
    df_ext_run2_FHC,
    df_ext_run4c_FHC,
    df_ext_run4d,
    df_ext_run5
], ignore_index=True)

df_mc_FHC = pd.concat([
    df_mc_run1_FHC,
    df_mc_run2_FHC,
    df_mc_run4c_FHC,
    df_mc_run4d,
    df_mc_run5
], ignore_index=True)

df_dirt_FHC = pd.concat([
    df_dirt_run1_FHC,
    df_dirt_run2_FHC,
    df_dirt_run4c_FHC,
    df_dirt_run4d,
    df_dirt_run5
], ignore_index=True)

print("Merged RHC/FHC dataframes created.")
print(f"df_beam_on_RHC : {len(df_beam_on_RHC)} rows")
print(f"df_ext_RHC     : {len(df_ext_RHC)} rows")
print(f"df_mc_RHC      : {len(df_mc_RHC)} rows")
print(f"df_dirt_RHC    : {len(df_dirt_RHC)} rows")
print()
print(f"df_beam_on_FHC : {len(df_beam_on_FHC)} rows")
print(f"df_ext_FHC     : {len(df_ext_FHC)} rows")
print(f"df_mc_FHC      : {len(df_mc_FHC)} rows")
print(f"df_dirt_FHC    : {len(df_dirt_FHC)} rows")

In [ ]:
print("=== RHC weighted sums ===")
print("Beam-on :", df_beam_on_RHC['weight_ubtune'].sum())
print("EXT     :", df_ext_RHC['weight_ubtune'].sum())
print("MC      :", df_mc_RHC['weight_ubtune'].sum())
print("DIRT    :", df_dirt_RHC['weight_ubtune'].sum())

print("\n=== FHC weighted sums ===")
print("Beam-on :", df_beam_on_FHC['weight_ubtune'].sum())
print("EXT     :", df_ext_FHC['weight_ubtune'].sum())
print("MC      :", df_mc_FHC['weight_ubtune'].sum())
print("DIRT    :", df_dirt_FHC['weight_ubtune'].sum())

In [ ]:
print("\nRHC total prediction :", df_ext_RHC['weight_ubtune'].sum() + df_mc_RHC['weight_ubtune'].sum() + df_dirt_RHC['weight_ubtune'].sum())
print("FHC total prediction :", df_ext_FHC['weight_ubtune'].sum() + df_mc_FHC['weight_ubtune'].sum() + df_dirt_FHC['weight_ubtune'].sum())

## COMBINED FHC FC sample and RHC FC sample

In [ ]:
# ============================================================
# Apply cuts to merged RHC dataframe
# ============================================================

df_data_RHC, df_ext_RHC_sel, df_mc_RHC_sel, df_dirt_RHC_sel = applyCuts(
    'None',
    df_beam_on_RHC, df_ext_RHC, df_mc_RHC, df_dirt_RHC,
    'weight_ubtune', True, POT_data=RHC_full_POT
)

df_data1_RHC, df_ext1_RHC, df_mc1_RHC, df_dirt1_RHC = applyCuts(
    'genNuSelection',
    df_data_RHC, df_ext_RHC_sel, df_mc_RHC_sel, df_dirt_RHC_sel,
    'weight_ubtune', True, POT_data=RHC_full_POT
)

df_data2_RHC, df_ext2_RHC, df_mc2_RHC, df_dirt2_RHC = applyCuts(
    'fiducialVol',
    df_data1_RHC, df_ext1_RHC, df_mc1_RHC, df_dirt1_RHC,
    'weight_ubtune', True, POT_data=RHC_full_POT
)

df_data4_RHC, df_ext4_RHC, df_mc4_RHC, df_dirt4_RHC = applyCuts(
    'muonCut',
    df_data2_RHC, df_ext2_RHC, df_mc2_RHC, df_dirt2_RHC,
    'weight_ubtune', True, POT_data=RHC_full_POT
)

df_dataFC_RHC, df_extFC_RHC, df_mcFC_RHC, df_dirtFC_RHC = applyCuts(
    'FC',
    df_data4_RHC, df_ext4_RHC, df_mc4_RHC, df_dirt4_RHC,
    'weight_ubtune', True, POT_data=RHC_full_POT
)

df_dataPC_RHC, df_extPC_RHC, df_mcPC_RHC, df_dirtPC_RHC = applyCuts(
    'PC',
    df_data4_RHC, df_ext4_RHC, df_mc4_RHC, df_dirt4_RHC,
    'weight_ubtune', True, POT_data=RHC_full_POT
)

In [ ]:
# ============================================================
# Apply cuts to merged FHC dataframe
# ============================================================

df_data_FHC, df_ext_FHC_sel, df_mc_FHC_sel, df_dirt_FHC_sel = applyCuts(
    'None',
    df_beam_on_FHC, df_ext_FHC, df_mc_FHC, df_dirt_FHC,
    'weight_ubtune', True, POT_data=FHC_full_POT
)

df_data1_FHC, df_ext1_FHC, df_mc1_FHC, df_dirt1_FHC = applyCuts(
    'genNuSelection',
    df_data_FHC, df_ext_FHC_sel, df_mc_FHC_sel, df_dirt_FHC_sel,
    'weight_ubtune', True, POT_data=FHC_full_POT
)

df_data2_FHC, df_ext2_FHC, df_mc2_FHC, df_dirt2_FHC = applyCuts(
    'fiducialVol',
    df_data1_FHC, df_ext1_FHC, df_mc1_FHC, df_dirt1_FHC,
    'weight_ubtune', True, POT_data=FHC_full_POT
)

df_data4_FHC, df_ext4_FHC, df_mc4_FHC, df_dirt4_FHC = applyCuts(
    'muonCut',
    df_data2_FHC, df_ext2_FHC, df_mc2_FHC, df_dirt2_FHC,
    'weight_ubtune', True, POT_data=FHC_full_POT
)

df_dataFC_FHC, df_extFC_FHC, df_mcFC_FHC, df_dirtFC_FHC = applyCuts(
    'FC',
    df_data4_FHC, df_ext4_FHC, df_mc4_FHC, df_dirt4_FHC,
    'weight_ubtune', True, POT_data=FHC_full_POT
)

df_dataPC_FHC, df_extPC_FHC, df_mcPC_FHC, df_dirtPC_FHC = applyCuts(
    'PC',
    df_data4_FHC, df_ext4_FHC, df_mc4_FHC, df_dirt4_FHC,
    'weight_ubtune', True, POT_data=FHC_full_POT
)

## Calculate selection efficiency and purity after each selection stage

In [ ]:
mask_numu_run1_rhc    = (df_mc_run1_RHC.truth_nuPdg == 14)  & (df_mc_run1_RHC.truth_isCC == 1) & (df_mc_run1_RHC.truth_vtxInside == 1)
mask_numubar_run1_rhc = (df_mc_run1_RHC.truth_nuPdg == -14) & (df_mc_run1_RHC.truth_isCC == 1) & (df_mc_run1_RHC.truth_vtxInside == 1)

mask_numu_run1_fhc    = (df_mc_run1_FHC.truth_nuPdg == 14)  & (df_mc_run1_FHC.truth_isCC == 1) & (df_mc_run1_FHC.truth_vtxInside == 1)
mask_numubar_run1_fhc = (df_mc_run1_FHC.truth_nuPdg == -14) & (df_mc_run1_FHC.truth_isCC == 1) & (df_mc_run1_FHC.truth_vtxInside == 1)  

mask_numu_run2_rhc    = (df_mc_run2_RHC.truth_nuPdg == 14)  & (df_mc_run2_RHC.truth_isCC == 1) & (df_mc_run2_RHC.truth_vtxInside == 1)
mask_numubar_run2_rhc = (df_mc_run2_RHC.truth_nuPdg == -14) & (df_mc_run2_RHC.truth_isCC == 1) & (df_mc_run2_RHC.truth_vtxInside == 1)

mask_numu_run2_fhc    = (df_mc_run2_FHC.truth_nuPdg == 14)  & (df_mc_run2_FHC.truth_isCC == 1) & (df_mc_run2_FHC.truth_vtxInside == 1)
mask_numubar_run2_fhc = (df_mc_run2_FHC.truth_nuPdg == -14) & (df_mc_run2_FHC.truth_isCC == 1) & (df_mc_run2_FHC.truth_vtxInside == 1)

mask_numu_run3    = (df_mc_run3.truth_nuPdg == 14)  & (df_mc_run3.truth_isCC == 1) & (df_mc_run3.truth_vtxInside == 1)
mask_numubar_run3 = (df_mc_run3.truth_nuPdg == -14) & (df_mc_run3.truth_isCC == 1) & (df_mc_run3.truth_vtxInside == 1)

mask_numu_run4a    = (df_mc_run4a.truth_nuPdg == 14)  & (df_mc_run4a.truth_isCC == 1) & (df_mc_run4a.truth_vtxInside == 1)
mask_numubar_run4a = (df_mc_run4a.truth_nuPdg == -14) & (df_mc_run4a.truth_isCC == 1) & (df_mc_run4a.truth_vtxInside == 1)

mask_numu_run4b    = (df_mc_run4b.truth_nuPdg == 14)  & (df_mc_run4b.truth_isCC == 1) & (df_mc_run4b.truth_vtxInside == 1)
mask_numubar_run4b = (df_mc_run4b.truth_nuPdg == -14) & (df_mc_run4b.truth_isCC == 1) & (df_mc_run4b.truth_vtxInside == 1)

mask_numu_run4c_rhc    = (df_mc_run4c_RHC.truth_nuPdg == 14)  & (df_mc_run4c_RHC.truth_isCC == 1) & (df_mc_run4c_RHC.truth_vtxInside == 1)
mask_numubar_run4c_rhc = (df_mc_run4c_RHC.truth_nuPdg == -14) & (df_mc_run4c_RHC.truth_isCC == 1) & (df_mc_run4c_RHC.truth_vtxInside == 1)

mask_numu_run4c_fhc  = (df_mc_run4c_FHC.truth_nuPdg == 14)  & (df_mc_run4c_FHC.truth_isCC == 1) & (df_mc_run4c_FHC.truth_vtxInside == 1)
mask_numubar_run4c_fhc = (df_mc_run4c_FHC.truth_nuPdg == -14) & (df_mc_run4c_FHC.truth_isCC == 1) & (df_mc_run4c_FHC.truth_vtxInside == 1)

mask_numu_run4d  = (df_mc_run4d.truth_nuPdg == 14)  & (df_mc_run4d.truth_isCC == 1) & (df_mc_run4d.truth_vtxInside == 1)
mask_numubar_run4d = (df_mc_run4d.truth_nuPdg == -14) & (df_mc_run4d.truth_isCC == 1) & (df_mc_run4d.truth_vtxInside == 1)

mask_numu_run5  = (df_mc_run5.truth_nuPdg == 14)  & (df_mc_run5.truth_isCC == 1) & (df_mc_run5.truth_vtxInside == 1)
mask_numubar_run5 = (df_mc_run5.truth_nuPdg == -14) & (df_mc_run5.truth_isCC == 1) & (df_mc_run5.truth_vtxInside == 1)

In [ ]:
mask_numu_RHC  = (df_mc_RHC.truth_nuPdg == 14)  & (df_mc_RHC.truth_isCC == 1) & (df_mc_RHC.truth_vtxInside == 1)
mask_numubar_RHC = (df_mc_RHC.truth_nuPdg == -14) & (df_mc_RHC.truth_isCC == 1) & (df_mc_RHC.truth_vtxInside == 1)

In [ ]:
mask_numu_FHC  = (df_mc_FHC.truth_nuPdg == 14)  & (df_mc_FHC.truth_isCC == 1) & (df_mc_FHC.truth_vtxInside == 1)
mask_numubar_FHC = (df_mc_FHC.truth_nuPdg == -14) & (df_mc_FHC.truth_isCC == 1) & (df_mc_FHC.truth_vtxInside == 1)

In [ ]:
def compute_bayesian_cutflow_weighted(mc_dfs,
                                      mask_numu,
                                      mask_numubar,
                                      weight_col='weight_ubtune',
                                      alpha=0.683,
                                      digits=3):
    """
    Compute Bayesian efficiencies & purities (with 1σ-style "val(err)") 
    using POT-normalized weights.

    Parameters
    ----------
    mc_dfs : dict[str, pd.DataFrame]
        MC DataFrames after each cut; must include a 'None' key for pre-cut.
    mask_numu, mask_numubar : pd.Series (bool)
        Truth masks on the original (pre-cut) MC index.
    weight_col : str
        Column name containing the POT-normalized event weight.
    alpha : float
        Credible-level (default 0.683 for ~1σ).
    digits : int
        Decimal places in the formatted value.

    Returns
    -------
    pd.DataFrame
        One row per cut, columns:
        ['Cut',
         'Eff νμ CC', 'Pur νμ CC',
         'Eff ν̄μ CC','Pur ν̄μ CC',
         'Selected (w)']
    """
    def fmt(val, lo, hi):
        # half-width of CI in last digit
        err = (hi - lo) / 2
        err_int = int(round(err * 10**digits))
        return f"{val:.{digits}f}({err_int})"

    # Get the pre-cut (“None”) DataFrame
    df0 = mc_dfs['None']
    # Sum of weights before any cuts = S0
    S0_numu    = df0.loc[mask_numu.loc[df0.index],    weight_col].sum()
    S0_numubar = df0.loc[mask_numubar.loc[df0.index], weight_col].sum()

    rows = []
    for cut, df in mc_dfs.items():
        # total weight surviving this cut = Tj
        Tj = df[weight_col].sum()

        # surviving signal weights = Sj
        Sj_numu    = df.loc[mask_numu.loc[df.index],    weight_col].sum()
        Sj_numubar = df.loc[mask_numubar.loc[df.index], weight_col].sum()

        # efficiencies & purities
        eff_numu    = Sj_numu    / S0_numu    if S0_numu>0    else np.nan
        eff_numubar = Sj_numubar / S0_numubar if S0_numubar>0 else np.nan
        pur_numu    = Sj_numu    / Tj         if Tj>0         else np.nan
        pur_numubar = Sj_numubar / Tj         if Tj>0         else np.nan

        # Beta parameters (can be non-integer)
        # Efficiency posterior ~ Beta(Sj+1, S0-Sj+1)
        aE_nu,    bE_nu    = Sj_numu+1,    S0_numu -    Sj_numu +1
        aE_nubar, bE_nubar = Sj_numubar+1, S0_numubar-Sj_numubar+1

        # Purity posterior    ~ Beta(Sj+1, Tj-Sj+1)
        aP_nu,    bP_nu    = Sj_numu+1,    Tj - Sj_numu    +1
        aP_nubar, bP_nubar = Sj_numubar+1, Tj - Sj_numubar +1

        loE_nu,    hiE_nu    = beta.interval(alpha, aE_nu,    bE_nu)
        loE_nubar, hiE_nubar = beta.interval(alpha, aE_nubar, bE_nubar)
        loP_nu,    hiP_nu    = beta.interval(alpha, aP_nu,    bP_nu)
        loP_nubar, hiP_nubar = beta.interval(alpha, aP_nubar, bP_nubar)

        rows.append({
            'Cut':         cut,
            'Eff νμ CC':   fmt(eff_numu,    loE_nu,    hiE_nu),
            'Pur νμ CC':   fmt(pur_numu,    loP_nu,    hiP_nu),
            'Eff ν̄μ CC':  fmt(eff_numubar, loE_nubar, hiE_nubar),
            'Pur ν̄μ CC':  fmt(pur_numubar, loP_nubar, hiP_nubar),
            'Selected (w)': Tj
        })

    return pd.DataFrame(rows)

In [ ]:
mc_dfs_run1_fhc = {
    'None':           df_mc_run1_FHC,
    'genNuSelection': df_mc1_run1_fhc,
    'fiducialVol':    df_mc2_run1_fhc,
    'muonCut':        df_mc4_run1_fhc,
    'FC':             df_mcFC_run1_fhc
}
cutflow_fmt_run1_fhc = compute_bayesian_cutflow_weighted(
    mc_dfs_run1_fhc,
    mask_numu_run1_fhc,
    mask_numubar_run1_fhc,
    weight_col='weight_ubtune'
)
cutflow_fmt_run1_fhc

In [ ]:
mc_dfs_run1_rhc = {
    'None':           df_mc_run1_RHC,
    'genNuSelection': df_mc1_run1_rhc,
    'fiducialVol':    df_mc2_run1_rhc,
    'muonCut':        df_mc4_run1_rhc,
    'FC':             df_mcFC_run1_rhc
}
cutflow_fmt_run1_rhc = compute_bayesian_cutflow_weighted(
    mc_dfs_run1_rhc,
    mask_numu_run1_rhc,
    mask_numubar_run1_rhc,
    weight_col='weight_ubtune'
)
cutflow_fmt_run1_rhc

In [ ]:
mc_dfs_run2_fhc = {
    'None':           df_mc_run2_FHC,
    'genNuSelection': df_mc1_run2_fhc,
    'fiducialVol':    df_mc2_run2_fhc,
    'muonCut':        df_mc4_run2_fhc,
    'FC':             df_mcFC_run2_fhc
}
cutflow_fmt_run2_fhc = compute_bayesian_cutflow_weighted(
    mc_dfs_run2_fhc,
    mask_numu_run2_fhc,
    mask_numubar_run2_fhc,
    weight_col='weight_ubtune'
)
cutflow_fmt_run2_fhc

In [ ]:
mc_dfs_run2_rhc = {
    'None':           df_mc_run2_RHC,
    'genNuSelection': df_mc1_run2_rhc,
    'fiducialVol':    df_mc2_run2_rhc,
    'muonCut':        df_mc4_run2_rhc,
    'FC':             df_mcFC_run2_rhc
}
cutflow_fmt_run2_rhc = compute_bayesian_cutflow_weighted(
    mc_dfs_run2_rhc,
    mask_numu_run2_rhc,
    mask_numubar_run2_rhc,
    weight_col='weight_ubtune'
)
cutflow_fmt_run2_rhc

In [ ]:
mc_dfs_run3 = {
    'None':           df_mc_run3,
    'genNuSelection': df_mc1_run3,
    'fiducialVol':    df_mc2_run3,
    'muonCut':        df_mc4_run3,
    'FC':             df_mcFC_run3
}
cutflow_fmt_run3 = compute_bayesian_cutflow_weighted(
    mc_dfs_run3,
    mask_numu_run3,
    mask_numubar_run3,
    weight_col='weight_ubtune'
)
cutflow_fmt_run3

In [ ]:
mc_dfs_run4a = {
    'None':           df_mc_run4a,
    'genNuSelection': df_mc1_run4a,
    'fiducialVol':    df_mc2_run4a,
    'muonCut':        df_mc4_run4a,
    'FC':             df_mcFC_run4a
}
cutflow_fmt_run4a = compute_bayesian_cutflow_weighted(
    mc_dfs_run4a,
    mask_numu_run4a,
    mask_numubar_run4a,
    weight_col='weight_ubtune'
)
cutflow_fmt_run4a

In [ ]:
mc_dfs_run4b = {
    'None':           df_mc_run4b,
    'genNuSelection': df_mc1_run4b,
    'fiducialVol':    df_mc2_run4b,
    'muonCut':        df_mc4_run4b,
    'FC':             df_mcFC_run4b
}
cutflow_fmt_run4b = compute_bayesian_cutflow_weighted(
    mc_dfs_run4b,
    mask_numu_run4b,
    mask_numubar_run4b,
    weight_col='weight_ubtune'
)
cutflow_fmt_run4b

In [ ]:
mc_dfs_run4c_fhc = {
    'None':           df_mc_run4c_FHC,
    'genNuSelection': df_mc1_run4c_fhc,
    'fiducialVol':    df_mc2_run4c_fhc,
    'muonCut':        df_mc4_run4c_fhc,
    'FC':             df_mcFC_run4c_fhc
}
cutflow_fmt_run4c_fhc = compute_bayesian_cutflow_weighted(
    mc_dfs_run4c_fhc,
    mask_numu_run4c_fhc,
    mask_numubar_run4c_fhc,
    weight_col='weight_ubtune'
)
cutflow_fmt_run4c_fhc

In [ ]:
mc_dfs_run4c_rhc = {
    'None':           df_mc_run4c_RHC,
    'genNuSelection': df_mc1_run4c_rhc,
    'fiducialVol':    df_mc2_run4c_rhc,
    'muonCut':        df_mc4_run4c_rhc,
    'FC':             df_mcFC_run4c_rhc
}
cutflow_fmt_run4c_rhc = compute_bayesian_cutflow_weighted(
    mc_dfs_run4c_rhc,
    mask_numu_run4c_rhc,
    mask_numubar_run4c_rhc,
    weight_col='weight_ubtune'
)
cutflow_fmt_run4c_rhc

In [ ]:
mc_dfs_run4d = {
    'None':           df_mc_run4d,
    'genNuSelection': df_mc1_run4d,
    'fiducialVol':    df_mc2_run4d,
    'muonCut':        df_mc4_run4d,
    'FC':             df_mcFC_run4d
}
cutflow_fmt_run4d = compute_bayesian_cutflow_weighted(
    mc_dfs_run4d,
    mask_numu_run4d,
    mask_numubar_run4d,
    weight_col='weight_ubtune'
)
cutflow_fmt_run4d

In [ ]:
mc_dfs_run5 = {
    'None':           df_mc_run5,
    'genNuSelection': df_mc1_run5,
    'fiducialVol':    df_mc2_run5,
    'muonCut':        df_mc4_run5,
    'FC':             df_mcFC_Run5
}
cutflow_fmt_run5 = compute_bayesian_cutflow_weighted(
    mc_dfs_run5,
    mask_numu_run5,
    mask_numubar_run5,
    weight_col='weight_ubtune'
)
cutflow_fmt_run5

## Plot for run1 eff/pur

In [ ]:
import matplotlib.pyplot as plt

# 1) Prepare
cuts = list(mc_dfs_run1_fhc.keys())
S0_numu    = mc_dfs_run1_fhc['None'].loc[mask_numu_run1_fhc.loc[mc_dfs_run1_fhc['None'].index], 'weight_ubtune'].sum()
S0_numubar = mc_dfs_run1_fhc['None'].loc[mask_numubar_run1_fhc.loc[mc_dfs_run1_fhc['None'].index], 'weight_ubtune'].sum()

# 2) Compute νμ CC efficiency & purity at each step
eff_numu = []
pur_numu = []
for cut in cuts:
    df = mc_dfs_run1_fhc[cut]
    Sj = df.loc[mask_numu_run1_fhc.loc[df.index], 'weight_ubtune'].sum()
    Tj = df['weight_ubtune'].sum()
    eff_numu.append(Sj / S0_numu if S0_numu>0 else 0)
    pur_numu.append(Sj / Tj     if Tj>0     else 0)

# 3) Plot νμ CC
plt.figure()
plt.plot(cuts, eff_numu,        label=r'Eff $\nu_{\mu}$CC')
plt.plot(cuts, pur_numu,        label=r'Pur $\nu_{\mu}$CC')
plt.ylim(0, 1.1)
plt.xlabel('Selection Step')
plt.ylabel('Fraction')
plt.title(r'$\nu_{\mu}\,$CC Efficiency & Purity')
plt.xticks(rotation=45)
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt

# 1) Compute ν̄μ CC efficiency & purity
eff_numubar = []
pur_numubar = []
for cut in cuts:
    df = mc_dfs_run1_fhc[cut]
    Sj_bar = df.loc[mask_numubar_run1_fhc.loc[df.index], 'weight_ubtune'].sum()
    Tj     = df['weight_ubtune'].sum()
    eff_numubar.append(Sj_bar / S0_numubar if S0_numubar>0 else 0)
    pur_numubar.append(Sj_bar / Tj      if Tj>0      else 0)

# 2) Plot ν̄μ CC
plt.figure()
plt.plot(cuts, eff_numubar,     label=r'Eff $\bar\nu_{\mu}$CC')
plt.plot(cuts, pur_numubar,     label=r'Pur $\bar\nu_{\mu}$CC')
plt.ylim(0, 1.1)
plt.xlabel('Selection Step')
plt.ylabel('Fraction')
plt.title(r'$\bar\nu_{\mu}\,$CC Efficiency & Purity')
plt.xticks(rotation=45)
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt

# 1) Prepare
cuts = list(mc_dfs_run1_rhc.keys())
S0_numu    = mc_dfs_run1_rhc['None'].loc[mask_numu_run1_rhc.loc[mc_dfs_run1_rhc['None'].index], 'weight_ubtune'].sum()
S0_numubar = mc_dfs_run1_rhc['None'].loc[mask_numubar_run1_rhc.loc[mc_dfs_run1_rhc['None'].index], 'weight_ubtune'].sum()

# 2) Compute νμ CC efficiency & purity at each step
eff_numu = []
pur_numu = []
for cut in cuts:
    df = mc_dfs_run1_rhc[cut]
    Sj = df.loc[mask_numu_run1_rhc.loc[df.index], 'weight_ubtune'].sum()
    Tj = df['weight_ubtune'].sum()
    eff_numu.append(Sj / S0_numu if S0_numu>0 else 0)
    pur_numu.append(Sj / Tj     if Tj>0     else 0)

# 3) Plot νμ CC
plt.figure()
plt.plot(cuts, eff_numu,        label=r'Eff $\nu_{\mu}$CC')
plt.plot(cuts, pur_numu,        label=r'Pur $\nu_{\mu}$CC')
plt.ylim(0, 1.1)
plt.xlabel('Selection Step')
plt.ylabel('Fraction')
plt.title(r'$\nu_{\mu}\,$CC Efficiency & Purity')
plt.xticks(rotation=45)
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt

# 1) Compute ν̄μ CC efficiency & purity
eff_numubar = []
pur_numubar = []
for cut in cuts:
    df = mc_dfs_run1_rhc[cut]
    Sj_bar = df.loc[mask_numubar_run1_rhc.loc[df.index], 'weight_ubtune'].sum()
    Tj     = df['weight_ubtune'].sum()
    eff_numubar.append(Sj_bar / S0_numubar if S0_numubar>0 else 0)
    pur_numubar.append(Sj_bar / Tj      if Tj>0      else 0)

# 2) Plot ν̄μ CC
plt.figure()
plt.plot(cuts, eff_numubar,     label=r'Eff $\bar\nu_{\mu}$CC')
plt.plot(cuts, pur_numubar,     label=r'Pur $\bar\nu_{\mu}$CC')
plt.ylim(0, 1.1)
plt.xlabel('Selection Step')
plt.ylabel('Fraction')
plt.title(r'$\bar\nu_{\mu}\,$CC Efficiency & Purity')
plt.xticks(rotation=45)
plt.legend()
plt.tight_layout()
plt.show()


## Plot for run3 eff/pur

In [ ]:
import matplotlib.pyplot as plt

# 1) Prepare
cuts = list(mc_dfs_run3.keys())
S0_numu    = mc_dfs_run3['None'].loc[mask_numu_run3.loc[mc_dfs_run3['None'].index],    'weight_ubtune'].sum()
S0_numubar = mc_dfs_run3['None'].loc[mask_numubar_run3.loc[mc_dfs_run3['None'].index], 'weight_ubtune'].sum()

# 2) Compute νμ CC efficiency & purity at each step
eff_numu = []
pur_numu = []
for cut in cuts:
    df = mc_dfs_run3[cut]
    Sj = df.loc[mask_numu_run3.loc[df.index], 'weight_ubtune'].sum()
    Tj = df['weight_ubtune'].sum()
    eff_numu.append(Sj / S0_numu if S0_numu>0 else 0)
    pur_numu.append(Sj / Tj     if Tj>0     else 0)

# 3) Plot νμ CC
plt.figure()
plt.plot(cuts, eff_numu,        label=r'Eff $\nu_{\mu}$CC')
plt.plot(cuts, pur_numu,        label=r'Pur $\nu_{\mu}$CC')
plt.ylim(0, 1.1)
plt.xlabel('Selection Step')
plt.ylabel('Fraction')
plt.title(r'$\nu_{\mu}\,$CC Efficiency & Purity')
plt.xticks(rotation=45)
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt

# 1) Prepare
cuts = list(mc_dfs_run3.keys())
S0_numu    = mc_dfs_run3['None'].loc[mask_numu_run3.loc[mc_dfs_run3['None'].index],    'weight_ubtune'].sum()
S0_numubar = mc_dfs_run3['None'].loc[mask_numubar_run3.loc[mc_dfs_run3['None'].index], 'weight_ubtune'].sum()

# 2) Compute νμ CC efficiency & purity at each step
eff_numu = []
pur_numu = []
# temp cuts
cuts_wo_FC_run3 = ['None', 'genNuSelection', 'fiducialVol', 'muonCut']
for cut in cuts_wo_FC_run3:
    df = mc_dfs_run3[cut]
    Sj = df.loc[mask_numu_run3.loc[df.index], 'weight_ubtune'].sum()
    Tj = df['weight_ubtune'].sum()
    eff_numu.append(Sj / S0_numu if S0_numu>0 else 0)
    pur_numu.append(Sj / Tj     if Tj>0     else 0)

# 3) Plot νμ CC
plt.figure()
plt.plot(cuts_wo_FC_run3, eff_numu,        label=r'Eff $\nu_{\mu}$CC')
plt.plot(cuts_wo_FC_run3, pur_numu,        label=r'Pur $\nu_{\mu}$CC')
plt.ylim(0, 1.1)
plt.xlabel('Selection Step')
plt.ylabel('Fraction')
plt.title(r'$\nu_{\mu}\,$CC Efficiency & Purity')
plt.xticks(rotation=45)
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt

# 1) Compute ν̄μ CC efficiency & purity
eff_numubar = []
pur_numubar = []
for cut in cuts:
    df = mc_dfs_run3[cut]
    Sj_bar = df.loc[mask_numubar_run3.loc[df.index], 'weight_ubtune'].sum()
    Tj     = df['weight_ubtune'].sum()
    eff_numubar.append(Sj_bar / S0_numubar if S0_numubar>0 else 0)
    pur_numubar.append(Sj_bar / Tj      if Tj>0      else 0)

# 2) Plot ν̄μ CC
plt.figure()
plt.plot(cuts, eff_numubar,     label=r'Eff $\bar\nu_{\mu}$CC')
plt.plot(cuts, pur_numubar,     label=r'Pur $\bar\nu_{\mu}$CC')
plt.ylim(0, 1.1)
plt.xlabel('Selection Step')
plt.ylabel('Fraction')
plt.title(r'$\bar\nu_{\mu}\,$CC Efficiency & Purity')
plt.xticks(rotation=45)
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt

# 1) Compute ν̄μ CC efficiency & purity
eff_numubar = []
pur_numubar = []
# temp cuts
cuts_wo_FC_run3 = ['None', 'genNuSelection', 'fiducialVol', 'muonCut']
for cut in cuts_wo_FC_run3:
    df = mc_dfs_run3[cut]
    Sj_bar = df.loc[mask_numubar_run3.loc[df.index], 'weight_ubtune'].sum()
    Tj     = df['weight_ubtune'].sum()
    eff_numubar.append(Sj_bar / S0_numubar if S0_numubar>0 else 0)
    pur_numubar.append(Sj_bar / Tj      if Tj>0      else 0)

# 2) Plot ν̄μ CC
plt.figure()
plt.plot(cuts_wo_FC_run3, eff_numubar,     label=r'Eff $\bar\nu_{\mu}$CC')
plt.plot(cuts_wo_FC_run3, pur_numubar,     label=r'Pur $\bar\nu_{\mu}$CC')
plt.ylim(0, 1.1)
plt.xlabel('Selection Step')
plt.ylabel('Fraction')
plt.title(r'$\bar\nu_{\mu}\,$CC Efficiency & Purity')
plt.xticks(rotation=45)
plt.legend()
plt.tight_layout()
plt.show()


### COMMON RUN1 FHC and RUN3 PLOTS eff/pur on the same plot

In [ ]:
def eff_pur_vs_cuts(mc_dfs, mask_signal, weight_col='weight_ubtune', cuts=None):
    """
    Returns cuts, eff[], pur[] where:
      eff(cut) = S_j / S_0
      pur(cut) = S_j / T_j
    with weights.
    """
    if cuts is None:
        cuts = list(mc_dfs.keys())

    df0 = mc_dfs['None']
    S0 = df0.loc[mask_signal.loc[df0.index], weight_col].sum()

    eff, pur = [], []
    for cut in cuts:
        df = mc_dfs[cut]
        Sj = df.loc[mask_signal.loc[df.index], weight_col].sum()
        Tj = df[weight_col].sum()

        eff.append(Sj / S0 if S0 > 0 else np.nan)
        pur.append(Sj / Tj if Tj > 0 else np.nan)

    return cuts, np.array(eff), np.array(pur)


In [ ]:
cuts_common = ['None', 'genNuSelection', 'fiducialVol', 'muonCut', 'FC']


In [ ]:
# --- compute ---
cuts, eff_r1_numu, pur_r1_numu = eff_pur_vs_cuts(
    mc_dfs_run1_fhc, mask_numu_run1_fhc, cuts=cuts_common
)
_,    eff_r3_numu, pur_r3_numu = eff_pur_vs_cuts(
    mc_dfs_run3,     mask_numu_run3,     cuts=cuts_common
)

# --- plot ---
plt.figure()
plt.plot(cuts, eff_r1_numu, linestyle='-',  marker='o', label=r'Run 1 FHC Eff $\nu_{\mu}$CC')
plt.plot(cuts, pur_r1_numu, linestyle='--', marker='o', label=r'Run 1 FHC Pur $\nu_{\mu}$CC')

plt.plot(cuts, eff_r3_numu, linestyle='-',  marker='s', label=r'Run 3 Eff $\nu_{\mu}$CC')
plt.plot(cuts, pur_r3_numu, linestyle='--', marker='s', label=r'Run 3 Pur $\nu_{\mu}$CC')

plt.ylim(0, 1.1)
plt.xlabel('Selection Step')
plt.ylabel('Fraction')
plt.title(r'$\nu_{\mu}$CC Efficiency & Purity: Run 1 vs Run 3')
plt.xticks(rotation=45)
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
# --- compute ---
cuts, eff_r1_nub, pur_r1_nub = eff_pur_vs_cuts(
    mc_dfs_run1_fhc, mask_numubar_run1_fhc, cuts=cuts_common
)
_,    eff_r3_nub, pur_r3_nub = eff_pur_vs_cuts(
    mc_dfs_run3,     mask_numubar_run3,     cuts=cuts_common
)

# --- plot ---
plt.figure()
plt.plot(cuts, eff_r1_nub, linestyle='-',  marker='o', label=r'Run 1 FHC Eff $\bar\nu_{\mu}$CC')
plt.plot(cuts, pur_r1_nub, linestyle='--', marker='o', label=r'Run 1 FHC Pur $\bar\nu_{\mu}$CC')

plt.plot(cuts, eff_r3_nub, linestyle='-',  marker='s', label=r'Run 3 Eff $\bar\nu_{\mu}$CC', alpha=0.6)
plt.plot(cuts, pur_r3_nub, linestyle='--', marker='s', label=r'Run 3 Pur $\bar\nu_{\mu}$CC', alpha=0.7)

plt.ylim(0, 1.1)
plt.xlabel('Selection Step')
plt.ylabel('Fraction')
plt.title(r'$\bar\nu_{\mu}$CC Efficiency & Purity: Run 1 vs Run 3')
plt.xticks(rotation=45)
plt.legend()
plt.tight_layout()
plt.show()


## EFF/PUR FHC AND RHC FULL

In [ ]:
# ============================================================
# Combined νμ + ν̄μ CC signal masks
# ============================================================

mask_numu_numubar_RHC = (
    ((df_mc_RHC.truth_nuPdg == 14) | (df_mc_RHC.truth_nuPdg == -14)) &
    (df_mc_RHC.truth_isCC == 1) &
    (df_mc_RHC.truth_vtxInside == 1)
)

mask_numu_numubar_FHC = (
    ((df_mc_FHC.truth_nuPdg == 14) | (df_mc_FHC.truth_nuPdg == -14)) &
    (df_mc_FHC.truth_isCC == 1) &
    (df_mc_FHC.truth_vtxInside == 1)
)

In [ ]:
def compute_bayesian_cutflow_weighted_combined(mc_dfs,
                                               mask_signal,
                                               weight_col='weight_ubtune',
                                               alpha=0.683,
                                               digits=3):
    """
    Combined signal version:
      signal = νμ CC + ν̄μ CC (or any user-provided mask)

    Returns efficiency and purity for the combined signal.
    """
    def fmt(val, lo, hi):
        err = (hi - lo) / 2
        err_int = int(round(err * 10**digits))
        return f"{val:.{digits}f}({err_int})"

    df0 = mc_dfs['None']
    S0 = df0.loc[mask_signal.loc[df0.index], weight_col].sum()

    rows = []
    for cut, df in mc_dfs.items():
        Tj = df[weight_col].sum()
        Sj = df.loc[mask_signal.reindex(df.index, fill_value=False), weight_col].sum()

        eff = Sj / S0 if S0 > 0 else np.nan
        pur = Sj / Tj if Tj > 0 else np.nan

        aE, bE = Sj + 1, S0 - Sj + 1
        aP, bP = Sj + 1, Tj - Sj + 1

        loE, hiE = beta.interval(alpha, aE, bE)
        loP, hiP = beta.interval(alpha, aP, bP)

        rows.append({
            'Cut': cut,
            'Eff νμ+ν̄μ CC': fmt(eff, loE, hiE),
            'Pur νμ+ν̄μ CC': fmt(pur, loP, hiP),
            'Selected (w)': Tj
        })

    return pd.DataFrame(rows)

In [ ]:
mc_dfs_RHC = {
    'None':           df_mc_RHC,
    'genNuSelection': df_mc1_RHC,
    'fiducialVol':    df_mc2_RHC,
    'muonCut':        df_mc4_RHC,
    'FC':             df_mcFC_RHC
}

mc_dfs_FHC = {
    'None':           df_mc_FHC,
    'genNuSelection': df_mc1_FHC,
    'fiducialVol':    df_mc2_FHC,
    'muonCut':        df_mc4_FHC,
    'FC':             df_mcFC_FHC
}

cutflow_fmt_RHC_combined = compute_bayesian_cutflow_weighted_combined(
    mc_dfs_RHC,
    mask_numu_numubar_RHC,
    weight_col='weight_ubtune'
)

cutflow_fmt_FHC_combined = compute_bayesian_cutflow_weighted_combined(
    mc_dfs_FHC,
    mask_numu_numubar_FHC,
    weight_col='weight_ubtune'
)

cutflow_fmt_RHC_combined
cutflow_fmt_FHC_combined

In [ ]:
mask_numu_numubar_RHC = (
    ((df_mc_RHC.truth_nuPdg == 14) | (df_mc_RHC.truth_nuPdg == -14)) &
    (df_mc_RHC.truth_isCC == 1) &
    (df_mc_RHC.truth_vtxInside == 1)
)

mask_numu_numubar_FHC = (
    ((df_mc_FHC.truth_nuPdg == 14) | (df_mc_FHC.truth_nuPdg == -14)) &
    (df_mc_FHC.truth_isCC == 1) &
    (df_mc_FHC.truth_vtxInside == 1)
)

cuts, eff_rhc_comb, pur_rhc_comb = eff_pur_vs_cuts(mc_dfs_RHC, mask_numu_numubar_RHC, cuts=cuts_common)
_,    eff_fhc_comb, pur_fhc_comb = eff_pur_vs_cuts(mc_dfs_FHC, mask_numu_numubar_FHC, cuts=cuts_common)

plt.figure()
plt.plot(cuts, eff_rhc_comb, linestyle='-', marker='o', label='RHC Eff')
plt.plot(cuts, pur_rhc_comb, linestyle='--', marker='o', label='RHC Pur')
plt.plot(cuts, eff_fhc_comb, linestyle='-', marker='s', label='FHC Eff')
plt.plot(cuts, pur_fhc_comb, linestyle='--', marker='s', label='FHC Pur')
plt.ylim(0, 1.1)
plt.xlabel('Selection Step')
plt.ylabel('Fraction')
plt.title(r'$(\nu_{\mu}+\bar{\nu}_{\mu})$ CC Efficiency & Purity')
plt.xticks(rotation=45)
plt.legend()
plt.tight_layout()
plt.show()

## Plot later BDT training var against the efficiency to look for model dependency

## Applying reconstructed michel selection (for the creation of Michel variable and analysis for Michel selection)

In [ ]:
df_dataFC_RHC_michel, df_extFC_RHC_michel, df_mcFC_RHC_michel, df_dirtFC_RHC_michel = applyCuts('recoMichel', df_dataFC_RHC, df_extFC_RHC, df_mcFC_RHC, df_dirtFC_RHC, 'weight_ubtune', True, POT_data=RHC_full_POT)

In [ ]:
df_dataFC_FHC_michel, df_extFC_FHC_michel, df_mcFC_FHC_michel, df_dirtFC_FHC_michel = applyCuts('recoMichel', df_dataFC_FHC, df_extFC_FHC, df_mcFC_FHC, df_dirtFC_FHC, 'weight_ubtune', True, POT_data=FHC_full_POT)

# Add a new variable called "has_reco_Michel"

In [ ]:
# Data
keys_data_RHC = set(zip(df_dataFC_RHC_michel['run'], df_dataFC_RHC_michel['subrun'], df_dataFC_RHC_michel['event']))
df_dataFC_RHC['has_reco_michel'] = df_dataFC_RHC.apply(
    lambda row: 1 if (row['run'], row['subrun'], row['event']) in keys_data_RHC else 0,
    axis=1
)

# EXT
keys_ext_RHC = set(zip(df_extFC_RHC_michel['run'], df_extFC_RHC_michel['subrun'], df_extFC_RHC_michel['event']))
df_extFC_RHC['has_reco_michel'] = df_extFC_RHC.apply(
    lambda row: 1 if (row['run'], row['subrun'], row['event']) in keys_ext_RHC else 0,
    axis=1
)

# MC
keys_mc_RHC = set(zip(df_mcFC_RHC_michel['run'], df_mcFC_RHC_michel['subrun'], df_mcFC_RHC_michel['event']))
df_mcFC_RHC['has_reco_michel'] = df_mcFC_RHC.apply(
    lambda row: 1 if (row['run'], row['subrun'], row['event']) in keys_mc_RHC else 0,
    axis=1
)

# Dirt
keys_dirt_RHC = set(zip(df_dirtFC_RHC_michel['run'], df_dirtFC_RHC_michel['subrun'], df_dirtFC_RHC_michel['event']))
df_dirtFC_RHC['has_reco_michel'] = df_dirtFC_RHC.apply(
    lambda row: 1 if (row['run'], row['subrun'], row['event']) in keys_dirt_RHC else 0,
    axis=1
)


In [ ]:
# Data
keys_data_FHC = set(zip(df_dataFC_FHC_michel['run'], df_dataFC_FHC_michel['subrun'], df_dataFC_FHC_michel['event']))
df_dataFC_FHC['has_reco_michel'] = df_dataFC_FHC.apply(
    lambda row: 1 if (row['run'], row['subrun'], row['event']) in keys_data_FHC else 0,
    axis=1
)

# EXT
keys_ext_FHC = set(zip(df_extFC_FHC_michel['run'], df_extFC_FHC_michel['subrun'], df_extFC_FHC_michel['event']))
df_extFC_FHC['has_reco_michel'] = df_extFC_FHC.apply(
    lambda row: 1 if (row['run'], row['subrun'], row['event']) in keys_ext_FHC else 0,
    axis=1
)

# MC
keys_mc_FHC = set(zip(df_mcFC_FHC_michel['run'], df_mcFC_FHC_michel['subrun'], df_mcFC_FHC_michel['event']))
df_mcFC_FHC['has_reco_michel'] = df_mcFC_FHC.apply(
    lambda row: 1 if (row['run'], row['subrun'], row['event']) in keys_mc_FHC else 0,
    axis=1
)

# Dirt
keys_dirt_FHC = set(zip(df_dirtFC_FHC_michel['run'], df_dirtFC_FHC_michel['subrun'], df_dirtFC_FHC_michel['event']))
df_dirtFC_FHC['has_reco_michel'] = df_dirtFC_FHC.apply(
    lambda row: 1 if (row['run'], row['subrun'], row['event']) in keys_dirt_FHC else 0,
    axis=1
)


In [ ]:
# # Initialize the column with NaNs
# df_dataFC_RHC['reco_michel_energy'] = np.nan
# df_extFC_RHC['reco_michel_energy'] = np.nan
# df_mcFC_RHC['reco_michel_energy'] = np.nan
# df_dirtFC_RHC['reco_michel_energy'] = np.nan

# # Loop only through rows where has_truth_michel == 1
# for idx, row in df_mcFC_RHC[df_mcFC_RHC['has_reco_michel'] == 1].iterrows():
#     reco_pdg = row['reco_pdg']
#     reco_mother = row['reco_mother']
#     reco_id = row['reco_id']
#     reco_startMomentum = row['reco_startMomentum']

#     for i, pdg in enumerate(reco_pdg):
#         if pdg in (11, -11):  # electron or positron
#             mother_id = reco_mother[i]
#             mother_index = np.where(reco_id == mother_id)[0]
#             if len(mother_index) == 0:
#                 continue
#             mother_pdg = reco_pdg[mother_index]

#             if ((pdg == 11 and mother_pdg == 13) or (pdg == -11 and mother_pdg == -13)) and reco_mother[mother_index] == 0:
#                 df_mcFC_RHC.at[idx, 'reco_michel_energy'] = reco_startMomentum[i][3]  # energy in GeV
#                 break

# df_mcFC_RHC['reco_michel_energy'] = df_mcFC_RHC['reco_michel_energy'].fillna(0)

# for idx, row in df_dirtFC_run3[df_dirtFC_run3['has_reco_michel'] == 1].iterrows():
#     reco_pdg = row['reco_pdg']
#     reco_mother = row['reco_mother']
#     reco_id = row['reco_id']
#     reco_startMomentum = row['reco_startMomentum']

#     for i, pdg in enumerate(reco_pdg):
#         if pdg in (11, -11):  # electron or positron
#             mother_id = reco_mother[i]
#             mother_index = np.where(reco_id == mother_id)[0]
#             if len(mother_index) == 0:
#                 continue
#             mother_pdg = reco_pdg[mother_index]

#             if ((pdg == 11 and mother_pdg == 13) or (pdg == -11 and mother_pdg == -13)) and reco_mother[mother_index] == 0:
#                 df_dirtFC_run3.at[idx, 'reco_michel_energy'] = reco_startMomentum[i][3]  # energy in GeV
#                 break

# df_dirtFC_run3['reco_michel_energy'] = df_dirtFC_run3['reco_michel_energy'].fillna(0)

# for idx, row in df_dataFC_run3[df_dataFC_run3['has_reco_michel'] == 1].iterrows():
#     reco_pdg = row['reco_pdg']
#     reco_mother = row['reco_mother']
#     reco_id = row['reco_id']
#     reco_startMomentum = row['reco_startMomentum']

#     for i, pdg in enumerate(reco_pdg):
#         if pdg in (11, -11):  # electron or positron
#             mother_id = reco_mother[i]
#             mother_index = np.where(reco_id == mother_id)[0]
#             if len(mother_index) == 0:
#                 continue
#             mother_pdg = reco_pdg[mother_index]

#             if ((pdg == 11 and mother_pdg == 13) or (pdg == -11 and mother_pdg == -13)) and reco_mother[mother_index] == 0:
#                 df_dataFC_run3.at[idx, 'reco_michel_energy'] = reco_startMomentum[i][3]  # energy in GeV
#                 break

# df_dataFC_run3['reco_michel_energy'] = df_dataFC_run3['reco_michel_energy'].fillna(0)

# for idx, row in df_extFC_run3[df_extFC_run3['has_reco_michel'] == 1].iterrows():
#     reco_pdg = row['reco_pdg']
#     reco_mother = row['reco_mother']
#     reco_id = row['reco_id']
#     reco_startMomentum = row['reco_startMomentum']

#     for i, pdg in enumerate(reco_pdg):
#         if pdg in (11, -11):  # electron or positron
#             mother_id = reco_mother[i]
#             mother_index = np.where(reco_id == mother_id)[0]
#             if len(mother_index) == 0:
#                 continue
#             mother_pdg = reco_pdg[mother_index]

#             if ((pdg == 11 and mother_pdg == 13) or (pdg == -11 and mother_pdg == -13)) and reco_mother[mother_index] == 0:
#                 df_extFC_run3.at[idx, 'reco_michel_energy'] = reco_startMomentum[i][3]  # energy in GeV
#                 break

# df_extFC_run3['reco_michel_energy'] = df_extFC_run3['reco_michel_energy'].fillna(0)

In [ ]:
# def reco_michel_muon_angle(df_):
#     # Initialize new columns

#     df_['reco_michel_muon_costheta'] = np.nan

#     for idx, row in df_[(df_['has_reco_michel'] == 1) | (df_['reco_michel_energy'] >= 0)].iterrows():
#         reco_pdg = row['reco_pdg']
#         reco_mother = row['reco_mother']
#         reco_id = row['reco_id']
#         reco_startMomentum = row['reco_startMomentum']

#         for i, pdg in enumerate(reco_pdg):
#             if pdg in (11, -11):  # electron/positron
#                 mother_id = reco_mother[i]
#                 mother_index = np.where(reco_id == mother_id)[0]
#                 if len(mother_index) == 0:
#                     continue
#                 m_idx = mother_index[0]

#                 mother_pdg = reco_pdg[m_idx]

#                 if ((pdg == 11 and mother_pdg == 13) or (pdg == -11 and mother_pdg == -13)) and reco_mother[m_idx] == 0:
#                     df_.loc[idx, 'reco_michel_px'] = reco_startMomentum[i][0]
#                     df_.loc[idx, 'reco_michel_py'] = reco_startMomentum[i][1]
#                     df_.loc[idx, 'reco_michel_pz'] = reco_startMomentum[i][2]
#                     df_.loc[idx, 'reco_muon_px'] = reco_startMomentum[m_idx][0]
#                     df_.loc[idx, 'reco_muon_py'] = reco_startMomentum[m_idx][1]
#                     df_.loc[idx, 'reco_muon_pz'] = reco_startMomentum[m_idx][2]
#                     break

#     # Compute norms and cosine of the angle
#     df_.eval('norm_reco_michel = sqrt(reco_michel_px**2 + reco_michel_py**2 + reco_michel_pz**2)', inplace=True)
#     df_.eval('norm_reco_muon = sqrt(reco_muon_px**2 + reco_muon_py**2 + reco_muon_pz**2)', inplace=True)

#     df_.eval('reco_michel_muon_costheta = (reco_michel_px * reco_muon_px + reco_michel_py * reco_muon_py + reco_michel_pz * reco_muon_pz) / (norm_reco_michel * norm_reco_muon)', inplace=True)

#     df_.loc[(df_['has_reco_michel'] == 0) | (df_['reco_michel_energy'] <= 0), 'reco_michel_muon_costheta'] = -999
#     return df_


In [ ]:
# df_mcFC = reco_michel_muon_angle(df_mcFC_run3)
# df_dirtFC = reco_michel_muon_angle(df_dirtFC_run3)
# df_dataFC = reco_michel_muon_angle(df_dataFC_run3)
# df_extFC = reco_michel_muon_angle(df_extFC_run3)

## Applying truth michel selection (for the creation of truth Michel variable and analysis for Michel selection)

In [ ]:
df_dataFC_RHC, df_extFC_RHC, df_mcFC_RHC_truthMichel, df_dirtFC_RHC_truthMichel = applyCuts('truthMichel', df_dataFC_RHC, df_extFC_RHC, df_mcFC_RHC, df_dirtFC_RHC, 'weight_ubtune', True, POT_data=RHC_full_POT, plotCondition='truthMichel')
df_dataFC_RHC, df_extFC_RHC, df_mcFC_RHC_truthMichelPositron, df_dirtFC_RHC_truthMichelPositron = applyCuts('truthMichelPositrons', df_dataFC_RHC, df_extFC_RHC, df_mcFC_RHC, df_dirtFC_RHC, 'weight_ubtune', True, POT_data=RHC_full_POT, plotCondition='truthMichelPositrons')
df_dataFC_RHC, df_extFC_RHC, df_mcFC_RHC_truthMichelElectron, df_dirtFC_RHC_truthMichelElectron = applyCuts('truthMichelElectrons', df_dataFC_RHC, df_extFC_RHC, df_mcFC_RHC, df_dirtFC_RHC, 'weight_ubtune', True, POT_data=RHC_full_POT, plotCondition='truthMichelElectrons')

In [ ]:
df_dataFC_FHC, df_extFC_FHC, df_mcFC_FHC_truthMichel, df_dirtFC_FHC_truthMichel = applyCuts('truthMichel', df_dataFC_FHC, df_extFC_FHC, df_mcFC_FHC, df_dirtFC_FHC, 'weight_ubtune', True, POT_data=FHC_full_POT, plotCondition='truthMichel')
df_dataFC_FHC, df_extFC_FHC, df_mcFC_FHC_truthMichelPositron, df_dirtFC_FHC_truthMichelPositron = applyCuts('truthMichelPositrons', df_dataFC_FHC, df_extFC_FHC, df_mcFC_FHC, df_dirtFC_FHC, 'weight_ubtune', True, POT_data=FHC_full_POT, plotCondition='truthMichelPositrons')
df_dataFC_FHC, df_extFC_FHC, df_mcFC_FHC_truthMichelElectron, df_dirtFC_FHC_truthMichelElectron = applyCuts('truthMichelElectrons', df_dataFC_FHC, df_extFC_FHC, df_mcFC_FHC, df_dirtFC_FHC, 'weight_ubtune', True, POT_data=FHC_full_POT, plotCondition='truthMichelElectrons')

# Add a new variable called "has_truth_Michel"

In [ ]:
# MC
keys_mc_RHC_truth = set(zip(df_mcFC_RHC_truthMichel['run'], df_mcFC_RHC_truthMichel['subrun'], df_mcFC_RHC_truthMichel['event']))
df_mcFC_RHC['has_truth_michel'] = df_mcFC_RHC.apply(
    lambda row: 1 if (row['run'], row['subrun'], row['event']) in keys_mc_RHC_truth else 0,
    axis=1
)

# Dirt
keys_dirt_RHC_truth = set(zip(df_dirtFC_RHC_truthMichel['run'], df_dirtFC_RHC_truthMichel['subrun'], df_dirtFC_RHC_truthMichel['event']))
df_dirtFC_RHC['has_truth_michel'] = df_dirtFC_RHC.apply(
    lambda row: 1 if (row['run'], row['subrun'], row['event']) in keys_dirt_RHC_truth else 0,
    axis=1
)


In [ ]:
# MC
keys_mc_FHC_truth = set(zip(df_mcFC_FHC_truthMichel['run'], df_mcFC_FHC_truthMichel['subrun'], df_mcFC_FHC_truthMichel['event']))
df_mcFC_FHC['has_truth_michel'] = df_mcFC_FHC.apply(
    lambda row: 1 if (row['run'], row['subrun'], row['event']) in keys_mc_FHC_truth else 0,
    axis=1
)

# Dirt
keys_dirt_FHC_truth = set(zip(df_dirtFC_FHC_truthMichel['run'], df_dirtFC_FHC_truthMichel['subrun'], df_dirtFC_FHC_truthMichel['event']))
df_dirtFC_FHC['has_truth_michel'] = df_dirtFC_FHC.apply(
    lambda row: 1 if (row['run'], row['subrun'], row['event']) in keys_dirt_FHC_truth else 0,
    axis=1
)


In [ ]:
# # Initialize the column with NaNs
# df_mcFC_run3['truth_michel_energy'] = np.nan
# df_dirtFC_run3['truth_michel_energy'] = np.nan

# # Loop only through rows where has_truth_michel == 1
# for idx, row in df_mcFC_run3[df_mcFC_run3['has_truth_michel'] == 1].iterrows():
#     truth_pdg = row['truth_pdg']
#     truth_mother = row['truth_mother']
#     truth_id = row['truth_id']
#     truth_startMomentum = row['truth_startMomentum']

#     for i, pdg in enumerate(truth_pdg):
#         if pdg in (11, -11):  # electron or positron
#             mother_id = truth_mother[i]
#             mother_index = np.where(truth_id == mother_id)[0]
#             if len(mother_index) == 0:
#                 continue
#             mother_pdg = truth_pdg[mother_index]

#             if ((pdg == 11 and mother_pdg == 13) or (pdg == -11 and mother_pdg == -13)) and truth_mother[mother_index] == 0:
#                 df_mcFC.at[idx, 'truth_michel_energy'] = truth_startMomentum[i][3]  # energy in GeV
#                 break

# df_mcFC_run3['truth_michel_energy'] = df_mcFC_run3['truth_michel_energy'].fillna(0)

# for idx, row in df_dirtFC_run3[df_dirtFC_run3['has_truth_michel'] == 1].iterrows():
#     truth_pdg = row['truth_pdg']
#     truth_mother = row['truth_mother']
#     truth_id = row['truth_id']
#     truth_startMomentum = row['truth_startMomentum']

#     for i, pdg in enumerate(truth_pdg):
#         if pdg in (11, -11):  # electron or positron
#             mother_id = truth_mother[i]
#             mother_index = np.where(truth_id == mother_id)[0]
#             if len(mother_index) == 0:
#                 continue
#             mother_pdg = truth_pdg[mother_index]

#             if ((pdg == 11 and mother_pdg == 13) or (pdg == -11 and mother_pdg == -13)) and truth_mother[mother_index] == 0:
#                 df_dirtFC.at[idx, 'truth_michel_energy'] = truth_startMomentum[i][3]  # energy in GeV
#                 break

# # Fill in -999 for rows without valid Michel energy
# df_dirtFC_run3['truth_michel_energy'] = df_dirtFC_run3['truth_michel_energy'].fillna(0)


In [ ]:
# def truth_michel_muon_angle(df_):
#     # Initialize new columns

#     df_['truth_michel_muon_costheta'] = np.nan

#     for idx, row in df_[(df_['has_truth_michel'] == 1) | (df_['truth_michel_energy'] > 0)].iterrows():
#         truth_pdg = row['truth_pdg']
#         truth_mother = row['truth_mother']
#         truth_id = row['truth_id']
#         truth_startMomentum = row['truth_startMomentum']

#         for i, pdg in enumerate(truth_pdg):
#             if pdg in (11, -11):  # electron/positron
#                 mother_id = truth_mother[i]
#                 mother_index = np.where(truth_id == mother_id)[0]
#                 if len(mother_index) == 0:
#                     continue
#                 m_idx = mother_index[0]

#                 mother_pdg = truth_pdg[m_idx]

#                 if ((pdg == 11 and mother_pdg == 13) or (pdg == -11 and mother_pdg == -13)) and truth_mother[m_idx] == 0:
#                     df_.loc[idx, 'truth_michel_px'] = truth_startMomentum[i][0]
#                     df_.loc[idx, 'truth_michel_py'] = truth_startMomentum[i][1]
#                     df_.loc[idx, 'truth_michel_pz'] = truth_startMomentum[i][2]
#                     df_.loc[idx, 'truth_muon_px'] = truth_startMomentum[m_idx][0]
#                     df_.loc[idx, 'truth_muon_py'] = truth_startMomentum[m_idx][1]
#                     df_.loc[idx, 'truth_muon_pz'] = truth_startMomentum[m_idx][2]
#                     break

#     # Compute norms and cosine of the angle
#     df_.eval('norm_truth_michel = sqrt(truth_michel_px**2 + truth_michel_py**2 + truth_michel_pz**2)', inplace=True)
#     df_.eval('norm_truth_muon = sqrt(truth_muon_px**2 + truth_muon_py**2 + truth_muon_pz**2)', inplace=True)

#     df_.eval('truth_michel_muon_costheta = (truth_michel_px * truth_muon_px + truth_michel_py * truth_muon_py + truth_michel_pz * truth_muon_pz) / (norm_truth_michel * norm_truth_muon)', inplace=True)

#     df_.loc[(df_['has_truth_michel'] == 0) | (df_['truth_michel_energy'] <= 0), 'truth_michel_muon_costheta'] = -999

#     return df_


In [ ]:
# df_mcFC_run3 = truth_michel_muon_angle(df_mcFC_run3)
# df_dirtFC_run3 = truth_michel_muon_angle(df_dirtFC_run3)

In [ ]:
reco_keys_RHC = set(zip(
    df_mcFC_RHC[df_mcFC_RHC['has_reco_michel'] == 1]['run'],
    df_mcFC_RHC[df_mcFC_RHC['has_reco_michel'] == 1]['subrun'],
    df_mcFC_RHC[df_mcFC_RHC['has_reco_michel'] == 1]['event']
))

truth_keys_RHC = set(zip(
    df_mcFC_RHC[df_mcFC_RHC['has_truth_michel'] == 1]['run'],
    df_mcFC_RHC[df_mcFC_RHC['has_truth_michel'] == 1]['subrun'],
    df_mcFC_RHC[df_mcFC_RHC['has_truth_michel'] == 1]['event']
))

# Intersection: events that have both reco and truth Michel candidates
matched_keys_RHC = reco_keys_RHC & truth_keys_RHC

matched_df_RHC = df_mcFC_RHC[
    df_mcFC_RHC.apply(lambda row: (row['run'], row['subrun'], row['event']) in matched_keys_RHC, axis=1)
]

# Step 4: Sum the weights
total_weight_RHC = matched_df_RHC['weight_ubtune'].sum()
print(f"Total weight for matched reco + truth Michel events for full RHC: {total_weight_RHC:.3f}")

In [ ]:
reco_keys_FHC = set(zip(
    df_mcFC_FHC[df_mcFC_FHC['has_reco_michel'] == 1]['run'],
    df_mcFC_FHC[df_mcFC_FHC['has_reco_michel'] == 1]['subrun'],
    df_mcFC_FHC[df_mcFC_FHC['has_reco_michel'] == 1]['event']
))

truth_keys_FHC = set(zip(
    df_mcFC_FHC[df_mcFC_FHC['has_truth_michel'] == 1]['run'],
    df_mcFC_FHC[df_mcFC_FHC['has_truth_michel'] == 1]['subrun'],
    df_mcFC_FHC[df_mcFC_FHC['has_truth_michel'] == 1]['event']
))

# Intersection: events that have both reco and truth Michel candidates
matched_keys_FHC = reco_keys_FHC & truth_keys_FHC

matched_df_FHC = df_mcFC_FHC[
    df_mcFC_FHC.apply(lambda row: (row['run'], row['subrun'], row['event']) in matched_keys_FHC, axis=1)
]

# Step 4: Sum the weights
total_weight_FHC = matched_df_FHC['weight_ubtune'].sum()
print(f"Total weight for matched reco + truth Michel events for full FHC: {total_weight_FHC:.3f}")

In [ ]:
df_mc4_RHC.to_pickle(os.path.join(dfs_dir, 'df_mc4_RHC.pkl'))
df_data4_RHC.to_pickle(os.path.join(dfs_dir, 'df_data4_RHC.pkl'))
df_dirt4_RHC.to_pickle(os.path.join(dfs_dir, 'df_dirt4_RHC.pkl'))
df_ext4_RHC.to_pickle(os.path.join(dfs_dir, 'df_ext4_RHC.pkl'))

In [ ]:
df_mcFC_RHC.to_pickle(os.path.join(dfs_dir, 'df_mcFC_RHC.pkl'))
df_dataFC_RHC.to_pickle(os.path.join(dfs_dir, 'df_dataFC_RHC.pkl'))
df_dirtFC_RHC.to_pickle(os.path.join(dfs_dir, 'df_dirtFC_RHC.pkl'))
df_extFC_RHC.to_pickle(os.path.join(dfs_dir, 'df_extFC_RHC.pkl'))

In [ ]:
df_mc4_FHC.to_pickle(os.path.join(dfs_dir, 'df_mc4_FHC.pkl'))
df_data4_FHC.to_pickle(os.path.join(dfs_dir, 'df_data4_FHC.pkl'))
df_dirt4_FHC.to_pickle(os.path.join(dfs_dir, 'df_dirt4_FHC.pkl'))
df_ext4_FHC.to_pickle(os.path.join(dfs_dir, 'df_ext4_FHC.pkl'))

In [ ]:
df_mcFC_FHC.to_pickle(os.path.join(dfs_dir, 'df_mcFC_FHC.pkl'))
df_dataFC_FHC.to_pickle(os.path.join(dfs_dir, 'df_dataFC_FHC.pkl'))
df_dirtFC_FHC.to_pickle(os.path.join(dfs_dir, 'df_dirtFC_FHC.pkl'))
df_extFC_FHC.to_pickle(os.path.join(dfs_dir, 'df_extFC_FHC.pkl'))

In [ ]:
# lock6 = True
# if lock6==True:    
#     # Storing Selected mixed numu & anti-numu sample 
#     df_mcFC_run3.to_hdf('dfs/df_run3b_mc_PC_mixed_numu_numubar.hdf5', key='df', mode='w')
#     df_dataFC_run3.to_hdf('dfs/df_run3b_data_PC_mixed_numu_numubar.hdf5', key='df', mode='w')
#     df_dirtFC_run3.to_hdf('dfs/df_run3b_dirt_PC_mixed_numu_numubar.hdf5', key='df', mode='w')
#     df_extFC_run3.to_hdf('dfs/df_run3b_ext_PC_mixed_numu_numubar.hdf5', key='df', mode='w')

## Investigating, selecting Michel Electrons for Hand-scanning and Evt Display purpose

Should quickly select Michel electrons using truth variables and than get the event/subrun/run/start or end position for hand-scanning purposes

GOAL: Find positrons in truth_pdg (-11) that has a truth_mother of anti-muon (-13) [NOT taking into account of muon absorption, so just use mu+ that won't be absorbed]

In [ ]:
# #==========================================#
# # A Function for selecting truth electrons #
# #==========================================#

# def find_michel_with_mu(df):
#     selected_events = []
#     bad_mother_count = 0  # Counter for bad mother indices

#     for idx, row in df.iterrows():
#         truth_pdg = row['truth_pdg']
#         truth_mother = row['truth_mother']
#         truth_id = row['truth_id']
#         truth_startMomentum = row['truth_startMomentum']

#         for i, pdg in enumerate(truth_pdg):
#             if pdg == -11 or pdg == 11:  # electron or positron
#                 mother_id = truth_mother[i]
#                 mother_index = np.where(truth_id == mother_id)[0]

#                 # if mother_index < 0 or mother_index >= len(truth_pdg):
#                 #     # Bad mother index
#                 #     bad_mother_count += 1
#                 #     break  # Skip to next row
            
#                 if truth_pdg[mother_index] == -13 or truth_pdg[mother_index] == 13:
#                     energy = truth_startMomentum[i][3]  # 4th component is energy
#                     if energy <= 0.07: # 50 MeV = 0.05 GeV
#                         selected_events.append((
#                             row['run'], row['subrun'], row['event'], i, energy,
#                             row['truth_muonendX'], row['truth_muonendY'], row['truth_muonendZ'], row['weight_ubtune']
#                         ))

                    
#                     break  # One match per event is enough

#     # print(f"Number of bad mother indices: {bad_mother_count}")
#     return selected_events


In [ ]:
#==========================================#
# A Function for selecting truth electrons #
#==========================================#

def find_truth_michel_with_mu(df):
    selected_events = []
    bad_mother_count = 0  # Counter for bad mother indices

    for idx, row in df.iterrows():
        truth_pdg = row['truth_pdg']
        truth_mother = row['truth_mother']
        truth_id = row['truth_id']
        truth_startMomentum = row['truth_startMomentum']

        for i, pdg in enumerate(truth_pdg):
            if pdg == 11 or pdg == -11:  # electron or positron
            #if pdg == -11:  # electron or positron
                mother_id = truth_mother[i]
                mother_index = np.where(truth_id == mother_id)[0]
                mother_pdg = truth_pdg[mother_index]

                # if mother_index < 0 or mother_index >= len(truth_pdg):
                #     # Bad mother index
                #     bad_mother_count += 1
                #     break  # Skip to next row
            
                if (pdg == 11 and mother_pdg == 13) or (pdg == -11 and mother_pdg == -13):
                #if (pdg == -11 and mother_pdg == -13):
                    energy = truth_startMomentum[i][3]  # 4th component is energy
                    if energy <= 0.07: # 50 MeV = 0.05 GeV
                        selected_events.append((
                            row['run'], row['subrun'], row['event'], i, energy,
                            row['truth_muonendX'], row['truth_muonendY'], row['truth_muonendZ'], row['weight_ubtune']
                        ))

                    
                    break  # One match per event is enough

    # print(f"Number of bad mother indices: {bad_mother_count}")
    return selected_events


In [ ]:
# Apply the function to df_mc
truth_michel_events_FC = find_truth_michel_with_mu(df_mcFC_run3)
total_weighted_number_michel = sum(event[-1] for event in truth_michel_events_FC)

# Display the results
print(f"Total weighted number of Michel electrons: {total_weighted_number_michel}")
print("Selected Events with Positrons from Anti-Muons:")
for event in truth_michel_events_FC:
    print(f"Run: {event[0]}, Subrun: {event[1]}, Event: {event[2]}, Michel Index: {event[3]}, Energy: {event[4]*1000:.4f} MeV, muon_endX: {event[5]:.2f}, muon_endY: {event[6]:.2f}, muon_endZ: {event[7]:.2f}, Weight: {event[8]:.2f}")

In [ ]:
# Convert the result into a dataframe
truth_michel_df_FC = pd.DataFrame(truth_michel_events_FC, columns=['Run', 'Subrun', 'Event', 'Michel_Index', 'truth_energy', 'Muon_endX', 'Muon_endY', 'Muon_endZ', 'weight_ubtune'])

# Display the new dataframe
truth_michel_df_FC.head(-5)

In [ ]:
# # Save positron_df_FC to a text file
# positron_df_FC[['Run', 'Subrun', 'Event']].to_csv('positron_df_FC.txt', sep=' ', header=False, index=False)

In [ ]:
#===========================================#
# A Function for selecting reco M Electrons #
#===========================================#

def find_reco_michel_with_mu(df):
    selected_events = []
    bad_mother_count = 0  # Counter for bad mother indices

    for idx, row in df.iterrows():
        reco_pdg = row['reco_pdg']
        reco_mother = row['reco_mother']
        reco_id = row['reco_id']
        reco_startMomentum = row['reco_startMomentum']

        for i, pdg in enumerate(reco_pdg):
            if pdg == 11 or pdg == -11:  # electron or positron
            #if pdg == -11:
                mother_id = reco_mother[i]
                mother_index = np.where(reco_id == mother_id)[0]
                mother_pdg = reco_pdg[mother_index]

                if len(mother_index) == 0:
                    bad_mother_count += 1
                    break
            
                if (
                    ((pdg == 11 and mother_pdg == 13) or (pdg == -11 and mother_pdg == -13)) and
                    reco_mother[mother_index] == 0
                ):
                #if (pdg == -11 and mother_pdg == -13):
                    energy = reco_startMomentum[i][3]  # 4th component is energy
                    if energy <= 0.07: # 50 MeV = 0.05 GeV
                        selected_events.append((
                            row['run'], row['subrun'], row['event'], i, energy,
                            row['truth_muonendX'], row['truth_muonendY'], row['truth_muonendZ'], row['weight_ubtune']
                        ))

                    
                    break  # One match per event is enough

    # print(f"Number of bad mother indices: {bad_mother_count}")
    return selected_events


In [ ]:
# Apply the function to df_mc
reco_michel_events_FC = find_reco_michel_with_mu(df_mcFC_run3)
total_weighted_number_michel = sum(event[-1] for event in reco_michel_events_FC)

# Display the results
print(f"Total weighted number of Michel electrons: {total_weighted_number_michel}")
#print("Selected Events with Positrons from Anti-Muons:")
for event in reco_michel_events_FC:
    print(f"Run: {event[0]}, Subrun: {event[1]}, Event: {event[2]}, Michel Index: {event[3]}, Energy: {event[4]*1000:.4f} MeV, muon_endX: {event[5]:.2f}, muon_endY: {event[6]:.2f}, muon_endZ: {event[7]:.2f}, Weight: {event[8]:.2f}")

In [ ]:
# Convert the result into a dataframe
reco_michel_df_FC = pd.DataFrame(reco_michel_events_FC, columns=['Run', 'Subrun', 'Event', 'Michel_Index', 'reco_energy','Muon_endX', 'Muon_endY', 'Muon_endZ', 'weight_ubtune'])

# Display the new dataframe
reco_michel_df_FC.head(-5)

In [ ]:
# Merge to find matching Run/Subrun/Event between reco and truth
matched_michel_df_FC = pd.merge(
    reco_michel_df_FC,
    truth_michel_df_FC,
    on=['Run', 'Subrun', 'Event'],
    suffixes=('_reco', '_truth')
)

# Print how many matched
print(f"Number of matching reco/truth Michel events: {len(matched_michel_df_FC)}")

# Show the matched dataframe
matched_michel_df_FC.head()

PRINT THE UNWEIGHTED ENTRIES FOR COMPARISON

In [ ]:
df = df_mc1_run1_fhc.copy()  # after genNuSelection, before FV

fv_xmin = tpc_xmin + 7.55
fv_xmax = tpc_xmax - 3.
fv_ymin = tpc_ymin + 3.
fv_ymax = tpc_ymax - 17.47
fv_zmin = tpc_zmin + 15
fv_zmax = tpc_zmax - 3.

unshifted = df[
    (df.reco_nuvtxX > fv_xmin) & (df.reco_nuvtxX < fv_xmax) &
    (df.reco_nuvtxY > fv_ymin) & (df.reco_nuvtxY < fv_ymax) &
    (df.reco_nuvtxZ > fv_zmin) & (df.reco_nuvtxZ < fv_zmax)
]

shifted = df[
    ((df.reco_nuvtxX - 1.0) > fv_xmin) & ((df.reco_nuvtxX - 1.0) < fv_xmax) &
    (df.reco_nuvtxY > fv_ymin) & (df.reco_nuvtxY < fv_ymax) &
    (df.reco_nuvtxZ > fv_zmin) & (df.reco_nuvtxZ < fv_zmax)
]

print("unshifted FV:", len(unshifted))
print("shifted FV:  ", len(shifted))
print("difference:  ", len(unshifted) - len(shifted))

In [ ]:
#===============================================================#
# Unweighted per-run cutflow tables
#===============================================================#
import pandas as pd
from IPython.display import display, Markdown

CUT_ORDER = ["None", "genNuSelection", "fiducialVol", "muonCut", "FC", "PC"]
SAMPLE_ORDER = ["DATA", "EXT", "MC", "DIRT"]

RUN_SPLITS = [
    ("Run 1", "RHC", "run1_rhc"),
    ("Run 1", "FHC", "run1_fhc"),
    ("Run 2", "RHC", "run2_rhc"),
    ("Run 2", "FHC", "run2_fhc"),
    ("Run 3", "RHC", "run3"),
    ("Run 4a", "RHC", "run4a"),
    ("Run 4b", "RHC", "run4b"),
    ("Run 4c", "RHC", "run4c_rhc"),
    ("Run 4c", "FHC", "run4c_fhc"),
    ("Run 4d", "FHC", "run4d"),
    ("Run 5", "FHC", "run5", "Run5"),
]

def _resolve_name(*candidates):
    for name in candidates:
        if name in globals():
            return name
    return candidates[0]

def _none_stage_names(suffix):
    if suffix.endswith("_rhc") or suffix.endswith("_fhc"):
        base, beam = suffix.rsplit("_", 1)
        return {
            "DATA": f"df_data_{suffix}",
            "EXT":  _resolve_name(f"df_ext_{base}_sel_{beam}",  f"df_ext_{suffix}_sel"),
            "MC":   _resolve_name(f"df_mc_{base}_sel_{beam}",   f"df_mc_{suffix}_sel"),
            "DIRT": _resolve_name(f"df_dirt_{base}_sel_{beam}", f"df_dirt_{suffix}_sel"),
        }

    return {
        "DATA": f"df_data_{suffix}",
        "EXT":  f"df_ext_{suffix}_sel",
        "MC":   f"df_mc_{suffix}_sel",
        "DIRT": f"df_dirt_{suffix}_sel",
    }

def _stage_var_names(suffix, fcpc_suffix=None):
    fcpc_suffix = fcpc_suffix or suffix
    return {
        "None": _none_stage_names(suffix),
        "genNuSelection": {
            "DATA": f"df_data1_{suffix}",
            "EXT":  f"df_ext1_{suffix}",
            "MC":   f"df_mc1_{suffix}",
            "DIRT": f"df_dirt1_{suffix}",
        },
        "fiducialVol": {
            "DATA": f"df_data2_{suffix}",
            "EXT":  f"df_ext2_{suffix}",
            "MC":   f"df_mc2_{suffix}",
            "DIRT": f"df_dirt2_{suffix}",
        },
        "muonCut": {
            "DATA": f"df_data4_{suffix}",
            "EXT":  f"df_ext4_{suffix}",
            "MC":   f"df_mc4_{suffix}",
            "DIRT": f"df_dirt4_{suffix}",
        },
        "FC": {
            "DATA": f"df_dataFC_{fcpc_suffix}",
            "EXT":  f"df_extFC_{fcpc_suffix}",
            "MC":   f"df_mcFC_{fcpc_suffix}",
            "DIRT": f"df_dirtFC_{fcpc_suffix}",
        },
        "PC": {
            "DATA": f"df_dataPC_{fcpc_suffix}",
            "EXT":  f"df_extPC_{fcpc_suffix}",
            "MC":   f"df_mcPC_{fcpc_suffix}",
            "DIRT": f"df_dirtPC_{fcpc_suffix}",
        },
    }

missing = []
rows = []

for spec in RUN_SPLITS:
    run, beam, suffix, *optional_fcpc_suffix = spec
    fcpc_suffix = optional_fcpc_suffix[0] if optional_fcpc_suffix else None
    stage_names = _stage_var_names(suffix, fcpc_suffix)

    for cut in CUT_ORDER:
        names = stage_names[cut]
        for sample in SAMPLE_ORDER:
            if names[sample] not in globals():
                missing.append(names[sample])

if missing:
    raise NameError(
        "Run the dataframe-loading and per-run applyCuts cells before this summary cell. "
        f"Missing: {', '.join(sorted(set(missing)))}"
    )

for spec in RUN_SPLITS:
    run, beam, suffix, *optional_fcpc_suffix = spec
    fcpc_suffix = optional_fcpc_suffix[0] if optional_fcpc_suffix else None
    stage_names = _stage_var_names(suffix, fcpc_suffix)

    for cut in CUT_ORDER:
        names = stage_names[cut]
        data_n = len(globals()[names["DATA"]])
        ext_n  = len(globals()[names["EXT"]])
        mc_n   = len(globals()[names["MC"]])
        dirt_n = len(globals()[names["DIRT"]])

        rows.append({
            "Run": run,
            "Beam": beam,
            "Cut": cut,
            "DATA": data_n,
            "EXT": ext_n,
            "MC": mc_n,
            "DIRT": dirt_n,
            "MC+EXT+DIRT": mc_n + ext_n + dirt_n,
        })

unweighted_cutflow_by_run = pd.DataFrame(rows)
unweighted_cutflow_by_run["Cut"] = pd.Categorical(
    unweighted_cutflow_by_run["Cut"],
    categories=CUT_ORDER,
    ordered=True,
)

for (run, beam), table in unweighted_cutflow_by_run.groupby(["Run", "Beam"], sort=False):
    display(Markdown(f"### {run} {beam}"))
    display(
        table.sort_values("Cut")
             .set_index("Cut")[["DATA", "EXT", "MC", "DIRT", "MC+EXT+DIRT"]]
             .style.format("{:,}")
    )

unweighted_cutflow_by_run